# 弯曲角度分类：21模型训练、Optuna与原模板绘图

本Notebook包含全部训练、评估、SHAP解释和绘图代码。输入数据内置，下载单个`.ipynb`即可从头运行，无需原项目文件夹或本机路径。默认正式配置为21模型×2种划分，最终搜索100次、嵌套搜索20次，随机种子42。

最新混淆矩阵使用**按真实类别逐行归一化的百分比**，每行未四舍五入前合计100%。原配色、字号和模型顺序沿用原绘图代码。SVG保持Arial及可编辑文字。

本文件为清洁源码版，历史运行输出已清空。GitHub页面不会把旧结果误显示为本次运行结果。首次运行建议将下一单元的`PROFILE`改为`"smoke"`，验证后改回`"full"`进行正式计算。

## 1. 配置与依赖

默认从当前工作目录创建`bend_angle_reproduction/`，所有运行产物保存在该目录。`smoke`只验证流程，不能作为论文性能结果；`full`才使用正式搜索预算。

字体文件不随Notebook分发。运行环境需安装Arial；也可以通过`FONT_DIRECTORY`指定自己的字体目录。内置数据包含196个高度重叠窗口、28个弯曲—回位事件；目标角度类别不等同于逐点瞬时角度。

In [ ]:
from pathlib import Path
import os

# 默认为正式配置；首次检查环境可把"full"改为"smoke"。
PROFILE = os.getenv("BEND_PROFILE", "full")
RUN_MODE = os.getenv("BEND_RUN_MODE", "train_and_plot")
SEED = 42
TEST_SIZE = 0.30
N_JOBS = 1
RESUME = True
FEATURE_SET = "raw_plus_stats"
SHAP_PERMUTATIONS = int(os.getenv("BEND_SHAP_PERMUTATIONS", "128" if PROFILE == "full" else "2"))
ENABLE_TAYLOR = False  # 可选的角度误差辅助诊断，不用于分类模型排名。
FONT_DIRECTORY = None  # 可指定自己已安装/持有的Arial字体目录；不随仓库分发字体文件。
INSTALL_MISSING_PACKAGES = False
OUTPUT_DIRECTORY = Path.cwd() / "bend_angle_reproduction"


### 1.1 写出并校验内置输入
输入包含窗口特征CSV及带相对采样时间的NPZ；已有不同内容文件不会被静默覆盖。

In [ ]:
from pathlib import Path
import base64, hashlib
_EMBEDDED_REQUIREMENTS = 'numpy==1.26.4\npandas==2.2.3\nscipy==1.15.2\nscikit-learn==1.6.1\nmatplotlib==3.10.1\nseaborn==0.13.2\noptuna==4.9.0\nxgboost-cpu==3.2.0; platform_system == "Linux"\nxgboost==3.2.0; platform_system != "Linux"\nlightgbm==4.7.0\ncatboost==1.2.10\njoblib==1.4.2\nthreadpoolctl==3.5.0\nnbformat==5.10.4\nnbclient==0.10.2\nipykernel==6.29.5\nmatplotlib-inline==0.1.7\nshap==0.49.1\nnumba==0.68.0\nllvmlite==0.50.0\ncloudpickle==3.1.2\nslicer==0.0.8\n\nbeautifulsoup4==4.13.3\nnetworkx==3.4.2\nPyMuPDF==1.28.2\nnbconvert==7.16.6\n'
_EMBEDDED_WINDOWS_CSV = '\ufeffwindow_id,cycle_id,raw_00,raw_01,raw_02,raw_03,raw_04,raw_05,raw_06,raw_07,raw_08,raw_09,raw_10,raw_11,raw_12,window_peak_drr,window_mean_drr,window_rms_drr,target_angle_deg\r\nANG_C001_W01,ANG_C001,0.00059256,0.00111,0.00157,0.00197,0.00237,0.00282,0.00316,0.00337,0.00338,0.00338,0.00338,0.00336,0.00331,0.00338,0.0026456502606349987,0.002786147049489122,30\r\nANG_C001_W02,ANG_C001,0.00111,0.00157,0.00197,0.00237,0.00282,0.00316,0.00337,0.00338,0.00338,0.00338,0.00336,0.00331,0.0032,0.00338,0.002844659137465971,0.0029288761898283023,30\r\nANG_C001_W03,ANG_C001,0.00157,0.00197,0.00237,0.00282,0.00316,0.00337,0.00338,0.00338,0.00338,0.00336,0.00331,0.0032,0.00312,0.00338,0.003001061340974165,0.0030464176789616972,30\r\nANG_C001_W04,ANG_C001,0.00197,0.00237,0.00282,0.00316,0.00337,0.00338,0.00338,0.00338,0.00336,0.00331,0.0032,0.00312,0.00298,0.00338,0.0031087404675454386,0.0031302845606078216,30\r\nANG_C001_W05,ANG_C001,0.00237,0.00282,0.00316,0.00337,0.00338,0.00338,0.00338,0.00336,0.00331,0.0032,0.00312,0.00298,0.00271,0.00338,0.0031675677982569597,0.003177208756492253,30\r\nANG_C001_W06,ANG_C001,0.00282,0.00316,0.00337,0.00338,0.00338,0.00338,0.00336,0.00331,0.0032,0.00312,0.00298,0.00271,0.00224,0.00338,0.0031587673371988364,0.0031703241810405016,30\r\nANG_C001_W07,ANG_C001,0.00316,0.00337,0.00338,0.00338,0.00338,0.00336,0.00331,0.0032,0.00312,0.00298,0.00271,0.00224,0.00171,0.00338,0.003073944292212647,0.003103283027012527,30\r\nANG_C002_W05,ANG_C002,0.00136,0.00184,0.00233,0.00274,0.00297,0.00311,0.00321,0.00331,0.00338,0.00339,0.00338,0.00338,0.00336,0.00339,0.002950236991209301,0.0030041048515655772,30\r\nANG_C002_W06,ANG_C002,0.00184,0.00233,0.00274,0.00297,0.00311,0.00321,0.00331,0.00338,0.00339,0.00338,0.00338,0.00336,0.00315,0.00339,0.0030868172887194407,0.0031123178639429505,30\r\nANG_C002_W07,ANG_C002,0.00233,0.00274,0.00297,0.00311,0.00321,0.00331,0.00338,0.00339,0.00338,0.00338,0.00336,0.00315,0.00282,0.00339,0.003163614564087094,0.0031741938440833194,30\r\nANG_C002_W08,ANG_C002,0.00274,0.00297,0.00311,0.00321,0.00331,0.00338,0.00339,0.00338,0.00338,0.00336,0.00315,0.00282,0.00242,0.00339,0.0031683715581017876,0.003177792695944023,30\r\nANG_C002_W09,ANG_C002,0.00297,0.00311,0.00321,0.00331,0.00338,0.00339,0.00338,0.00338,0.00336,0.00315,0.00282,0.00242,0.00206,0.00339,0.0031145076317699596,0.00313422962845459,30\r\nANG_C002_W10,ANG_C002,0.00311,0.00321,0.00331,0.00338,0.00339,0.00338,0.00338,0.00336,0.00315,0.00282,0.00242,0.00206,0.00167,0.00339,0.0030147451106821727,0.0030552126317826855,30\r\nANG_C002_W11,ANG_C002,0.00321,0.00331,0.00338,0.00339,0.00338,0.00338,0.00336,0.00315,0.00282,0.00242,0.00206,0.00167,0.00118,0.00339,0.0028731192380976153,0.00294729965628978,30\r\nANG_C003_W01,ANG_C003,0.00062352,0.00108,0.0015,0.00187,0.00219,0.00256,0.00293,0.00321,0.00333,0.00334,0.00332,0.00327,0.00323,0.00334,0.002541757152100066,0.002680750807870729,30\r\nANG_C003_W02,ANG_C003,0.00108,0.0015,0.00187,0.00219,0.00256,0.00293,0.00321,0.00333,0.00334,0.00332,0.00327,0.00323,0.00315,0.00334,0.0027378682329055487,0.0028244869507568554,30\r\nANG_C003_W03,ANG_C003,0.0015,0.00187,0.00219,0.00256,0.00293,0.00321,0.00333,0.00334,0.00332,0.00327,0.00323,0.00315,0.00309,0.00334,0.0028884545326223012,0.002939495298231333,30\r\nANG_C003_W04,ANG_C003,0.00187,0.00219,0.00256,0.00293,0.00321,0.00333,0.00334,0.00332,0.00327,0.00323,0.00315,0.00309,0.00301,0.00334,0.0030043922048132447,0.003031297938561899,30\r\nANG_C003_W05,ANG_C003,0.00219,0.00256,0.00293,0.00321,0.00333,0.00334,0.00332,0.00327,0.00323,0.00315,0.00309,0.00301,0.0029,0.00334,0.0030836453317086604,0.003095817252678225,30\r\nANG_C003_W06,ANG_C003,0.00256,0.00293,0.00321,0.00333,0.00334,0.00332,0.00327,0.00323,0.00315,0.00309,0.00301,0.0029,0.00265,0.00334,0.003115426723402129,0.0031219552986060484,30\r\nANG_C003_W07,ANG_C003,0.00293,0.00321,0.00333,0.00334,0.00332,0.00327,0.00323,0.00315,0.00309,0.00301,0.0029,0.00265,0.00227,0.00334,0.0030918007701155174,0.0031022456595178887,30\r\nANG_C004_W08,ANG_C004,0.00269,0.0029,0.00302,0.0031,0.00317,0.00322,0.00324,0.00326,0.0033,0.00332,0.00322,0.00298,0.00264,0.00332,0.00311770523436969,0.003122644769260828,30\r\nANG_C004_W09,ANG_C004,0.0029,0.00302,0.0031,0.00317,0.00322,0.00324,0.00326,0.0033,0.00332,0.00322,0.00298,0.00264,0.00228,0.00332,0.003090566952863964,0.0030998295677237006,30\r\nANG_C004_W10,ANG_C004,0.00302,0.0031,0.00317,0.00322,0.00324,0.00326,0.0033,0.00332,0.00322,0.00298,0.00264,0.00228,0.00196,0.00332,0.0030236215293535217,0.0030446178460747826,30\r\nANG_C004_W11,ANG_C004,0.0031,0.00317,0.00322,0.00324,0.00326,0.0033,0.00332,0.00322,0.00298,0.00264,0.00228,0.00196,0.0016,0.00332,0.0029198920056217113,0.0029612087477962766,30\r\nANG_C004_W12,ANG_C004,0.00317,0.00322,0.00324,0.00326,0.0033,0.00332,0.00322,0.00298,0.00264,0.00228,0.00196,0.0016,0.00112,0.00332,0.002769153373167297,0.002844794860247071,30\r\nANG_C004_W13,ANG_C004,0.00322,0.00324,0.00326,0.0033,0.00332,0.00322,0.00298,0.00264,0.00228,0.00196,0.0016,0.00112,0.00071029,0.00332,0.0025809102277264888,0.002705437875119862,30\r\nANG_C004_W14,ANG_C004,0.00324,0.00326,0.0033,0.00332,0.00322,0.00298,0.00264,0.00228,0.00196,0.0016,0.00112,0.00071029,0.0003137,0.00332,0.0023500940718225874,0.002541039682946669,30\r\nANG_C005_W00,ANG_C005,1.635e-05,0.00050601,0.001,0.00145,0.00188,0.00224,0.00266,0.003,0.00323,0.00328,0.00328,0.00326,0.00324,0.00328,0.0022786041662153835,0.002505274047163547,30\r\nANG_C005_W01,ANG_C005,0.00050601,0.001,0.00145,0.00188,0.00224,0.00266,0.003,0.00323,0.00328,0.00328,0.00326,0.00324,0.00322,0.00328,0.0025308999960544955,0.002674812494564337,30\r\nANG_C005_W02,ANG_C005,0.001,0.00145,0.00188,0.00224,0.00266,0.003,0.00323,0.00328,0.00328,0.00326,0.00324,0.00322,0.00315,0.00328,0.00273092592942025,0.0028177982161456025,30\r\nANG_C005_W03,ANG_C005,0.00145,0.00188,0.00224,0.00266,0.003,0.00323,0.00328,0.00328,0.00326,0.00324,0.00322,0.00315,0.00307,0.00328,0.0028907239557173954,0.002937940730420265,30\r\nANG_C005_W04,ANG_C005,0.00188,0.00224,0.00266,0.003,0.00323,0.00328,0.00328,0.00326,0.00324,0.00322,0.00315,0.00307,0.00296,0.00328,0.003004129498419552,0.003026710484078907,30\r\nANG_C005_W05,ANG_C005,0.00224,0.00266,0.003,0.00323,0.00328,0.00328,0.00326,0.00324,0.00322,0.00315,0.00307,0.00296,0.00278,0.00328,0.003070092875636849,0.0030797775475182813,30\r\nANG_C005_W06,ANG_C005,0.00266,0.003,0.00323,0.00328,0.00328,0.00326,0.00324,0.00322,0.00315,0.00307,0.00296,0.00278,0.0025,0.00328,0.0030852472125544464,0.003092066061600006,30\r\nANG_C006_W08,ANG_C006,0.00214,0.00259,0.00287,0.00304,0.00311,0.00317,0.00323,0.00328,0.00333,0.00337,0.00337,0.0033,0.00305,0.00337,0.003104633505086263,0.0031180057750221477,30\r\nANG_C006_W09,ANG_C006,0.00259,0.00287,0.00304,0.00311,0.00317,0.00323,0.00328,0.00333,0.00337,0.00337,0.0033,0.00305,0.0027,0.00337,0.0031470546038165985,0.003153430252021638,30\r\nANG_C006_W10,ANG_C006,0.00287,0.00304,0.00311,0.00317,0.00323,0.00328,0.00333,0.00337,0.00337,0.0033,0.00305,0.0027,0.0023,0.00337,0.0031272725122416796,0.003137042013953747,30\r\nANG_C006_W11,ANG_C006,0.00304,0.00311,0.00317,0.00323,0.00328,0.00333,0.00337,0.00337,0.0033,0.00305,0.0027,0.0023,0.00194,0.00337,0.003055808050858,0.003078784083456419,30\r\nANG_C006_W12,ANG_C006,0.00311,0.00317,0.00323,0.00328,0.00333,0.00337,0.00337,0.0033,0.00305,0.0027,0.0023,0.00194,0.00155,0.00337,0.0029430779822773186,0.002989357980215263,30\r\nANG_C006_W13,ANG_C006,0.00317,0.00323,0.00328,0.00333,0.00337,0.00337,0.0033,0.00305,0.0027,0.0023,0.00194,0.00155,0.00109,0.00337,0.002791790601868351,0.0028745176260581403,30\r\nANG_C006_W14,ANG_C006,0.00323,0.00328,0.00333,0.00337,0.00337,0.0033,0.00305,0.0027,0.0023,0.00194,0.00155,0.00109,0.00057415,0.00337,0.002596287050936584,0.0027341852900182565,30\r\nANG_C007_W02,ANG_C007,3.249e-05,0.00092422,0.00175,0.00247,0.00301,0.00358,0.00421,0.00475,0.00503,0.00505,0.00504,0.005,0.005,0.00505,0.0036232072126013864,0.003938461982596804,45\r\nANG_C007_W03,ANG_C007,0.00092422,0.00175,0.00247,0.00301,0.00358,0.00421,0.00475,0.00503,0.00505,0.00504,0.005,0.005,0.00491,0.00505,0.003992880967115151,0.004185998389342314,45\r\nANG_C007_W04,ANG_C007,0.00175,0.00247,0.00301,0.00358,0.00421,0.00475,0.00503,0.00505,0.00504,0.005,0.005,0.00491,0.00481,0.00505,0.00428128567814876,0.0043939962321704554,45\r\nANG_C007_W05,ANG_C007,0.00247,0.00301,0.00358,0.00421,0.00475,0.00503,0.00505,0.00504,0.005,0.005,0.00491,0.00481,0.00467,0.00505,0.004496797833446176,0.004558524998403323,45\r\nANG_C007_W06,ANG_C007,0.00301,0.00358,0.00421,0.00475,0.00503,0.00505,0.00504,0.005,0.005,0.00491,0.00481,0.00467,0.00449,0.00505,0.0046532647974947714,0.0046823927579437005,45\r\nANG_C007_W07,ANG_C007,0.00358,0.00421,0.00475,0.00503,0.00505,0.00504,0.005,0.005,0.00491,0.00481,0.00467,0.00449,0.0041,0.00505,0.0047320389432916775,0.004745343779651947,45\r\nANG_C007_W08,ANG_C007,0.00421,0.00475,0.00503,0.00505,0.00504,0.005,0.005,0.00491,0.00481,0.00467,0.00449,0.0041,0.00346,0.00505,0.004722610494076327,0.004737754063592777,45\r\nANG_C008_W06,ANG_C008,0.00274,0.00345,0.00411,0.00446,0.00466,0.00475,0.0049,0.00499,0.00505,0.00505,0.00505,0.00497,0.0047,0.00505,0.0045938811468216705,0.0046315178438831985,45\r\nANG_C008_W07,ANG_C008,0.00345,0.00411,0.00446,0.00466,0.00475,0.0049,0.00499,0.00505,0.00505,0.00505,0.00497,0.0047,0.00423,0.00505,0.004710092950927958,0.004725355477379299,45\r\nANG_C008_W08,ANG_C008,0.00411,0.00446,0.00466,0.00475,0.0049,0.00499,0.00505,0.00505,0.00505,0.00497,0.0047,0.00423,0.00365,0.00505,0.004724659082663151,0.004737276516676406,45\r\nANG_C008_W09,ANG_C008,0.00446,0.00466,0.00475,0.0049,0.00499,0.00505,0.00505,0.00505,0.00497,0.0047,0.00423,0.00365,0.0031,0.00505,0.0046523100075557796,0.004678846069323293,45\r\nANG_C008_W10,ANG_C008,0.00466,0.00475,0.0049,0.00499,0.00505,0.00505,0.00505,0.00497,0.0047,0.00423,0.00365,0.0031,0.00251,0.00505,0.004503087524543906,0.0045599500827032055,45\r\nANG_C008_W11,ANG_C008,0.00475,0.0049,0.00499,0.00505,0.00505,0.00505,0.00497,0.0047,0.00423,0.00365,0.0031,0.00251,0.00178,0.00505,0.004293417668538861,0.00440006264802206,45\r\nANG_C008_W12,ANG_C008,0.0049,0.00499,0.00505,0.00505,0.00505,0.00497,0.0047,0.00423,0.00365,0.0031,0.00251,0.00178,0.00111,0.00505,0.004008299176233568,0.0041922157452635505,45\r\nANG_C009_W00,ANG_C009,5.64e-06,0.00073699,0.00151,0.00219,0.00285,0.00341,0.00399,0.00448,0.00481,0.00489,0.00487,0.00482,0.00481,0.00489,0.00340655239492317,0.0037417833386830524,45\r\nANG_C009_W01,ANG_C009,0.00073699,0.00151,0.00219,0.00285,0.00341,0.00399,0.00448,0.00481,0.00489,0.00487,0.00482,0.00481,0.00469,0.00489,0.0037771505379679484,0.003985754270560688,45\r\nANG_C009_W02,ANG_C009,0.00151,0.00219,0.00285,0.00341,0.00399,0.00448,0.00481,0.00489,0.00487,0.00482,0.00481,0.00469,0.00456,0.00489,0.004063324625761916,0.0041867250700197345,45\r\nANG_C009_W03,ANG_C009,0.00219,0.00285,0.00341,0.00399,0.00448,0.00481,0.00489,0.00487,0.00482,0.00481,0.00469,0.00456,0.0044,0.00489,0.004288861855902206,0.0043535340002574426,45\r\nANG_C009_W04,ANG_C009,0.00285,0.00341,0.00399,0.00448,0.00481,0.00489,0.00487,0.00482,0.00481,0.00469,0.00456,0.0044,0.00421,0.00489,0.004439224615693255,0.004469673057590957,45\r\nANG_C009_W05,ANG_C009,0.00341,0.00399,0.00448,0.00481,0.00489,0.00487,0.00482,0.00481,0.00469,0.00456,0.0044,0.00421,0.00386,0.00489,0.004515482064304524,0.00453031774743517,45\r\nANG_C009_W06,ANG_C009,0.00399,0.00448,0.00481,0.00489,0.00487,0.00482,0.00481,0.00469,0.00456,0.0044,0.00421,0.00386,0.00324,0.00489,0.0045026952650537196,0.004520055618029844,45\r\nANG_C010_W08,ANG_C010,0.00344,0.00407,0.00444,0.00461,0.00472,0.00481,0.00491,0.00492,0.00497,0.00498,0.00493,0.00464,0.00415,0.00498,0.00465025464764738,0.00466393841144196,45\r\nANG_C010_W09,ANG_C010,0.00407,0.00444,0.00461,0.00472,0.00481,0.00491,0.00492,0.00497,0.00498,0.00493,0.00464,0.00415,0.00359,0.00498,0.0046590884543703075,0.004671027697559097,45\r\nANG_C010_W10,ANG_C010,0.00444,0.00461,0.00472,0.00481,0.00491,0.00492,0.00497,0.00498,0.00493,0.00464,0.00415,0.00359,0.00306,0.00498,0.00457892104580469,0.00460549022373281,45\r\nANG_C010_W11,ANG_C010,0.00461,0.00472,0.00481,0.00491,0.00492,0.00497,0.00498,0.00493,0.00464,0.00415,0.00359,0.00306,0.00253,0.00498,0.004429695165538,0.004485498522999734,45\r\nANG_C010_W12,ANG_C010,0.00472,0.00481,0.00491,0.00492,0.00497,0.00498,0.00493,0.00464,0.00415,0.00359,0.00306,0.00253,0.00179,0.00498,0.004215559722752687,0.004319965072987089,45\r\nANG_C010_W13,ANG_C010,0.00481,0.00491,0.00492,0.00497,0.00498,0.00493,0.00464,0.00415,0.00359,0.00306,0.00253,0.00179,0.00114,0.00498,0.00394047541021247,0.004117174488993202,45\r\nANG_C010_W14,ANG_C010,0.00491,0.00492,0.00497,0.00498,0.00493,0.00464,0.00415,0.00359,0.00306,0.00253,0.00179,0.00114,0.0004851,0.00498,0.0035984873475314793,0.0038754316412458515,45\r\nANG_C011_W01,ANG_C011,2.441e-05,0.00087941,0.00164,0.00232,0.00295,0.00353,0.00412,0.00465,0.00493,0.00499,0.00495,0.0049,0.00486,0.00499,0.003514854939316391,0.003839979020383903,45\r\nANG_C011_W02,ANG_C011,0.00087941,0.00164,0.00232,0.00295,0.00353,0.00412,0.00465,0.00493,0.00499,0.00495,0.0049,0.00486,0.00477,0.00499,0.003887073493953321,0.004085717492905521,45\r\nANG_C011_W03,ANG_C011,0.00164,0.00232,0.00295,0.00353,0.00412,0.00465,0.00493,0.00499,0.00495,0.0049,0.00486,0.00477,0.00465,0.00499,0.004169768964384115,0.00428679939624098,45\r\nANG_C011_W04,ANG_C011,0.00232,0.00295,0.00353,0.00412,0.00465,0.00493,0.00499,0.00495,0.0049,0.00486,0.00477,0.00465,0.00453,0.00499,0.004394938534811332,0.004455641137898029,45\r\nANG_C011_W05,ANG_C011,0.00295,0.00353,0.00412,0.00465,0.00493,0.00499,0.00495,0.0049,0.00486,0.00477,0.00465,0.00453,0.00434,0.00499,0.00454100739067824,0.004569515111876632,45\r\nANG_C011_W06,ANG_C011,0.00353,0.00412,0.00465,0.00493,0.00499,0.00495,0.0049,0.00486,0.00477,0.00465,0.00453,0.00434,0.00398,0.00499,0.004616375293055676,0.0046298488550589574,45\r\nANG_C011_W07,ANG_C011,0.00412,0.00465,0.00493,0.00499,0.00495,0.0049,0.00486,0.00477,0.00465,0.00453,0.00434,0.00398,0.00336,0.00499,0.004603383010961682,0.004619366186941139,45\r\nANG_C012_W07,ANG_C012,0.00267,0.00345,0.00409,0.00449,0.00469,0.00482,0.00491,0.00499,0.00502,0.00504,0.00504,0.00501,0.0047,0.00504,0.004597829621708158,0.004637171089604401,45\r\nANG_C012_W08,ANG_C012,0.00345,0.00409,0.00449,0.00469,0.00482,0.00491,0.00499,0.00502,0.00504,0.00504,0.00501,0.0047,0.00418,0.00504,0.004716862237055903,0.004732391924329219,45\r\nANG_C012_W09,ANG_C012,0.00409,0.00449,0.00469,0.00482,0.00491,0.00499,0.00502,0.00504,0.00504,0.00501,0.0047,0.00418,0.00358,0.00504,0.004730989965978526,0.004744284935043471,45\r\nANG_C012_W10,ANG_C012,0.00449,0.00469,0.00482,0.00491,0.00499,0.00502,0.00504,0.00504,0.00501,0.0047,0.00418,0.00358,0.00301,0.00504,0.004644758796685102,0.00467465253187723,45\r\nANG_C012_W11,ANG_C012,0.00469,0.00482,0.00491,0.00499,0.00502,0.00504,0.00504,0.00501,0.0047,0.00418,0.00358,0.00301,0.00243,0.00504,0.004492979588620894,0.004554788010542242,45\r\nANG_C012_W12,ANG_C012,0.00482,0.00491,0.00499,0.00502,0.00504,0.00504,0.00501,0.0047,0.00418,0.00358,0.00301,0.00243,0.0017,0.00504,0.004266914028144239,0.004382717593982825,45\r\nANG_C012_W13,ANG_C012,0.00491,0.00499,0.00502,0.00504,0.00504,0.00501,0.0047,0.00418,0.00358,0.00301,0.00243,0.0017,0.00089116,0.00504,0.003976119616294072,0.004173681110068357,45\r\nANG_C013_W00,ANG_C013,7.99e-06,0.00116,0.00225,0.00317,0.00395,0.0047,0.00547,0.00614,0.00653,0.00667,0.00663,0.00661,0.00655,0.00667,0.0047239164620918985,0.005150994012400268,60\r\nANG_C013_W01,ANG_C013,0.00116,0.00225,0.00317,0.00395,0.0047,0.00547,0.00614,0.00653,0.00667,0.00663,0.00661,0.00655,0.00647,0.00667,0.0052219935823493635,0.0054824522860037845,60\r\nANG_C013_W02,ANG_C013,0.00225,0.00317,0.00395,0.0047,0.00547,0.00614,0.00653,0.00667,0.00663,0.00661,0.00655,0.00647,0.00629,0.00667,0.0056041219942087345,0.005756586347211942,60\r\nANG_C013_W03,ANG_C013,0.00317,0.00395,0.0047,0.00547,0.00614,0.00653,0.00667,0.00663,0.00661,0.00655,0.00647,0.00629,0.00612,0.00667,0.005896633812762168,0.005978318745752923,60\r\nANG_C013_W04,ANG_C013,0.00395,0.0047,0.00547,0.00614,0.00653,0.00667,0.00663,0.00661,0.00655,0.00647,0.00629,0.00612,0.00589,0.00667,0.006097439709885944,0.0061364330033657245,60\r\nANG_C013_W05,ANG_C013,0.0047,0.00547,0.00614,0.00653,0.00667,0.00663,0.00661,0.00655,0.00647,0.00629,0.00612,0.00589,0.00551,0.00667,0.006209138727705937,0.006226693884324107,60\r\nANG_C013_W06,ANG_C013,0.00547,0.00614,0.00653,0.00667,0.00663,0.00661,0.00655,0.00647,0.00629,0.00612,0.00589,0.00551,0.00477,0.00667,0.006207756979252416,0.006224933333686406,60\r\nANG_C014_W07,ANG_C014,0.00375,0.00471,0.00542,0.00582,0.00604,0.00624,0.00642,0.0065,0.00654,0.00655,0.00655,0.00642,0.00602,0.00655,0.00600451389483846,0.006044840412776491,60\r\nANG_C014_W08,ANG_C014,0.00471,0.00542,0.00582,0.00604,0.00624,0.00642,0.0065,0.00654,0.00655,0.00655,0.00642,0.00602,0.00536,0.00655,0.006126220198128321,0.006143702239926035,60\r\nANG_C014_W09,ANG_C014,0.00542,0.00582,0.00604,0.00624,0.00642,0.0065,0.00654,0.00655,0.00655,0.00642,0.00602,0.00536,0.00458,0.00655,0.006124205745842876,0.006142728989005593,60\r\nANG_C014_W10,ANG_C014,0.00582,0.00604,0.00624,0.00642,0.0065,0.00654,0.00655,0.00655,0.00642,0.00602,0.00536,0.00458,0.0038,0.00655,0.006005360075173241,0.006047524929029569,60\r\nANG_C014_W11,ANG_C014,0.00604,0.00624,0.00642,0.0065,0.00654,0.00655,0.00655,0.00642,0.00602,0.00536,0.00458,0.0038,0.00304,0.00655,0.005789652860175946,0.005878538958408298,60\r\nANG_C014_W12,ANG_C014,0.00624,0.00642,0.0065,0.00654,0.00655,0.00655,0.00642,0.00602,0.00536,0.00458,0.0038,0.00304,0.00212,0.00655,0.005484292690979915,0.0056480724633507454,60\r\nANG_C014_W13,ANG_C014,0.00642,0.0065,0.00654,0.00655,0.00655,0.00642,0.00602,0.00536,0.00458,0.0038,0.00304,0.00212,0.00112,0.00655,0.005091749884752487,0.005365573931255619,60\r\nANG_C015_W01,ANG_C015,0.00118,0.00228,0.00323,0.00403,0.00474,0.00554,0.00619,0.00663,0.00672,0.00672,0.00672,0.00671,0.00664,0.00672,0.005286173039868358,0.005553566579825858,60\r\nANG_C015_W02,ANG_C015,0.00228,0.00323,0.00403,0.00474,0.00554,0.00619,0.00663,0.00672,0.00672,0.00672,0.00671,0.00664,0.00646,0.00672,0.005693521427022114,0.005848758129217074,60\r\nANG_C015_W03,ANG_C015,0.00323,0.00403,0.00474,0.00554,0.00619,0.00663,0.00672,0.00672,0.00672,0.00671,0.00664,0.00646,0.00629,0.00672,0.005994612512711351,0.00607875103688541,60\r\nANG_C015_W04,ANG_C015,0.00403,0.00474,0.00554,0.00619,0.00663,0.00672,0.00672,0.00672,0.00671,0.00664,0.00646,0.00629,0.00603,0.00672,0.006203814516610498,0.006244530438538513,60\r\nANG_C015_W05,ANG_C015,0.00474,0.00554,0.00619,0.00663,0.00672,0.00672,0.00672,0.00671,0.00664,0.00646,0.00629,0.00603,0.00566,0.00672,0.00632469525840752,0.006342661027056219,60\r\nANG_C015_W06,ANG_C015,0.00554,0.00619,0.00663,0.00672,0.00672,0.00672,0.00671,0.00664,0.00646,0.00629,0.00603,0.00566,0.00493,0.00672,0.006336003539786588,0.006351757137855584,60\r\nANG_C015_W07,ANG_C015,0.00619,0.00663,0.00672,0.00672,0.00672,0.00671,0.00664,0.00646,0.00629,0.00603,0.00566,0.00493,0.00395,0.00672,0.006222576286870789,0.006259477577096788,60\r\nANG_C016_W03,ANG_C016,0.0015,0.00214,0.00297,0.00395,0.00496,0.0057,0.00614,0.00638,0.00657,0.00672,0.00671,0.00671,0.00671,0.00672,0.005247408919768408,0.005515591871013886,60\r\nANG_C016_W04,ANG_C016,0.00214,0.00297,0.00395,0.00496,0.0057,0.00614,0.00638,0.00657,0.00672,0.00671,0.00671,0.00671,0.00671,0.00672,0.005651340229276751,0.0058196268299358905,60\r\nANG_C016_W05,ANG_C016,0.00297,0.00395,0.00496,0.0057,0.00614,0.00638,0.00657,0.00672,0.00671,0.00671,0.00671,0.00671,0.00671,0.00672,0.006007601092349598,0.0060962051997739365,60\r\nANG_C016_W06,ANG_C016,0.00395,0.00496,0.0057,0.00614,0.00638,0.00657,0.00672,0.00671,0.00671,0.00671,0.00671,0.00671,0.00626,0.00672,0.006262897653506221,0.006300601379239353,60\r\nANG_C016_W07,ANG_C016,0.00496,0.0057,0.00614,0.00638,0.00657,0.00672,0.00671,0.00671,0.00671,0.00671,0.00671,0.00626,0.0056,0.00672,0.006385773472392025,0.006400694915597225,60\r\nANG_C016_W08,ANG_C016,0.0057,0.00614,0.00638,0.00657,0.00672,0.00671,0.00671,0.00671,0.00671,0.00671,0.00626,0.0056,0.0048,0.00672,0.006373650172898997,0.00639064135732997,60\r\nANG_C016_W09,ANG_C016,0.00614,0.00638,0.00657,0.00672,0.00671,0.00671,0.00671,0.00671,0.00671,0.00626,0.0056,0.0048,0.00403,0.00672,0.0062469829725530805,0.00628737895468372,60\r\nANG_C017_W03,ANG_C017,0.00221,0.00313,0.00388,0.00462,0.00541,0.00608,0.00644,0.00652,0.00651,0.00652,0.00651,0.00637,0.0062,0.00652,0.005530009360139205,0.005679621887783491,60\r\nANG_C017_W04,ANG_C017,0.00313,0.00388,0.00462,0.00541,0.00608,0.00644,0.00652,0.00651,0.00652,0.00651,0.00637,0.0062,0.00602,0.00652,0.005811321882674406,0.005892578752299089,60\r\nANG_C017_W05,ANG_C017,0.00388,0.00462,0.00541,0.00608,0.00644,0.00652,0.00651,0.00652,0.00651,0.00637,0.0062,0.00602,0.00577,0.00652,0.006009588812729548,0.0060482175420854545,60\r\nANG_C017_W06,ANG_C017,0.00462,0.00541,0.00608,0.00644,0.00652,0.00651,0.00652,0.00651,0.00637,0.0062,0.00602,0.00577,0.00525,0.00652,0.006113398883632962,0.006131341120814964,60\r\nANG_C017_W07,ANG_C017,0.00541,0.00608,0.00644,0.00652,0.00651,0.00652,0.00651,0.00637,0.0062,0.00602,0.00577,0.00525,0.00443,0.00652,0.006099156394098403,0.006119912997157798,60\r\nANG_C017_W08,ANG_C017,0.00608,0.00644,0.00652,0.00651,0.00652,0.00651,0.00637,0.0062,0.00602,0.00577,0.00525,0.00443,0.00341,0.00652,0.005945171190941572,0.00599719129401459,60\r\nANG_C017_W09,ANG_C017,0.00644,0.00652,0.00651,0.00652,0.00651,0.00637,0.0062,0.00602,0.00577,0.00525,0.00443,0.00341,0.00244,0.00652,0.005661274062173541,0.00577626979479481,60\r\nANG_C018_W07,ANG_C018,0.00457,0.00536,0.00579,0.00603,0.0062,0.00639,0.00646,0.00649,0.00646,0.00652,0.00644,0.00617,0.00553,0.00652,0.006109644305040346,0.006127064309657241,60\r\nANG_C018_W08,ANG_C018,0.00536,0.00579,0.00603,0.0062,0.00639,0.00646,0.00649,0.00646,0.00652,0.00644,0.00617,0.00553,0.00476,0.00652,0.006125370240748397,0.006140066282481895,60\r\nANG_C018_W09,ANG_C018,0.00579,0.00603,0.0062,0.00639,0.00646,0.00649,0.00646,0.00652,0.00644,0.00617,0.00553,0.00476,0.00397,0.00652,0.0060270752401414066,0.006060445951237768,60\r\nANG_C018_W10,ANG_C018,0.00603,0.0062,0.00639,0.00646,0.00649,0.00646,0.00652,0.00644,0.00617,0.00553,0.00476,0.00397,0.00323,0.00652,0.005830096857364036,0.005904144594916021,60\r\nANG_C018_W11,ANG_C018,0.0062,0.00639,0.00646,0.00649,0.00646,0.00652,0.00644,0.00617,0.00553,0.00476,0.00397,0.00323,0.0023,0.00652,0.005559264153266108,0.005697326328778757,60\r\nANG_C018_W12,ANG_C018,0.00639,0.00646,0.00649,0.00646,0.00652,0.00644,0.00617,0.00553,0.00476,0.00397,0.00323,0.0023,0.00146,0.00652,0.00519224835346003,0.005427766064756266,60\r\nANG_C018_W13,ANG_C018,0.00646,0.00649,0.00646,0.00652,0.00644,0.00617,0.00553,0.00476,0.00397,0.00323,0.0023,0.00146,0.00063449,0.00652,0.004730540014710338,0.005100313496671482,60\r\nANG_C019_W01,ANG_C019,3.689e-05,0.00176,0.00317,0.00444,0.00561,0.00668,0.00784,0.00893,0.00957,0.00978,0.00969,0.00965,0.00961,0.00978,0.006828644577609628,0.007459735425672686,90\r\nANG_C019_W02,ANG_C019,0.00176,0.00317,0.00444,0.00561,0.00668,0.00784,0.00893,0.00957,0.00978,0.00969,0.00965,0.00961,0.00956,0.00978,0.007534085582271632,0.007939143882132483,90\r\nANG_C019_W03,ANG_C019,0.00317,0.00444,0.00561,0.00668,0.00784,0.00893,0.00957,0.00978,0.00969,0.00965,0.00961,0.00956,0.00931,0.00978,0.008112547283420863,0.008360351805876668,90\r\nANG_C019_W04,ANG_C019,0.00444,0.00561,0.00668,0.00784,0.00893,0.00957,0.00978,0.00969,0.00965,0.00961,0.00956,0.00931,0.00911,0.00978,0.008576761960338943,0.008713398166958194,90\r\nANG_C019_W05,ANG_C019,0.00561,0.00668,0.00784,0.00893,0.00957,0.00978,0.00969,0.00965,0.00961,0.00956,0.00931,0.00911,0.00871,0.00978,0.008899687026669675,0.0089673337500077,90\r\nANG_C019_W06,ANG_C019,0.00668,0.00784,0.00893,0.00957,0.00978,0.00969,0.00965,0.00961,0.00956,0.00931,0.00911,0.00871,0.00813,0.00978,0.009097693436272338,0.009127050318389895,90\r\nANG_C019_W07,ANG_C019,0.00784,0.00893,0.00957,0.00978,0.00969,0.00965,0.00961,0.00956,0.00931,0.00911,0.00871,0.00813,0.00697,0.00978,0.009122040546051553,0.00914679929063777,90\r\nANG_C020_W08,ANG_C020,0.00728,0.0083,0.00881,0.00919,0.00945,0.00975,0.00985,0.00992,0.00993,0.01001,0.00981,0.00922,0.0082,0.01001,0.009336175607540598,0.009360384281007977,90\r\nANG_C020_W09,ANG_C020,0.0083,0.00881,0.00919,0.00945,0.00975,0.00985,0.00992,0.00993,0.01001,0.00981,0.00922,0.0082,0.00707,0.01001,0.009323516541597325,0.009350052670069274,90\r\nANG_C020_W10,ANG_C020,0.00881,0.00919,0.00945,0.00975,0.00985,0.00992,0.00993,0.01001,0.00981,0.00922,0.0082,0.00707,0.006,0.01001,0.009155676403136684,0.009213770343063307,90\r\nANG_C020_W11,ANG_C020,0.00919,0.00945,0.00975,0.00985,0.00992,0.00993,0.01001,0.00981,0.00922,0.0082,0.00707,0.006,0.00493,0.01001,0.00886524572031376,0.008983209403947947,90\r\nANG_C020_W12,ANG_C020,0.00945,0.00975,0.00985,0.00992,0.00993,0.01001,0.00981,0.00922,0.0082,0.00707,0.006,0.00493,0.0035,0.01001,0.008444462892076121,0.00866119398496683,90\r\nANG_C020_W13,ANG_C020,0.00975,0.00985,0.00992,0.00993,0.01001,0.00981,0.00922,0.0082,0.00707,0.006,0.00493,0.0035,0.00223,0.01001,0.007870163484599357,0.008238092943086162,90\r\nANG_C020_W14,ANG_C020,0.00985,0.00992,0.00993,0.01001,0.00981,0.00922,0.0082,0.00707,0.006,0.00493,0.0035,0.00223,0.00096345,0.01001,0.007196678292095676,0.0077586103406771904,90\r\nANG_C021_W01,ANG_C021,0.00185,0.00333,0.0047,0.00598,0.00724,0.00844,0.0095,0.00998,0.01009,0.01009,0.01009,0.01002,0.00981,0.01009,0.007947749086980066,0.008350756464524501,90\r\nANG_C021_W02,ANG_C021,0.00333,0.0047,0.00598,0.00724,0.00844,0.0095,0.00998,0.01009,0.01009,0.01009,0.01002,0.00981,0.00954,0.01009,0.008517448253255613,0.008758269443225528,90\r\nANG_C021_W03,ANG_C021,0.0047,0.00598,0.00724,0.00844,0.0095,0.00998,0.01009,0.01009,0.01009,0.01002,0.00981,0.00954,0.00937,0.01009,0.008970172796860808,0.009099038686414792,90\r\nANG_C021_W04,ANG_C021,0.00598,0.00724,0.00844,0.0095,0.00998,0.01009,0.01009,0.01009,0.01002,0.00981,0.00954,0.00937,0.00901,0.01009,0.009298891206588285,0.009357458132525615,90\r\nANG_C021_W05,ANG_C021,0.00724,0.00844,0.0095,0.00998,0.01009,0.01009,0.01009,0.01002,0.00981,0.00954,0.00937,0.00901,0.00823,0.01009,0.009470097302355887,0.009495767711796641,90\r\nANG_C021_W06,ANG_C021,0.00844,0.0095,0.00998,0.01009,0.01009,0.01009,0.01002,0.00981,0.00954,0.00937,0.00901,0.00823,0.00685,0.01009,0.009449086682638348,0.009479374665375984,90\r\nANG_C021_W07,ANG_C021,0.0095,0.00998,0.01009,0.01009,0.01009,0.01002,0.00981,0.00954,0.00937,0.00901,0.00823,0.00685,0.00521,0.01009,0.009203543307416564,0.009283709175013805,90\r\nANG_C022_W07,ANG_C022,0.00701,0.00821,0.0089,0.00921,0.00947,0.00975,0.00993,0.00997,0.00997,0.00998,0.00981,0.0093,0.00841,0.00998,0.009347541601865456,0.009374732960770307,90\r\nANG_C022_W08,ANG_C022,0.00821,0.0089,0.00921,0.00947,0.00975,0.00993,0.00997,0.00997,0.00998,0.00981,0.0093,0.00841,0.00734,0.00998,0.009373627904039616,0.00939652872207859,90\r\nANG_C022_W09,ANG_C022,0.0089,0.00921,0.00947,0.00975,0.00993,0.00997,0.00997,0.00998,0.00981,0.0093,0.00841,0.00734,0.00623,0.00998,0.009229709089096663,0.009279003397944949,90\r\nANG_C022_W10,ANG_C022,0.00921,0.00947,0.00975,0.00993,0.00997,0.00997,0.00998,0.00981,0.0093,0.00841,0.00734,0.00623,0.00508,0.00998,0.008956256115961323,0.009060291251741287,90\r\nANG_C022_W11,ANG_C022,0.00947,0.00975,0.00993,0.00997,0.00997,0.00998,0.00981,0.0093,0.00841,0.00734,0.00623,0.00508,0.00359,0.00998,0.008539808666788593,0.008741211521083155,90\r\nANG_C022_W12,ANG_C022,0.00975,0.00993,0.00997,0.00997,0.00998,0.00981,0.0093,0.00841,0.00734,0.00623,0.00508,0.00359,0.00228,0.00998,0.007994887677571987,0.008341253757134933,90\r\nANG_C022_W13,ANG_C022,0.00993,0.00997,0.00997,0.00998,0.00981,0.0093,0.00841,0.00734,0.00623,0.00508,0.00359,0.00228,0.00097445,0.00998,0.007316901903649446,0.007860663184632556,90\r\nANG_C023_W01,ANG_C023,0.00169,0.00312,0.00444,0.00562,0.00665,0.00786,0.00886,0.00965,0.0098,0.00983,0.00969,0.0097,0.00957,0.00983,0.007570507308025192,0.007985306687618715,90\r\nANG_C023_W02,ANG_C023,0.00312,0.00444,0.00562,0.00665,0.00786,0.00886,0.00965,0.0098,0.00983,0.00969,0.0097,0.00957,0.00947,0.00983,0.0081592576343019,0.008412436504011538,90\r\nANG_C023_W03,ANG_C023,0.00444,0.00562,0.00665,0.00786,0.00886,0.00965,0.0098,0.00983,0.00969,0.0097,0.00957,0.00947,0.00925,0.00983,0.008641832862291051,0.00878056996889503,90\r\nANG_C023_W04,ANG_C023,0.00562,0.00665,0.00786,0.00886,0.00965,0.0098,0.00983,0.00969,0.0097,0.00957,0.00947,0.00925,0.00904,0.00983,0.008977108252599866,0.00904650914304655,90\r\nANG_C023_W05,ANG_C023,0.00665,0.00786,0.00886,0.00965,0.0098,0.00983,0.00969,0.0097,0.00957,0.00947,0.00925,0.00904,0.0086,0.00983,0.00919827566605601,0.009227238951536933,90\r\nANG_C023_W06,ANG_C023,0.00786,0.00886,0.00965,0.0098,0.00983,0.00969,0.0097,0.00957,0.00947,0.00925,0.00904,0.0086,0.00789,0.00983,0.009278063305427523,0.009293454826406854,90\r\nANG_C023_W07,ANG_C023,0.00886,0.00965,0.0098,0.00983,0.00969,0.0097,0.00957,0.00947,0.00925,0.00904,0.0086,0.00789,0.00675,0.00983,0.009194139678825146,0.009222642891307106,90\r\nANG_C024_W09,ANG_C024,0.01095,0.01176,0.0122,0.01251,0.01282,0.01307,0.01317,0.01325,0.0134,0.01346,0.01306,0.01205,0.0106,0.01346,0.012632540163079218,0.012653505391770702,120\r\nANG_C024_W10,ANG_C024,0.01176,0.0122,0.01251,0.01282,0.01307,0.01317,0.01325,0.0134,0.01346,0.01306,0.01205,0.0106,0.00919,0.01346,0.012506972700318867,0.012547821823951278,120\r\nANG_C024_W11,ANG_C024,0.0122,0.01251,0.01282,0.01307,0.01317,0.01325,0.0134,0.01346,0.01306,0.01205,0.0106,0.00919,0.00788,0.01346,0.012220144945839404,0.01231144861605952,120\r\nANG_C024_W12,ANG_C024,0.01251,0.01282,0.01307,0.01317,0.01325,0.0134,0.01346,0.01306,0.01205,0.0106,0.00919,0.00788,0.00648,0.01346,0.011803289746942546,0.011976951010619026,120\r\nANG_C024_W13,ANG_C024,0.01282,0.01307,0.01317,0.01325,0.0134,0.01346,0.01306,0.01205,0.0106,0.00919,0.00788,0.00648,0.00454,0.01346,0.01122095149761989,0.011527694832407998,120\r\nANG_C024_W14,ANG_C024,0.01307,0.01317,0.01325,0.0134,0.01346,0.01306,0.01205,0.0106,0.00919,0.00788,0.00648,0.00454,0.00281,0.01346,0.010427784955292143,0.010941968862835995,120\r\nANG_C024_W15,ANG_C024,0.01317,0.01325,0.0134,0.01346,0.01306,0.01205,0.0106,0.00919,0.00788,0.00648,0.00454,0.00281,0.00121,0.01346,0.009495922646057284,0.010277835783611059,120\r\nANG_C025_W00,ANG_C025,1.1e-05,0.00191,0.00387,0.00564,0.00727,0.00866,0.01025,0.01188,0.01306,0.01346,0.01345,0.01339,0.01345,0.01346,0.009156667210134318,0.01011763070497964,120\r\nANG_C025_W01,ANG_C025,0.00191,0.00387,0.00564,0.00727,0.00866,0.01025,0.01188,0.01306,0.01346,0.01345,0.01339,0.01345,0.01345,0.01346,0.010205803082918505,0.010840210461260123,120\r\nANG_C025_W02,ANG_C025,0.00387,0.00564,0.00727,0.00866,0.01025,0.01188,0.01306,0.01346,0.01345,0.01339,0.01345,0.01345,0.01316,0.01346,0.011049317364192967,0.011451936769381527,120\r\nANG_C025_W03,ANG_C025,0.00564,0.00727,0.00866,0.01025,0.01188,0.01306,0.01346,0.01345,0.01339,0.01345,0.01345,0.01316,0.01278,0.01346,0.011730081705147274,0.011966446475686857,120\r\nANG_C025_W04,ANG_C025,0.00727,0.00866,0.01025,0.01188,0.01306,0.01346,0.01345,0.01339,0.01345,0.01345,0.01316,0.01278,0.01238,0.01346,0.012244556213450896,0.012368226734350177,120\r\nANG_C025_W05,ANG_C025,0.00866,0.01025,0.01188,0.01306,0.01346,0.01345,0.01339,0.01345,0.01345,0.01316,0.01278,0.01238,0.012,0.01346,0.012588326455023138,0.012644302978306413,120\r\nANG_C025_W06,ANG_C025,0.01025,0.01188,0.01306,0.01346,0.01345,0.01339,0.01345,0.01345,0.01316,0.01278,0.01238,0.012,0.0113,0.01346,0.012776519208211436,0.012800370564793629,120\r\nANG_C026_W08,ANG_C026,0.0101,0.01145,0.0122,0.01266,0.01292,0.01313,0.01322,0.01331,0.01335,0.01346,0.01346,0.0131,0.01209,0.01346,0.012771336789248574,0.012795471082764343,120\r\nANG_C026_W09,ANG_C026,0.01145,0.0122,0.01266,0.01292,0.01313,0.01322,0.01331,0.01335,0.01346,0.01346,0.0131,0.01209,0.01062,0.01346,0.012821130614546943,0.012838554649718914,120\r\nANG_C026_W10,ANG_C026,0.0122,0.01266,0.01292,0.01313,0.01322,0.01331,0.01335,0.01346,0.01346,0.0131,0.01209,0.01062,0.00914,0.01346,0.012662107127391652,0.012704205053170712,120\r\nANG_C026_W11,ANG_C026,0.01266,0.01292,0.01313,0.01322,0.01331,0.01335,0.01346,0.01346,0.0131,0.01209,0.01062,0.00914,0.00783,0.01346,0.012332070389630627,0.012430015397037348,120\r\nANG_C026_W12,ANG_C026,0.01292,0.01313,0.01322,0.01331,0.01335,0.01346,0.01346,0.0131,0.01209,0.01062,0.00914,0.00783,0.00652,0.01346,0.011849360581797789,0.012036594821552143,120\r\nANG_C026_W13,ANG_C026,0.01313,0.01322,0.01331,0.01335,0.01346,0.01346,0.0131,0.01209,0.01062,0.00914,0.00783,0.00652,0.00467,0.01346,0.011215436930631007,0.011537228371166039,120\r\nANG_C026_W14,ANG_C026,0.01322,0.01331,0.01335,0.01346,0.01346,0.0131,0.01209,0.01062,0.00914,0.00783,0.00652,0.00467,0.00301,0.01346,0.010415375017204043,0.010932897694890016,120\r\nANG_C027_W00,ANG_C027,8.448e-05,0.00242,0.00424,0.00592,0.00748,0.0088,0.01033,0.01177,0.01279,0.01306,0.01306,0.01298,0.01305,0.01306,0.009090723692079212,0.009943332170451973,120\r\nANG_C027_W01,ANG_C027,0.00242,0.00424,0.00592,0.00748,0.0088,0.01033,0.01177,0.01279,0.01306,0.01306,0.01298,0.01305,0.01294,0.01306,0.010067994267475792,0.010617954944435024,120\r\nANG_C027_W02,ANG_C027,0.00424,0.00592,0.00748,0.0088,0.01033,0.01177,0.01279,0.01306,0.01306,0.01298,0.01305,0.01294,0.01276,0.01306,0.010881927048357222,0.011219098010343896,120\r\nANG_C027_W03,ANG_C027,0.00592,0.00748,0.0088,0.01033,0.01177,0.01279,0.01306,0.01306,0.01298,0.01305,0.01294,0.01276,0.01237,0.01306,0.011503201987784954,0.011696836164910566,120\r\nANG_C027_W04,ANG_C027,0.00748,0.0088,0.01033,0.01177,0.01279,0.01306,0.01306,0.01298,0.01305,0.01294,0.01276,0.01237,0.01206,0.01306,0.011953046323943268,0.012054223256188146,120\r\nANG_C027_W05,ANG_C027,0.0088,0.01033,0.01177,0.01279,0.01306,0.01306,0.01298,0.01305,0.01294,0.01276,0.01237,0.01206,0.01153,0.01306,0.012267313092919539,0.012311023632317194,120\r\nANG_C027_W06,ANG_C027,0.01033,0.01177,0.01279,0.01306,0.01306,0.01298,0.01305,0.01294,0.01276,0.01237,0.01206,0.01153,0.01068,0.01306,0.012404786642767701,0.012425993513260488,120\r\nANG_C028_W08,ANG_C028,0.00912,0.01087,0.01177,0.01238,0.01268,0.01304,0.01329,0.01339,0.01337,0.01341,0.01336,0.01307,0.01206,0.01341,0.012589530668517375,0.012630831373181434,120\r\nANG_C028_W09,ANG_C028,0.01087,0.01177,0.01238,0.01268,0.01304,0.01329,0.01339,0.01337,0.01341,0.01336,0.01307,0.01206,0.01065,0.01341,0.012711663741414681,0.012734033824056154,120\r\nANG_C028_W10,ANG_C028,0.01177,0.01238,0.01268,0.01304,0.01329,0.01339,0.01337,0.01341,0.01336,0.01307,0.01206,0.01065,0.00902,0.01341,0.012595284534713758,0.012638097866200827,120\r\nANG_C028_W11,ANG_C028,0.01238,0.01268,0.01304,0.01329,0.01339,0.01337,0.01341,0.01336,0.01307,0.01206,0.01065,0.00902,0.00753,0.01341,0.012278724729355506,0.012381380307295794,120\r\nANG_C028_W12,ANG_C028,0.01268,0.01304,0.01329,0.01339,0.01337,0.01341,0.01336,0.01307,0.01206,0.01065,0.00902,0.00753,0.00606,0.01341,0.011800305044637612,0.01200289330016996,120\r\nANG_C028_W13,ANG_C028,0.01304,0.01329,0.01339,0.01337,0.01341,0.01336,0.01307,0.01206,0.01065,0.00902,0.00753,0.00606,0.00423,0.01341,0.011140306585101038,0.011498295313836436,120\r\nANG_C028_W14,ANG_C028,0.01329,0.01339,0.01337,0.01341,0.01336,0.01307,0.01206,0.01065,0.00902,0.00753,0.00606,0.00423,0.00229,0.01341,0.010300376877617593,0.010883182619612056,120\r\n'
_EMBEDDED_WAVEFORMS = '...'

_project = OUTPUT_DIRECTORY.resolve()
_project.mkdir(parents=True, exist_ok=True)
(_project / "data").mkdir(exist_ok=True)
_inputs = {
    "data/peak_windows_13points.csv": _EMBEDDED_WINDOWS_CSV.encode("utf8"),
    "data/angle_waveforms.npz": base64.b64decode(_EMBEDDED_WAVEFORMS),
    "requirements.txt": _EMBEDDED_REQUIREMENTS.encode("utf8"),
}
_input_hashes = {}
for _relative, _content in _inputs.items():
    _path = _project / _relative
    _expected = hashlib.sha256(_content).hexdigest()
    if _path.exists():
        if hashlib.sha256(_path.read_bytes()).hexdigest() != _expected:
            raise ValueError(f"已有文件与内置输入不一致：{_relative}。请指定新的OUTPUT_DIRECTORY。")
    else:
        _path.write_bytes(_content)
    _input_hashes[_relative] = _expected
print("输入数据已核查。工作目录：", _project)


### 1.2 导入依赖
可先安装随附`requirements.txt`。单独下载Notebook时，本文件上一单元也会生成依赖清单；缺包时按提示安装。

In [ ]:
from pathlib import Path
import os, sys, json, time, hashlib, warnings, math, importlib.metadata
from io import StringIO
import subprocess,importlib.util
_required=['numpy','pandas','matplotlib','matplotlib_inline','seaborn','sklearn','optuna','xgboost','lightgbm','catboost','joblib','shap','bs4','networkx','pymupdf']
_missing=[p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    if not INSTALL_MISSING_PACKAGES:
        raise ModuleNotFoundError(
            f"当前内核缺少{_missing}。请在当前环境执行：python -m pip install -r {_project / 'requirements.txt'}；"
            "或将INSTALL_MISSING_PACKAGES设为True后重跑本单元。")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(_project / "requirements.txt")])
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import font_manager
import seaborn as sns
import optuna
from optuna.trial import TrialState
from sklearn import metrics as skm
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF, Matern
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier,
    HistGradientBoostingClassifier, AdaBoostClassifier, BaggingClassifier)
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib
from IPython.display import display, SVG, Markdown
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
optuna.logging.set_verbosity(optuna.logging.WARNING)
def package_version(name):
    try:return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        if name=='xgboost':return importlib.metadata.version('xgboost-cpu')
        raise
VERSIONS={p:package_version(p) for p in
          ['numpy','pandas','scipy','scikit-learn','matplotlib','seaborn','optuna','xgboost','lightgbm','catboost']}
print('Python:',sys.executable)
display(pd.Series(VERSIONS,name='version').to_frame())


### 1.3 确定运行配置与独立输出目录
交叉验证请求最多5折，按每类可用事件数自动减少。实际折数写入日志，不能把请求折数当作实际执行折数。

In [ ]:
PROFILES={
    'smoke':dict(final_trials=5,nested_trials=1,outer_folds=2,inner_folds=2,small_models=True),
    'quick':dict(final_trials=15,nested_trials=3,outer_folds=3,inner_folds=3,small_models=False),
    'full':dict(final_trials=100,nested_trials=20,outer_folds=5,inner_folds=5,small_models=False),
}
if PROFILE not in PROFILES: raise ValueError('PROFILE必须为smoke/quick/full')
CFG=PROFILES[PROFILE].copy()
LIVE_OPTUNA=PROFILE!='smoke'   # 正式搜索按下面的频率更新可视化
LIVE_EVERY=5
SCHEMA_VERSION='bend21-github-percent-v2'
np.random.seed(SEED)
candidates=[_project]
ROOT=next((p for p in candidates if (p/'data/peak_windows_13points.csv').exists()),None)
if ROOT is None: raise FileNotFoundError('内置数据准备失败，请从头运行Notebook。')
DATA_PATH=ROOT/'data/peak_windows_13points.csv'
data_hash=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
signature=hashlib.sha256(json.dumps(dict(profile=PROFILE,features=FEATURE_SET,seed=SEED,
    test_size=TEST_SIZE,cfg=CFG,data=data_hash,version=SCHEMA_VERSION),sort_keys=True).encode()).hexdigest()[:12]
OUT=ROOT/'artifacts'/f'{PROFILE}_{FEATURE_SET}_{signature}'
OUT.mkdir(parents=True,exist_ok=True)
def write_json(obj,path):
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    Path(path).write_text(json.dumps(obj,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
def write_csv(df,path):
    Path(path).parent.mkdir(parents=True,exist_ok=True)
    df.to_csv(path,index=False,encoding='utf-8-sig')
write_json(dict(profile=PROFILE,cfg=CFG,feature_set=FEATURE_SET,seed=SEED,test_size=TEST_SIZE,
    versions=VERSIONS,data_sha256=data_hash,scope='single-record exploratory classification'),OUT/'settings.json')
display(pd.Series(CFG,name=PROFILE).to_frame())
print('输出目录:',OUT)
if PROFILE=='smoke': display(Markdown('**当前为SMOKE程序验证：试验数不足以支撑正式模型结论。**'))
if RUN_MODE not in ["train_and_plot", "saved_results"]:
    raise ValueError("RUN_MODE应为train_and_plot或saved_results")
write_json(dict(_input_hashes), ROOT / "input_hashes.json")


### 1.4 Arial与矢量导出

In [ ]:
# 注册本机Arial；字体文件不内嵌，也不调用桌面应用。
_font_paths = []
if FONT_DIRECTORY is not None:
    _font_paths.extend(Path(FONT_DIRECTORY).glob("*.ttf"))
if os.environ.get("WINDIR"):
    _system_fonts = Path(os.environ["WINDIR"]) / "Fonts"
    _font_paths.extend(p for p in (_system_fonts / n for n in
                      ["arial.ttf", "arialbd.ttf", "ariali.ttf", "arialbi.ttf"]) if p.exists())
for _font in _font_paths:
    font_manager.fontManager.addfont(str(_font))
try:
    ARIAL_PATH = font_manager.findfont("Arial", fallback_to_default=False)
except ValueError as exc:
    raise RuntimeError("运行环境未找到Arial。请先安装Arial，或设置FONT_DIRECTORY后重跑本单元。") from exc
CLASS_ANGLES=np.array([30,45,60,90,120])
CLASS_LABELS=[f'{v} deg' for v in CLASS_ANGLES]
CLASS_COLORS=['#1f77b4','#ff7f0e','#2ca02c','#d62728','#9467bd']
SCHEME_LABELS={'row_random':'Random window split','cycle_grouped':'Cycle-grouped split'}
SCHEME_COLORS={'row_random':'#6EC1E4','cycle_grouped':'#F08A5D'}
def apply_style():
    sns.set_theme(style='white',font='Arial')
    mpl.rcParams.update({'font.family':'Arial','font.sans-serif':['Arial'],'font.size':11,
        'axes.labelsize':12,'axes.titlesize':14,'legend.fontsize':9,'axes.unicode_minus':False,
        'axes.linewidth':1.1,'figure.dpi':110,'figure.facecolor':'white','axes.facecolor':'white',
        'svg.fonttype':'none','svg.image_inline':True,'savefig.bbox':'tight'})
apply_style()
FIGURE_INDEX=[]
def vectorize(fig):
    for artist in fig.findobj():
        if hasattr(artist,'set_rasterized'):
            try: artist.set_rasterized(False)
            except (AttributeError,TypeError): pass
    for text in fig.findobj(mpl.text.Text): text.set_fontfamily('Arial')
def save_svg(fig,path,show=False):
    path=Path(path).with_suffix('.svg');path.parent.mkdir(parents=True,exist_ok=True)
    vectorize(fig)
    # 导出时再次强制文字保留为SVG text，避免第三方绘图改动全局配置。
    with mpl.rc_context({'svg.fonttype':'none','text.usetex':False}):
        fig.savefig(path,format='svg',bbox_inches='tight',metadata={'Date':None,'Creator':'Bend-angle notebook'})
    xml=path.read_text(encoding='utf-8')
    if '<image' in xml: raise RuntimeError(f'SVG出现嵌入位图：{path}')
    if '<text' not in xml: raise RuntimeError(f'SVG文字未保持为text：{path}')
    from xml.etree import ElementTree as _SVG_ET
    svg_root=_SVG_ET.fromstring(xml)
    for node in svg_root.iter():
        if node.tag.endswith('}text') and ''.join(node.itertext()).strip():
            # 科学计数法的SVG文字由多个tspan组成，字体可以声明在子节点上。
            text_parts=[part for part in node.iter() if part.text and part.text.strip()]
            if not all('Arial' in part.attrib.get('style','') or 'Arial' in node.attrib.get('style','') for part in text_parts):
                raise RuntimeError(f'SVG文本未显式使用Arial：{path}')
    FIGURE_INDEX.append(str(path.relative_to(OUT)))
    if show: display(SVG(filename=str(path)))
    plt.close(fig)
print('Arial:',ARIAL_PATH)


## 2. 数据、特征与分组划分
正式输入为13个实测响应点及峰值、时间加权均值、时间加权RMS，共16维。窗口重叠共享数据点，周期分组方案避免同一事件进入训练与测试两侧。随机窗口方案只作对照。

In [ ]:
data=pd.read_csv(DATA_PATH,float_precision='round_trip')
RAW_FEATURES=[f'raw_{i:02d}' for i in range(13)]
STAT_FEATURES=['window_peak_drr','window_mean_drr','window_rms_drr']
FEATURE_COLUMNS=RAW_FEATURES+(STAT_FEATURES if FEATURE_SET=='raw_plus_stats' else [])
if FEATURE_SET not in ['raw_only','raw_plus_stats']: raise ValueError(FEATURE_SET)
angle_to_code={int(a):i for i,a in enumerate(CLASS_ANGLES)}
y=data.target_angle_deg.map(angle_to_code).to_numpy(dtype=int)
X=data[FEATURE_COLUMNS].to_numpy(dtype=float)
groups=data.cycle_id.to_numpy(dtype=str)
source_npz=np.load(ROOT/'data/angle_waveforms.npz',allow_pickle=False)
assert np.array_equal(source_npz['window_id'],data.window_id.to_numpy())
source_rows=source_npz['source_rows_window']
assert data.window_id.is_unique and len(data)==196 and len(np.unique(groups))==28
assert np.isfinite(X).all() and set(np.unique(y))==set(range(5))
assert all(c not in FEATURE_COLUMNS for c in ['window_id','cycle_id','time_s','target_angle_deg','cv_fold'])
display(data[['window_id','cycle_id']+FEATURE_COLUMNS[:3]+['target_angle_deg']].head())
display(data.groupby('target_angle_deg').agg(windows=('window_id','size'),cycles=('cycle_id','nunique')))
print('X:',X.shape,'y:',y.shape,'groups:',len(np.unique(groups)))
# 从内置原始窗口及时间坐标独立复核三个统计量，不覆盖已冻结的正式输入。
_raw = source_npz["X_window_raw"]
_time = source_npz["T_window_relative"]
_dt = np.diff(_time, axis=1)
_duration = _time[:, -1] - _time[:, 0]
_mean_check = (_dt * (_raw[:, :-1] + _raw[:, 1:]) / 2).sum(axis=1) / _duration
_rms_check = np.sqrt((_dt * (_raw[:, :-1]**2 + _raw[:, :-1]*_raw[:, 1:] + _raw[:, 1:]**2) / 3).sum(axis=1) / _duration)
assert np.allclose(data[RAW_FEATURES].to_numpy(), _raw, rtol=0, atol=1e-14)
assert np.allclose(data.window_peak_drr, _raw.max(axis=1), rtol=0, atol=1e-14)
assert np.allclose(data.window_mean_drr, _mean_check, rtol=0, atol=1e-14)
assert np.allclose(data.window_rms_drr, _rms_check, rtol=0, atol=1e-14)
display(pd.DataFrame({"feature": FEATURE_COLUMNS,
    "definition": [f"窗口内第{i+1}个原始ΔR/R0响应点" for i in range(13)] +
                  ["窗口峰值", "按采样时间进行分段线性积分的均值", "分段线性函数平方精确积分后开方的RMS"]}))


### 2.1 冻结两套划分与交叉验证

In [ ]:
def cycle_table(y_values,group_values):
    t=pd.DataFrame({'group':group_values,'y':y_values})
    if not t.groupby('group').y.nunique().eq(1).all(): raise ValueError('周期内标签不一致')
    return t.drop_duplicates('group').reset_index(drop=True)
def make_folds(y_values,group_values,scheme,requested,seed):
    if scheme=='cycle_grouped':
        tab=cycle_table(y_values,group_values)
        n=min(requested,int(tab.y.value_counts().min()))
        if n<2: raise ValueError('分组CV每类至少需要2个训练周期')
        skf=StratifiedKFold(n_splits=n,shuffle=True,random_state=seed)
        result=[(np.flatnonzero(np.isin(group_values,tab.group.iloc[a])),
                 np.flatnonzero(np.isin(group_values,tab.group.iloc[b]))) for a,b in skf.split(tab,tab.y)]
    else:
        n=min(requested,int(pd.Series(y_values).value_counts().min()))
        skf=StratifiedKFold(n_splits=n,shuffle=True,random_state=seed)
        result=list(skf.split(np.zeros(len(y_values)),y_values))
    for a,b in result:
        assert set(np.unique(y_values[a]))==set(range(5)) and set(np.unique(y_values[b]))==set(range(5))
        if scheme=='cycle_grouped': assert set(group_values[a]).isdisjoint(group_values[b])
    return result
SPLITS={}
indices=np.arange(len(data))
tr,te=train_test_split(indices,test_size=TEST_SIZE,stratify=y,random_state=SEED)
SPLITS['row_random']=(np.sort(tr),np.sort(te))
ct=cycle_table(y,groups)
cycle_tr,cycle_te=train_test_split(ct,test_size=TEST_SIZE,stratify=ct.y,random_state=SEED)
SPLITS['cycle_grouped']=(np.flatnonzero(np.isin(groups,cycle_tr.group)),np.flatnonzero(np.isin(groups,cycle_te.group)))
split_audit=[]
for scheme,(tr,te) in SPLITS.items():
    shared_groups=set(groups[tr]) & set(groups[te])
    shared_points=set(source_rows[tr].ravel()) & set(source_rows[te].ravel())
    if scheme=='cycle_grouped': assert not shared_groups and not shared_points
    split_audit.append(dict(scheme=scheme,train_windows=len(tr),test_windows=len(te),
        train_cycles=len(set(groups[tr])),test_cycles=len(set(groups[te])),
        shared_cycles=len(shared_groups),shared_source_points=len(shared_points)))
    assignment=data[['window_id','cycle_id','target_angle_deg']].copy()
    assignment['split']='train';assignment.loc[te,'split']='test'
    write_csv(assignment,OUT/scheme/'split_assignment.csv')
split_audit=pd.DataFrame(split_audit)
write_csv(split_audit,OUT/'split_audit.csv')
display(split_audit)
display(Markdown('两套方案的测试窗口不同，后续差异是描述性对照，不是同一测试集上的配对因果比较。'))


### 2.2 类别样本数量

In [ ]:
# 数据数量保留为表；用户未提供此图模板，不另外设计数据图。
counts=data.target_angle_deg.value_counts().reindex(CLASS_ANGLES).rename_axis('angle_deg').reset_index(name='windows')
write_csv(counts,OUT/'window_counts.csv')
display(counts)


## 3. 模型目录与训练
### 3.1 21个分类器及预处理

In [ ]:
MODEL_CATALOG=pd.DataFrame([
 ('LR','Logistic regression','standard','正则化对尺度敏感；多分类逻辑回归'),
 ('RIDGE','Calibrated ridge classifier','standard','岭惩罚依赖尺度；训练内校准提供概率'),
 ('LINEAR_SVC','Calibrated linear SVC','standard','间隔与C依赖尺度；训练内校准提供概率'),
 ('SVM','RBF SVC','standard','RBF距离与gamma依赖尺度；启用概率输出'),
 ('SGD','SGD logistic classifier','standard','优化与正则化受尺度影响；log_loss输出概率'),
 ('KNN','K nearest neighbors','standard','近邻距离受尺度影响；限制邻居数'),
 ('GNB','Gaussian Naive Bayes','none','无需距离缩放；var_smoothing调方差稳定性'),
 ('LDA','Shrinkage LDA','standard','相关特征使用lsqr与shrinkage稳定协方差'),
 ('QDA','Regularized QDA','standard','样本少且共线；调reg_param，不使用无正则协方差'),
 ('GPC','Gaussian process classifier','standard','核距离依赖尺度；固定核优化器由Optuna搜索尺度'),
 ('DT','Decision tree','none','轴向阈值分裂通常无需缩放；限制树深和叶节点'),
 ('RF','Random forest','none','树集成；调深度、叶样本和特征抽样'),
 ('ET','Extra Trees','none','随机阈值树集成；无需距离缩放'),
 ('GBDT','Gradient boosting','none','小数据限制树深、学习率与树数'),
 ('HGB','Histogram gradient boosting','none','分箱树；最小叶样本必须适应小样本'),
 ('ADA','AdaBoost','none','弱树深度与学习率；不使用已弃用的SAMME.R配置'),
 ('BAG','Bagging trees','none','树基学习器；bootstrap需保持类别可学习'),
 ('XGB','XGBoost','none','multi:softprob五类；正则、深度及采样比例'),
 ('LGBM','LightGBM','none','五类；限制叶数和最小叶样本，启用行采样频率'),
 ('CAT','CatBoost','none','MultiClass；CPU单线程且不写隐式catboost_info'),
 ('ANN','Multilayer perceptron','standard','小表格用小网络及lbfgs；调L2而非盲目加深'),
],columns=['model','description','scaling','reason'])
MODELS=MODEL_CATALOG.model.tolist()
assert len(MODELS)==21
SCALE_MAP=MODEL_CATALOG.set_index('model').scaling.to_dict()
write_csv(MODEL_CATALOG,OUT/'model_preprocessing_catalog.csv')
display(MODEL_CATALOG)


### 3.2 Optuna搜索空间

In [ ]:
def suggest_params(trial,name):
    small=CFG['small_models']
    if name=='LR': return {'C':trial.suggest_float('C',1e-3,100,log=True)}
    if name=='RIDGE': return {'alpha':trial.suggest_float('alpha',1e-3,100,log=True)}
    if name=='LINEAR_SVC': return {'C':trial.suggest_float('C',.01,100,log=True)}
    if name=='SVM': return {'C':trial.suggest_float('C',.01,1000,log=True),'gamma':trial.suggest_float('gamma',1e-3,1,log=True)}
    if name=='SGD': return {'alpha':trial.suggest_float('alpha',1e-5,.1,log=True),'l1_ratio':trial.suggest_float('l1_ratio',0,1)}
    if name=='KNN': return {'n_neighbors':trial.suggest_int('n_neighbors',2,9),'weights':trial.suggest_categorical('weights',['uniform','distance']),'p':trial.suggest_int('p',1,2)}
    if name=='GNB': return {'var_smoothing':trial.suggest_float('var_smoothing',1e-11,1e-2,log=True)}
    if name=='LDA': return {'shrinkage':trial.suggest_float('shrinkage',.05,.95)}
    if name=='QDA': return {'reg_param':trial.suggest_float('reg_param',.1,.99)}
    if name=='GPC': return {'kernel':trial.suggest_categorical('kernel',['rbf','matern']),'length_scale':trial.suggest_float('length_scale',.1,30,log=True)}
    if name in ['DT','RF','ET']:
        p={'max_depth':trial.suggest_int('max_depth',2,5 if small else 10),
           'min_samples_leaf':trial.suggest_int('min_samples_leaf',1,8),
           'min_samples_split':trial.suggest_int('min_samples_split',2,12)}
        if name!='DT': p.update(n_estimators=trial.suggest_int('n_estimators',20 if small else 80,50 if small else 400,step=10 if small else 40),max_features=trial.suggest_float('max_features',.5,1))
        return p
    if name=='GBDT': return {'n_estimators':trial.suggest_int('n_estimators',20 if small else 50,50 if small else 300,step=10 if small else 50),'max_depth':trial.suggest_int('max_depth',1,3),
        'learning_rate':trial.suggest_float('learning_rate',.02,.25,log=True),'subsample':trial.suggest_float('subsample',.7,1),'min_samples_leaf':trial.suggest_int('min_samples_leaf',2,10)}
    if name=='HGB': return {'max_iter':trial.suggest_int('max_iter',20 if small else 50,50 if small else 300,step=10 if small else 50),
        'max_leaf_nodes':trial.suggest_int('max_leaf_nodes',4,16),'min_samples_leaf':trial.suggest_int('min_samples_leaf',3,15),
        'learning_rate':trial.suggest_float('learning_rate',.02,.25,log=True),'l2_regularization':trial.suggest_float('l2_regularization',1e-4,10,log=True)}
    if name in ['ADA','BAG']:
        p={'n_estimators':trial.suggest_int('n_estimators',10 if small else 30,30 if small else 150,step=10 if small else 30),'max_depth':trial.suggest_int('max_depth',1,3 if name=='ADA' else 6)}
        if name=='ADA': p['learning_rate']=trial.suggest_float('learning_rate',.03,1,log=True)
        else: p['max_samples']=trial.suggest_float('max_samples',.7,1)
        return p
    if name in ['XGB','LGBM']:
        p={'n_estimators':trial.suggest_int('n_estimators',20 if small else 50,50 if small else 350,step=10 if small else 50),
           'max_depth':trial.suggest_int('max_depth',2,5),'learning_rate':trial.suggest_float('learning_rate',.02,.25,log=True),
           'subsample':trial.suggest_float('subsample',.7,1),'colsample_bytree':trial.suggest_float('colsample_bytree',.6,1),
           'reg_lambda':trial.suggest_float('reg_lambda',.1,30,log=True)}
        if name=='XGB': p.update(min_child_weight=trial.suggest_int('min_child_weight',1,8),gamma=trial.suggest_float('gamma',0,2),reg_alpha=trial.suggest_float('reg_alpha',1e-8,1,log=True))
        else: p.update(num_leaves=trial.suggest_categorical('num_leaves',[4,8,16]),min_child_samples=trial.suggest_int('min_child_samples',3,15))
        return p
    if name=='CAT': return {'iterations':trial.suggest_int('iterations',20 if small else 100,50 if small else 500,step=10 if small else 100),'depth':trial.suggest_int('depth',2,4 if small else 6),
        'learning_rate':trial.suggest_float('learning_rate',.02,.25,log=True),'l2_leaf_reg':trial.suggest_float('l2_leaf_reg',1,30,log=True)}
    if name=='ANN': return {'hidden_layer_sizes':trial.suggest_categorical('hidden_layer_sizes',['16','32','32_16']),
        'alpha':trial.suggest_float('alpha',1e-4,1,log=True)}
    raise ValueError(name)


### 3.3 每折内部拟合预处理Pipeline

In [ ]:
CALIBRATION_LOG=[]
def build_model(name,params,seed,scheme,y_fit,groups_fit):
    p=dict(params)
    if name=='LR': est=LogisticRegression(**p,max_iter=3000,random_state=seed)
    elif name=='RIDGE': est=RidgeClassifier(**p)
    elif name=='LINEAR_SVC': est=LinearSVC(**p,max_iter=20000,random_state=seed)
    elif name=='SVM': est=SVC(**p,kernel='rbf',probability=True,random_state=seed)
    elif name=='SGD': est=SGDClassifier(**p,loss='log_loss',penalty='elasticnet',max_iter=5000,tol=1e-4,random_state=seed)
    elif name=='KNN': est=KNeighborsClassifier(**p,n_jobs=N_JOBS)
    elif name=='GNB': est=GaussianNB(**p)
    elif name=='LDA': est=LinearDiscriminantAnalysis(**p,solver='lsqr')
    elif name=='QDA': est=QuadraticDiscriminantAnalysis(**p)
    elif name=='GPC':
        kernel_name=p.pop('kernel');length=p.pop('length_scale')
        kernel=RBF(length_scale=length,length_scale_bounds='fixed') if kernel_name=='rbf' else Matern(length_scale=length,length_scale_bounds='fixed',nu=1.5)
        est=GaussianProcessClassifier(kernel=kernel,optimizer=None,max_iter_predict=40 if CFG['small_models'] else 100,random_state=seed,n_jobs=N_JOBS)
    elif name=='DT': est=DecisionTreeClassifier(**p,random_state=seed)
    elif name=='RF': est=RandomForestClassifier(**p,random_state=seed,n_jobs=N_JOBS)
    elif name=='ET': est=ExtraTreesClassifier(**p,random_state=seed,n_jobs=N_JOBS)
    elif name=='GBDT': est=GradientBoostingClassifier(**p,random_state=seed)
    elif name=='HGB': est=HistGradientBoostingClassifier(**p,early_stopping=False,random_state=seed)
    elif name in ['ADA','BAG']:
        depth=p.pop('max_depth');base=DecisionTreeClassifier(max_depth=depth,min_samples_leaf=2,random_state=seed)
        est=(AdaBoostClassifier(estimator=base,**p,random_state=seed) if name=='ADA' else BaggingClassifier(estimator=base,**p,random_state=seed,n_jobs=N_JOBS))
    elif name=='XGB': est=XGBClassifier(**p,objective='multi:softprob',num_class=5,eval_metric='mlogloss',tree_method='hist',n_jobs=N_JOBS,random_state=seed)
    elif name=='LGBM': est=LGBMClassifier(**p,objective='multiclass',num_class=5,subsample_freq=1,verbosity=-1,n_jobs=N_JOBS,random_state=seed)
    elif name=='CAT': est=CatBoostClassifier(**p,loss_function='MultiClass',verbose=False,allow_writing_files=False,thread_count=1,random_seed=seed)
    elif name=='ANN':
        p['hidden_layer_sizes']=tuple(map(int,p['hidden_layer_sizes'].split('_')))
        est=MLPClassifier(**p,solver='lbfgs',max_iter=500 if CFG['small_models'] else 2500,max_fun=20000,random_state=seed)
    else: raise ValueError(name)
    steps=[('imputer',SimpleImputer(strategy='median',keep_empty_features=True))]
    if SCALE_MAP[name]=='standard': steps.append(('scaler',StandardScaler()))
    steps.append(('classifier',est));pipeline=Pipeline(steps)
    if name in ['RIDGE','LINEAR_SVC']:
        calib_scheme=scheme
        if scheme=='cycle_grouped' and cycle_table(y_fit,groups_fit).y.value_counts().min()<2:
            calib_scheme='row_random'
            CALIBRATION_LOG.append(dict(model=name,fit_n=len(y_fit),reason='one cycle per class in fit subset; internal row calibration only'))
        cv=make_folds(y_fit,groups_fit,calib_scheme,2,seed)
        return CalibratedClassifierCV(estimator=pipeline,method='sigmoid',cv=cv,n_jobs=1)
    return pipeline
def aligned_probabilities(model,x):
    raw=np.asarray(model.predict_proba(x),dtype=float)
    p=np.zeros((len(x),5),dtype=float)
    for col,c in enumerate(model.classes_):p[:,int(c)]=raw[:,col]
    p=np.clip(p,1e-15,1);p/=p.sum(axis=1,keepdims=True)
    assert np.isfinite(p).all() and np.allclose(p.sum(axis=1),1)
    return p


### 3.4 分类、概率和角度误差指标

In [ ]:
def classification_metrics(y_true,p):
    pred=p.argmax(axis=1);one=np.eye(5)[y_true]
    cm=skm.confusion_matrix(y_true,pred,labels=np.arange(5))
    result=dict(accuracy=skm.accuracy_score(y_true,pred),balanced_accuracy=skm.balanced_accuracy_score(y_true,pred),
        mcc=skm.matthews_corrcoef(y_true,pred),cohen_kappa=skm.cohen_kappa_score(y_true,pred),
        quadratic_kappa=skm.cohen_kappa_score(y_true,pred,weights='quadratic'),
        log_loss=skm.log_loss(y_true,p,labels=np.arange(5)),brier_multiclass=float(np.mean(np.sum((p-one)**2,axis=1))),n=len(y_true))
    errors=CLASS_ANGLES[pred]-CLASS_ANGLES[y_true]
    result.update(angle_mae_deg=float(np.mean(np.abs(errors))),angle_rmse_deg=float(np.sqrt(np.mean(errors**2))),
        error_ge60deg_rate=float(np.mean(np.abs(errors)>=60)))
    for avg in ['macro','weighted','micro']:
        for key,fn in [('precision',skm.precision_score),('recall',skm.recall_score),('f1',skm.f1_score)]:
            result[f'{key}_{avg}']=fn(y_true,pred,labels=np.arange(5),average=avg,zero_division=0)
    rows=[]
    for c in range(5):
        tp=int(cm[c,c]);fn=int(cm[c,:].sum()-tp);fp=int(cm[:,c].sum()-tp);tn=int(cm.sum()-tp-fp-fn)
        b=(y_true==c).astype(int);has_both=len(np.unique(b))==2
        rows.append(dict(angle_deg=int(CLASS_ANGLES[c]),precision=tp/(tp+fp) if tp+fp else 0,
            recall=tp/(tp+fn) if tp+fn else 0,f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,
            specificity=tn/(tn+fp) if tn+fp else np.nan,support=int(b.sum()),tp=tp,fp=fp,fn=fn,tn=tn,
            roc_auc=skm.roc_auc_score(b,p[:,c]) if has_both else np.nan,
            average_precision=skm.average_precision_score(b,p[:,c]) if b.sum() else np.nan))
    pc=pd.DataFrame(rows)
    for key,col in [('auc','roc_auc'),('ap','average_precision')]:
        result[f'{key}_macro']=pc[col].mean() if pc[col].notna().all() else np.nan
        result[f'{key}_weighted']=np.average(pc[col],weights=pc.support) if pc[col].notna().all() else np.nan
    result['auc_micro']=skm.roc_auc_score(one.ravel(),p.ravel())
    result['ap_micro']=skm.average_precision_score(one.ravel(),p.ravel())
    result['auc_ovo_macro']=skm.roc_auc_score(y_true,p,multi_class='ovo',average='macro',labels=np.arange(5))
    result['specificity_macro']=pc.specificity.mean()
    return result,pc
def prediction_frame(indices,p,split,fold=-1):
    frame=data.iloc[indices][['window_id','cycle_id']].reset_index(drop=True).copy()
    frame['row_index']=indices;frame['split']=split;frame['fold']=fold
    frame['y_true']=y[indices];frame['y_pred']=p.argmax(axis=1)
    frame['angle_true']=CLASS_ANGLES[frame.y_true];frame['angle_pred']=CLASS_ANGLES[frame.y_pred]
    for c in range(5):frame[f'p{c}']=p[:,c]
    return frame


## 4. 原绘图模板代码

以下单元逐段保留原模板及来源哈希。模板注册在本Notebook中，后续执行器只做输入适配及明确记录的替换，不从网络或其他Python文件加载绘图函数。

混淆矩阵的原模板本身仍完整保留；执行时仅将计数转换为行百分比，并修改显示格式与色条单位。Taylor代码保留为可选辅助诊断，默认关闭，不用于分类模型排名。

In [ ]:
ORIGINAL_PLOT_SOURCES = {'shap': {'source': '期刊复现：多分类XGBoost模型解释SHAP柱状图与蜂群图组合绘制.html', 'sha256': '1cca34306423bbf43f731d9eb6f88e3b3be4dd18cffc4f00f128e9ceacd5e347', 'blocks': ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']}, 'network': {'source': '期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html', 'sha256': '3cda1903035cfad4f6b569bbdf8b9ce808cfabeb9fc00ffaaafbe351be9b5199', 'blocks': ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']}, 'taylor': {'source': '模型之间的泰勒图比较：从模型指标计算到局部放大 inset 优化.html', 'sha256': '9e65a3659c991510262a42e7683381499197562ccaddcf25641fcf9e579fa3a5', 'blocks': ['', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '', '']}, 'metrics': {'source': '分类.ipynb', 'sha256': '767f065cb4181ff74503ba8f1dd8639ec2c05d913fe0aea2e1c066d5a4e9fa94', 'blocks': ['', '', '', '', '']}, 'optuna': {'source': '基于Optuna的机器学习超参数调优与自定义可视化分析.ipynb', 'sha256': 'dcee70e4f3608c6c785b78ba457a80bf454cb55185ccc69cff41291eba90a2ba', 'blocks': ['', '', '', '', '', '', '', '', '', '']}, 'roc': {'source': '多分类如何绘制ROC曲线--宏平均ROC曲线.ipynb', 'sha256': 'b53e8dad1a032fe255ef7e6f5a474a3c199ec0152bbc26debbc5e3765e26bdda', 'blocks': ['', '', '', '', '', '', '', '']}, 'cvroc': {'source': '期刊配图：交叉验证与ROC曲线相结合的带置信区间可视化.ipynb', 'sha256': '72631a0f830bb1db56235753d827bb4e74ffdc78c541722ca06815e370d88ca7', 'blocks': ['', '', '', '', '', '', '', '', '', '']}}


### 模板 roc 单元 4
来源 多分类如何绘制ROC曲线--宏平均ROC曲线.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['roc']['blocks'][4] = r'''from sklearn.metrics import confusion_matrix
import seaborn as sns
# 输出混淆矩阵
conf_matrix = confusion_matrix(ytest, y_pred)

# 绘制热力图
plt.figure(figsize=(10, 7), dpi=1200)
sns.heatmap(conf_matrix, annot=True, annot_kws={'size':15}, 
            fmt='d', cmap='YlGnBu', cbar_kws={'shrink': 0.75})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Confusion matrix heat map', fontsize=15)
plt.show()'''


### 模板 roc 单元 6
来源 多分类如何绘制ROC曲线--宏平均ROC曲线.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['roc']['blocks'][6] = r'''plt.figure(figsize=(10, 5), dpi=300)
# 使用不同的颜色和线型
plt.plot(rf_FPR[0], rf_TPR[0], color='#1f77b4', linestyle='-', label='Class 1 ROC  AUC={:.4f}'.format(rf_AUC[0]), lw=2)
plt.plot(rf_FPR[1], rf_TPR[1], color='#ff7f0e', linestyle='-', label='Class 2 ROC  AUC={:.4f}'.format(rf_AUC[1]), lw=2)
plt.plot(rf_FPR[2], rf_TPR[2], color='#2ca02c', linestyle='-', label='Class 3 ROC  AUC={:.4f}'.format(rf_AUC[2]), lw=2)
plt.plot(rf_FPR[3], rf_TPR[3], color='#d62728', linestyle='-', label='Class 4 ROC  AUC={:.4f}'.format(rf_AUC[3]), lw=2)
plt.plot(rf_FPR[4], rf_TPR[4], color='#9467bd', linestyle='-', label='Class 5 ROC  AUC={:.4f}'.format(rf_AUC[4]), lw=2)
# 宏平均ROC曲线
plt.plot(rf_FPR_final, rf_TPR_final, color='#000000', linestyle='-', label='Macro Average ROC  AUC={:.4f}'.format(rf_AUC_final), lw=3)
# 45度参考线
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=2, label='45 Degree Reference Line')
plt.xlabel('False Positive Rate (FPR)', fontsize=15)
plt.ylabel('True Positive Rate (TPR)', fontsize=15)
plt.title('Random Forest Classification ROC Curves and AUC', fontsize=18)
plt.grid(linestyle='--', alpha=0.7)
plt.legend(loc='lower right', framealpha=0.9, fontsize=12)
plt.savefig('RF_optimized.pdf', format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 cvroc 单元 2
来源 期刊配图：交叉验证与ROC曲线相结合的带置信区间可视化.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['cvroc']['blocks'][2] = r'''# 绘制总模型和每一折的ROC曲线汇总图
plt.figure(figsize=(10, 8), dpi=1200)

# 绘制每一折的ROC曲线
for fold in roc_data_df['fold'].unique():
    fold_data = roc_data_df[roc_data_df['fold'] == fold]
    plt.plot(fold_data['fpr'], fold_data['tpr'], color='blue', alpha=0.3, lw=1, label=None)  # 每折颜色相同，透明度较高
# 图形细节设置
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)  # 对角线
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate (FPR)', fontsize=14)
plt.ylabel('True Positive Rate (TPR)', fontsize=14)
plt.title('ROC Curve with Total and Each Fold')
plt.grid(alpha=0.3)
plt.savefig("1.pdf", format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 cvroc 单元 3
来源 期刊配图：交叉验证与ROC曲线相结合的带置信区间可视化.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['cvroc']['blocks'][3] = r'''# 动态计算AUC均值和标准差
aucs = roc_data_df.groupby('fold')['roc_auc'].first().values  # 动态提取各折AUC值
mean_auc = np.mean(aucs)  # 计算均值
std_auc = np.std(aucs)    # 计算标准差

# 绘制总模型和每一折的ROC曲线汇总图（带置信区间）
plt.figure(figsize=(10, 8), dpi=1200)

# 插值网格
tprs = []

# 绘制每一折的ROC曲线
for fold in roc_data_df['fold'].unique():
    fold_data = roc_data_df[roc_data_df['fold'] == fold]
    interp_tpr = np.interp(mean_fpr, fold_data['fpr'], fold_data['tpr'])  # 插值
    interp_tpr[0] = 0.0  # 确保起点为0
    tprs.append(interp_tpr)
    plt.plot(fold_data['fpr'], fold_data['tpr'], color='blue', alpha=0.3, lw=1, label=None)

# 计算均值和标准差
mean_tpr = np.mean(tprs, axis=0)
mean_tpr[-1] = 1.0  # 确保终点为1
std_tpr = np.std(tprs, axis=0)

# 绘制总模型的ROC曲线（均值曲线）
plt.plot(mean_fpr, mean_tpr, color='blue', alpha=1.0, lw=2, label=r"Mean ROC (AUC = %.2f $\pm$ %.2f)" % (mean_auc, std_auc))

# 绘制置信区间
tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color='blue', alpha=0.2, label=r"$\pm$ 1 std. dev.")

# 图形细节设置
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1)  # 对角线
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate (FPR)', fontsize=14)
plt.ylabel('True Positive Rate (TPR)', fontsize=14)
plt.title('ROC Curve with Total and Each Fold (with Confidence Interval)')
plt.legend(loc="lower right", fontsize=12)  # 图例显示总模型和置信区间
plt.grid(alpha=0.3)
plt.savefig("2.pdf", format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 cvroc 单元 8
来源 期刊配图：交叉验证与ROC曲线相结合的带置信区间可视化.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['cvroc']['blocks'][8] = r'''# 绘图函数
def plot_model_roc(fold_roc_data, mean_fpr, color, label):
    tprs = []
    aucs = fold_roc_data.groupby('fold')['roc_auc'].first().values
    mean_auc = np.mean(aucs)
    std_auc = np.std(aucs)

    # 绘制每一折的ROC曲线
    for fold in fold_roc_data['fold'].unique():
        fold_data = fold_roc_data[fold_roc_data['fold'] == fold]
        interp_tpr = np.interp(mean_fpr, fold_data['fpr'], fold_data['tpr'])
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        plt.plot(fold_data['fpr'], fold_data['tpr'], color=color, alpha=0.3, lw=1, label=None)

    # 计算均值和标准差
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0)

    # 绘制总模型的ROC曲线（均值曲线）
    plt.plot(mean_fpr, mean_tpr, color=color, lw=2, label=f"{label} (AUC = {mean_auc:.3f} ± {std_auc:.3f})")

    # 绘制置信区间
    tprs_upper = np.minimum(mean_tpr + std_tpr, 1)
    tprs_lower = np.maximum(mean_tpr - std_tpr, 0)
    plt.fill_between(mean_fpr, tprs_lower, tprs_upper, color=color, alpha=0.2)

# 插值网格
mean_fpr = np.linspace(0, 1, 100)

# 绘制ROC曲线
plt.figure(figsize=(12, 8), dpi=1200)

# 插值网格
mean_fpr = np.linspace(0, 1, 100)

# 绘制XGBoost的ROC
plot_model_roc(roc_data_df, mean_fpr, color='blue', label='XGBoost')

# 绘制Random Forest的ROC
plot_model_roc(roc_data_rf, mean_fpr, color='green', label='Random Forest')

# 绘制LightGBM的ROC
plot_model_roc(roc_data_lgb, mean_fpr, color='red', label='LightGBM')

# 绘制Logistic Regression的ROC
plot_model_roc(roc_data_lr, mean_fpr, color='orange', label='Logistic Regression')

# 图形细节设置
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1, label=None)  # 对角线
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate (FPR)', fontsize=14)
plt.ylabel('True Positive Rate (TPR)', fontsize=14)
plt.legend(loc="lower right", frameon=True, fontsize=12)  
plt.grid(alpha=0.3)
plt.savefig("combined_roc_curves_all_models.pdf", format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 metrics 单元 2
来源 分类.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['metrics']['blocks'][2] = r'''metrics = ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]
categories = results_df["Model"].tolist()
angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]  

fig, axs = plt.subplots(2, 3, figsize=(15, 10), subplot_kw=dict(polar=True))
axs = axs.flatten()

for i, metric in enumerate(metrics):
    values = results_df[metric].tolist()
    values += values[:1]  # Close the radar
    axs[i].fill(angles, values, alpha=0.25, label=metric)
    axs[i].plot(angles, values, marker='o')
    axs[i].set_ylim(0, 1.001)
    axs[i].set_xticks(angles[:-1])
    axs[i].set_xticklabels(categories, fontsize=10)
    axs[i].set_title(metric, pad=20)
    axs[i].set_yticklabels([])

fig.delaxes(axs[-1])
plt.savefig("3.pdf", format='pdf', bbox_inches='tight', dpi=1200)
plt.tight_layout()
plt.show()'''


### 模板 metrics 单元 3
来源 分类.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['metrics']['blocks'][3] = r'''from matplotlib.lines import Line2D
from matplotlib import cm

def plot_polar_bars(data, num_metrics, num_models):
    """
    绘制极坐标柱状图，展示各模型在不同指标上的表现，并在顶部标注模型名称。
    
    参数:
    - data: 包含模型得分的 DataFrame，必须包含 'Model' 列和指标列
    - num_metrics: 指标数量
    - num_models: 模型数量
    """
    models = data['Model'].tolist()
    metrics = data.columns[:-1].tolist()  # 指标列

    # 检查输入的指标和模型数量是否正确
    assert len(metrics) == num_metrics, f"Expected {num_metrics} metrics, but got {len(metrics)}"
    assert len(models) == num_models, f"Expected {num_models} models, but got {len(models)}"

    # 设置极坐标图
    angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
    angles += angles[:1]  # 闭合图形

    # 设置颜色
    colors = cm.viridis(np.linspace(0, 1, num_metrics))

    # 图形设置
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    # 固定最外圈半径
    outer_radius = 1.2
    inner_radius = 0.2
    max_value = data.iloc[:, :-1].values.max()

    # 确保数据范围在可绘制范围内
    assert max_value + inner_radius <= outer_radius, \
        f"Data values exceed maximum allowed range (outer_radius = {outer_radius})"

    # 每个柱的宽度
    bar_width = (2 * np.pi / num_metrics) / num_models

    # 绘制柱状图
    for i, metric in enumerate(metrics):
        values = data[metric].tolist()
        angle = angles[i]
        for j, value in enumerate(values):
            offset_angle = angle + j * bar_width
            ax.bar(offset_angle, value, bottom=inner_radius, width=bar_width, label=models[j] if i == 0 else "",
                   alpha=0.6, color=colors[i])

            # 添加顶部标签
            ax.text(offset_angle, inner_radius + value + 0.05, models[j],
                    ha='center', va='center', fontsize=9.5, rotation=0)

    # 设置极坐标轴属性
    ax.set_ylim(0, outer_radius)
    ax.set_yticklabels([])
    ax.set_xticks([])

    # 添加图例
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor=colors[i], markersize=10, label=metrics[i]) 
        for i in range(num_metrics)
    ]
    ax.legend(handles=legend_elements, title='Metrics', bbox_to_anchor=(0.95, 0.9), loc='upper left')
    ax.spines['polar'].set_visible(False)

    plt.tight_layout()
    plt.savefig("4.pdf", format='pdf', bbox_inches='tight', dpi=1200)
    plt.show()

plot_polar_bars(results_df, 5, 6)'''


### 模板 optuna 单元 3
来源 基于Optuna的机器学习超参数调优与自定义可视化分析.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['optuna']['blocks'][3] = r'''# 找到最优试验
best_trial_idx = optimization_history['value'].idxmin()
best_trial_number = optimization_history['trial_number'][best_trial_idx]
best_value = optimization_history['value'][best_trial_idx]
plt.figure(figsize=(10, 6), dpi=1200)
# 绘制非最佳点
plt.plot(
    optimization_history['trial_number'], 
    optimization_history['value'], 
    marker='o', 
    linestyle='-', 
    color='skyblue', 
    alpha=0.4, 
    label='All Trials'
)
# 绘制最佳点
plt.plot(
    best_trial_number, 
    best_value, 
    marker='*', 
    color='darkblue', 
    markersize=15, 
    label=f'Best Trial (#{best_trial_number})'
)
plt.title("Optimization History")
plt.xlabel("Trial Number")
plt.ylabel("MSE")
plt.grid(True)
plt.legend(loc='upper right', bbox_to_anchor=(1, 1), frameon=True)
plt.savefig('1.pdf', format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 optuna 单元 5
来源 基于Optuna的机器学习超参数调优与自定义可视化分析.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['optuna']['blocks'][5] = r'''# 对参数按重要性进行排序，便于图表的展示
Hyperparameter_Importances = Hyperparameter_Importances.sort_values(by='Importance', ascending=False)
# 绘制超参数重要性条形图
plt.figure(figsize=(10, 6), dpi=1200)
bars = plt.barh(Hyperparameter_Importances['Parameter'], Hyperparameter_Importances['Importance'], color='skyblue')
plt.gca().invert_yaxis()  # 反转Y轴，使重要性最高的参数在顶部
# 为每个条形图显示数值
for bar, importance in zip(bars, Hyperparameter_Importances['Importance']):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2, 
             f'{importance:.3f}', va='center')
plt.title("Hyperparameter Importances")
plt.xlabel("Importance")
plt.ylabel("Parameter")
# 设置x轴范围
plt.xlim(0, max(Hyperparameter_Importances['Importance']) * 1.1) 
plt.savefig('2.pdf', format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 optuna 单元 7
来源 基于Optuna的机器学习超参数调优与自定义可视化分析.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['optuna']['blocks'][7] = r'''from mpl_toolkits.mplot3d import Axes3D
Parameter_Combinations['gamma'] = Parameter_Combinations['gamma'].astype(float)
Parameter_Combinations['learning_rate'] = Parameter_Combinations['learning_rate'].astype(float)

plt.figure(figsize=(10, 8))

# 绘制散点图，使用 'value' 进行颜色编码
sc = plt.scatter(
    Parameter_Combinations['gamma'],
    Parameter_Combinations['learning_rate'],
    c=Parameter_Combinations['value'],
    cmap='viridis',
    s=50,            # 设置散点大小
    alpha=0.7        # 设置透明度
)

cbar = plt.colorbar(sc, format='%.3f')
cbar.set_label('Objective Value')
plt.xlabel("gamma")
plt.ylabel("learning_rate")
plt.title("2D Plot of gamma and learning_rate vs Objective Value")
plt.savefig('3.pdf', format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 optuna 单元 8
来源 基于Optuna的机器学习超参数调优与自定义可视化分析.ipynb

In [ ]:
ORIGINAL_PLOT_SOURCES['optuna']['blocks'][8] = r'''# 确保 'colsample_bytree'、'gamma' 和 'learning_rate' 是数值类型
Parameter_Combinations['colsample_bytree'] = Parameter_Combinations['colsample_bytree'].astype(float)
Parameter_Combinations['gamma'] = Parameter_Combinations['gamma'].astype(float)
Parameter_Combinations['learning_rate'] = Parameter_Combinations['learning_rate'].astype(float)
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
x = Parameter_Combinations['colsample_bytree']
y = Parameter_Combinations['gamma']
z = Parameter_Combinations['learning_rate']
c = Parameter_Combinations['value']  # 用目标值作为颜色
sc = ax.scatter(x, y, z, c=c, cmap='viridis', marker='o', alpha=0.7)
cbar = fig.colorbar(sc, ax=ax, format='%.3f')
cbar.set_label('Objective Value')
ax.set_xlabel("colsample_bytree")
ax.set_ylabel("gamma")
ax.set_zlabel("learning_rate")
plt.title("3D Plot of colsample_bytree, gamma, learning_rate vs Objective MSE")
plt.savefig('4.pdf', format='pdf', bbox_inches='tight')
plt.show()'''


### 模板 taylor 单元 7
来源 模型之间的泰勒图比较：从模型指标计算到局部放大 inset 优化.html

In [ ]:
ORIGINAL_PLOT_SOURCES['taylor']['blocks'][7] = r'''# 图像尺寸与输出
FIG_WIDTH = 9.2
FIG_HEIGHT = 6.6
DPI = 300

# 坐标范围
MAX_STD = 1.75
REF_STD = 1.0
# 字体大小
AXIS_LABEL_FONTSIZE = 17
TICK_FONTSIZE = 14
CORR_TICK_FONTSIZE = 14
CORR_LABEL_FONTSIZE = 16
RMSE_LABEL_FONTSIZE = 13
LEGEND_FONTSIZE = 13
# 字体粗细
AXIS_LABEL_FONTWEIGHT = "bold"
TICK_FONTWEIGHT = "bold"
CORR_TICK_FONTWEIGHT = "bold"
CORR_LABEL_FONTWEIGHT = "bold"
RMSE_LABEL_FONTWEIGHT = "bold"
LEGEND_FONTWEIGHT = "bold"
# 点大小
OBS_POINT_SIZE = 75
MODEL_POINT_SIZE = 85
# 点边框线宽
MODEL_POINT_LINEWIDTH = 2.2
# 网格线和辅助线线宽
STD_ARC_LINEWIDTH = 0.7
CORR_RAY_LINEWIDTH = 0.7
OUTER_ARC_LINEWIDTH = 1.1
RMSE_LINEWIDTH = 0.8
REF_STD_LINEWIDTH = 1.0
SPINE_LINEWIDTH = 1.0
# 图例设置
LEGEND_MARKER_SCALE = 1.3
LEGEND_HANDLELENGTH = 1.0
LEGEND_HANDLE_TEXT_PAD = 0.6
LEGEND_BORDER_PAD = 0.7

LEGEND_BBOX_X = 1.14
LEGEND_BBOX_Y = 1.08

# Correlation 标签位置和角度
CORR_LABEL_X = 1.42
CORR_LABEL_Y = 1.32
CORR_LABEL_ROTATION = -50

# 相关系数刻度数字位置
CORR_TICK_RADIUS_FACTOR = 1.045

# RMSE 标签位置
RMSE_LABEL_ANGLE_DEG = 120
# 页面边距
SUBPLOT_LEFT = 0.08
SUBPLOT_RIGHT = 0.72
SUBPLOT_BOTTOM = 0.11
SUBPLOT_TOP = 0.91

def corr_to_xy(std, corr):
    """
    将标准差和相关系数转换为 Taylor 图中的 x, y 坐标。
    """
    corr = np.clip(corr, -1, 1)
    theta = np.arccos(corr)
    x = std * np.cos(theta)
    y = std * np.sin(theta)
    return x, y

def add_taylor_background(ax, ref_std=1.0, max_std=1.75):
    """
    绘制 Taylor diagram 背景。
    """
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0, max_std)
    ax.set_ylim(0, max_std)

    theta = np.linspace(0, np.pi / 2, 500)

    std_levels = np.arange(0.25, max_std + 0.001, 0.25)

    for s in std_levels:
        x = s * np.cos(theta)
        y = s * np.sin(theta)

        ax.plot(
            x,
            y,
            color="#c9d6dc",
            lw=STD_ARC_LINEWIDTH,
            zorder=0
        )

    corr_levels = [
        0.1, 0.2, 0.3, 0.4, 0.5,
        0.6, 0.7, 0.8, 0.9, 0.95, 0.99
    ]

    for c in corr_levels:
        th = np.arccos(c)

        x_ray = [0, max_std * np.cos(th)]
        y_ray = [0, max_std * np.sin(th)]

        ax.plot(
            x_ray,
            y_ray,
            color="#c9d6dc",
            lw=CORR_RAY_LINEWIDTH,
            zorder=0
        )

        r_label = max_std * CORR_TICK_RADIUS_FACTOR
        xt = r_label * np.cos(th)
        yt = r_label * np.sin(th)

        ax.text(
            xt,
            yt,
            f"{c:g}",
            ha="center",
            va="center",
            rotation=np.degrees(th) - 90,
            fontsize=CORR_TICK_FONTSIZE,
            fontweight=CORR_TICK_FONTWEIGHT,
            clip_on=False,
            zorder=10
        )

    x_outer = max_std * np.cos(theta)
    y_outer = max_std * np.sin(theta)

    ax.plot(
        x_outer,
        y_outer,
        color="black",
        lw=OUTER_ARC_LINEWIDTH
    )

    rmse_levels = [0.3, 0.6, 0.9, 1.2, 1.5, 1.8]
    phi = np.linspace(0, 2 * np.pi, 1000)

    for e in rmse_levels:
        x_rmse = ref_std + e * np.cos(phi)
        y_rmse = e * np.sin(phi)

        mask = (
            (x_rmse >= 0) &
            (y_rmse >= 0) &
            (x_rmse**2 + y_rmse**2 <= max_std**2 + 1e-8)
        )

        ax.plot(
            x_rmse[mask],
            y_rmse[mask],
            ls="--",
            color="#9bb2bd",
            lw=RMSE_LINEWIDTH,
            zorder=0
        )

        angle = np.deg2rad(RMSE_LABEL_ANGLE_DEG)
        xt = ref_std + e * np.cos(angle)
        yt = e * np.sin(angle)

        if 0 <= xt <= max_std and 0 <= yt <= max_std:
            ax.text(
                xt,
                yt,
                f"{e:.1f}",
                color="#a5a5a5",
                fontsize=RMSE_LABEL_FONTSIZE,
                fontweight=RMSE_LABEL_FONTWEIGHT,
                rotation=25,
                ha="center",
                va="center"
            )

    x_ref = ref_std * np.cos(theta)
    y_ref = ref_std * np.sin(theta)

    ax.plot(
        x_ref,
        y_ref,
        color="red",
        lw=REF_STD_LINEWIDTH,
        ls="--"
    )

    ax.set_xlabel(
        "Standard deviation",
        fontsize=AXIS_LABEL_FONTSIZE,
        fontweight=AXIS_LABEL_FONTWEIGHT
    )

    ticks = np.arange(0, max_std + 0.001, 0.25)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)

    tick_labels = []
    for t in ticks:
        if abs(t - 0) < 1e-10:
            tick_labels.append("0")
        elif abs(t - 1) < 1e-10:
            tick_labels.append("1")
        else:
            tick_labels.append(f"{t:.2f}")

    ax.set_xticklabels(
        tick_labels,
        fontsize=TICK_FONTSIZE,
        fontweight=TICK_FONTWEIGHT
    )

    ax.set_yticklabels(
        tick_labels,
        fontsize=TICK_FONTSIZE,
        fontweight=TICK_FONTWEIGHT
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.spines["left"].set_linewidth(SPINE_LINEWIDTH)
    ax.spines["bottom"].set_linewidth(SPINE_LINEWIDTH)

    ax.tick_params(
        axis="both",
        which="major",
        width=SPINE_LINEWIDTH,
        length=4
    )

    ax.text(
        CORR_LABEL_X,
        CORR_LABEL_Y,
        "Correlation",
        fontsize=CORR_LABEL_FONTSIZE,
        fontweight=CORR_LABEL_FONTWEIGHT,
        rotation=CORR_LABEL_ROTATION,
        ha="center",
        va="center",
        clip_on=False,
        zorder=10
    )


def plot_taylor_diagram(
    models,
    stds,
    corrs,
    colors,
    save_pdf_path="taylor_diagram.pdf",
    show=True,
    ref_std=REF_STD,
    max_std=MAX_STD,
    fig_width=FIG_WIDTH,
    fig_height=FIG_HEIGHT,
    dpi=DPI
):
    """
    绘制 Taylor 图，并保存为 PDF（不包含局部放大图）。

    参数
    ----------
    models : list[str]
        模型名称列表
    stds : array-like
        各模型标准差
    corrs : array-like
        各模型相关系数
    colors : list[str]
        各模型颜色
    save_pdf_path : str
        PDF 保存路径
    show : bool
        是否显示图像
    ref_std : float
        参考标准差
    max_std : float
        Taylor 图最大标准差范围
    fig_width : float
        图宽
    fig_height : float
        图高
    dpi : int
        分辨率
    """
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi)

    add_taylor_background(
        ax,
        ref_std=ref_std,
        max_std=max_std
    )

    ax.scatter(
        ref_std,
        0,
        s=OBS_POINT_SIZE,
        c="black",
        edgecolors="black",
        label="Observed",
        zorder=5
    )

    for model, std, corr, color in zip(models, stds, corrs, colors):
        x, y = corr_to_xy(std, corr)

        ax.scatter(
            x,
            y,
            s=MODEL_POINT_SIZE,
            facecolors="none",
            edgecolors=color,
            linewidths=MODEL_POINT_LINEWIDTH,
            label=model,
            zorder=6
        )

    legend = ax.legend(
        loc="upper left",
        bbox_to_anchor=(LEGEND_BBOX_X, LEGEND_BBOX_Y),
        frameon=True,
        fancybox=False,
        edgecolor="#dddddd",
        fontsize=LEGEND_FONTSIZE,
        markerscale=LEGEND_MARKER_SCALE,
        handlelength=LEGEND_HANDLELENGTH,
        handletextpad=LEGEND_HANDLE_TEXT_PAD,
        borderpad=LEGEND_BORDER_PAD
    )

    for text in legend.get_texts():
        text.set_fontweight(LEGEND_FONTWEIGHT)

    plt.subplots_adjust(
        left=SUBPLOT_LEFT,
        right=SUBPLOT_RIGHT,
        bottom=SUBPLOT_BOTTOM,
        top=SUBPLOT_TOP
    )

    fig.savefig(save_pdf_path, format="pdf", bbox_inches="tight",dpi=1200)
    print(f"PDF 已保存到: {save_pdf_path}")

    if show:
        plt.show()
    else:
        plt.close(fig)
'''


### 模板 taylor 单元 8
来源 模型之间的泰勒图比较：从模型指标计算到局部放大 inset 优化.html

In [ ]:
ORIGINAL_PLOT_SOURCES['taylor']['blocks'][8] = r'''plot_taylor_diagram(
    models=models,
    stds=stds,
    corrs=corrs,
    colors=colors,
    save_pdf_path="taylor_diagram.pdf",
    show=True
)
'''


### 模板 taylor 单元 10
来源 模型之间的泰勒图比较：从模型指标计算到局部放大 inset 优化.html

In [ ]:
ORIGINAL_PLOT_SOURCES['taylor']['blocks'][10] = r'''from matplotlib.patches import Rectangle, ConnectionPatch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# 图像尺寸与输出
FIG_WIDTH = 9.2
FIG_HEIGHT = 6.6
DPI = 300

# 坐标范围
MAX_STD = 1.75
REF_STD = 1.0

# 字体大小
AXIS_LABEL_FONTSIZE = 17
TICK_FONTSIZE = 14
CORR_TICK_FONTSIZE = 14
CORR_LABEL_FONTSIZE = 16
RMSE_LABEL_FONTSIZE = 13
LEGEND_FONTSIZE = 13

# 字体粗细
AXIS_LABEL_FONTWEIGHT = "bold"
TICK_FONTWEIGHT = "bold"
CORR_TICK_FONTWEIGHT = "bold"
CORR_LABEL_FONTWEIGHT = "bold"
RMSE_LABEL_FONTWEIGHT = "bold"
LEGEND_FONTWEIGHT = "bold"

# 点大小
OBS_POINT_SIZE = 75
MODEL_POINT_SIZE = 85
INSET_POINT_SIZE = 170

# 点边框线宽
MODEL_POINT_LINEWIDTH = 2.2
INSET_POINT_LINEWIDTH = 3.0

# 网格线和辅助线线宽
STD_ARC_LINEWIDTH = 0.7
CORR_RAY_LINEWIDTH = 0.7
OUTER_ARC_LINEWIDTH = 1.1
RMSE_LINEWIDTH = 0.8
REF_STD_LINEWIDTH = 1.0
SPINE_LINEWIDTH = 1.0

# 图例设置
LEGEND_MARKER_SCALE = 1.3
LEGEND_HANDLELENGTH = 1.0
LEGEND_HANDLE_TEXT_PAD = 0.6
LEGEND_BORDER_PAD = 0.7
LEGEND_BBOX_X = 1.14
LEGEND_BBOX_Y = 1.08

# Correlation 标签位置和角度
CORR_LABEL_X = 1.42
CORR_LABEL_Y = 1.32
CORR_LABEL_ROTATION = -50

# 相关系数刻度数字位置
CORR_TICK_RADIUS_FACTOR = 1.045

# RMSE 标签位置
RMSE_LABEL_ANGLE_DEG = 120

# 页面边距
SUBPLOT_LEFT = 0.08
SUBPLOT_RIGHT = 0.72
SUBPLOT_BOTTOM = 0.11
SUBPLOT_TOP = 0.91

# inset 参数
RECT_LINEWIDTH = 1.2
INSET_SPINE_LINEWIDTH = 1.2
INSET_GRID_LINEWIDTH = 0.6

CONNECT_LINE_COLOR = "gray"
CONNECT_LINEWIDTH = 0.9
CONNECT_LINE_ALPHA = 0.75
CONNECT_LINE_STYLE = "-"
CONNECT_LINE_ZORDER = 3

def corr_to_xy(std, corr):
    """
    将标准差和相关系数转换为 Taylor 图中的 x, y 坐标
    """
    corr = np.clip(corr, -1, 1)
    theta = np.arccos(corr)
    x = std * np.cos(theta)
    y = std * np.sin(theta)
    return x, y

def calc_taylor_stats(y_obs, y_pred, normalize=True):
    """
    根据观测值和预测值计算 Taylor 图所需指标
    返回: std, corr, centered_rmse
    """
    y_obs = np.asarray(y_obs)
    y_pred = np.asarray(y_pred)

    obs_std = np.std(y_obs, ddof=1)
    pred_std = np.std(y_pred, ddof=1)
    corr = np.corrcoef(y_obs, y_pred)[0, 1]

    centered_rmse = np.sqrt(
        pred_std ** 2 + obs_std ** 2 - 2 * pred_std * obs_std * corr
    )

    if normalize:
        std = pred_std / obs_std
        centered_rmse = centered_rmse / obs_std
    else:
        std = pred_std

    return std, corr, centered_rmse

def add_taylor_background(ax, ref_std=1.0, max_std=1.75):
    """
    主图 Taylor 背景
    """
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0, max_std)
    ax.set_ylim(0, max_std)

    theta = np.linspace(0, np.pi / 2, 500)

    # 标准差圆弧
    std_levels = np.arange(0.25, max_std + 0.001, 0.25)
    for s in std_levels:
        x = s * np.cos(theta)
        y = s * np.sin(theta)
        ax.plot(x, y, color="#c9d6dc", lw=STD_ARC_LINEWIDTH, zorder=0)

    # 相关系数射线与文字
    corr_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6,
                   0.7, 0.8, 0.9, 0.95, 0.99]

    for c in corr_levels:
        th = np.arccos(c)

        x_ray = [0, max_std * np.cos(th)]
        y_ray = [0, max_std * np.sin(th)]
        ax.plot(
            x_ray,
            y_ray,
            color="#c9d6dc",
            lw=CORR_RAY_LINEWIDTH,
            zorder=0
        )

        r_label = max_std * CORR_TICK_RADIUS_FACTOR
        xt = r_label * np.cos(th)
        yt = r_label * np.sin(th)

        ax.text(
            xt,
            yt,
            f"{c:g}",
            ha="center",
            va="center",
            rotation=np.degrees(th) - 90,
            fontsize=CORR_TICK_FONTSIZE,
            fontweight=CORR_TICK_FONTWEIGHT,
            clip_on=False,
            zorder=10
        )

    # 外边界圆弧
    x_outer = max_std * np.cos(theta)
    y_outer = max_std * np.sin(theta)
    ax.plot(x_outer, y_outer, color="black", lw=OUTER_ARC_LINEWIDTH)

    # RMSE 等值线
    rmse_levels = [0.3, 0.6, 0.9, 1.2, 1.5, 1.8]
    phi = np.linspace(0, 2 * np.pi, 1000)

    for e in rmse_levels:
        x_rmse = ref_std + e * np.cos(phi)
        y_rmse = e * np.sin(phi)

        mask = (
            (x_rmse >= 0) &
            (y_rmse >= 0) &
            (x_rmse ** 2 + y_rmse ** 2 <= max_std ** 2 + 1e-8)
        )

        ax.plot(
            x_rmse[mask],
            y_rmse[mask],
            ls="--",
            color="#9bb2bd",
            lw=RMSE_LINEWIDTH,
            zorder=0
        )

        angle = np.deg2rad(RMSE_LABEL_ANGLE_DEG)
        xt = ref_std + e * np.cos(angle)
        yt = e * np.sin(angle)

        if 0 <= xt <= max_std and 0 <= yt <= max_std:
            ax.text(
                xt,
                yt,
                f"{e:.1f}",
                color="#a5a5a5",
                fontsize=RMSE_LABEL_FONTSIZE,
                fontweight=RMSE_LABEL_FONTWEIGHT,
                rotation=25,
                ha="center",
                va="center"
            )

    # 参考标准差圆弧
    x_ref = ref_std * np.cos(theta)
    y_ref = ref_std * np.sin(theta)
    ax.plot(x_ref, y_ref, color="red", lw=REF_STD_LINEWIDTH, ls="--")

    # 坐标轴设置
    ax.set_xlabel(
        "Standard deviation",
        fontsize=AXIS_LABEL_FONTSIZE,
        fontweight=AXIS_LABEL_FONTWEIGHT
    )

    ticks = np.arange(0, max_std + 0.001, 0.25)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)

    tick_labels = []
    for t in ticks:
        if abs(t - 0) < 1e-10:
            tick_labels.append("0")
        elif abs(t - 1) < 1e-10:
            tick_labels.append("1")
        else:
            tick_labels.append(f"{t:.2f}")

    ax.set_xticklabels(
        tick_labels,
        fontsize=TICK_FONTSIZE,
        fontweight=TICK_FONTWEIGHT
    )
    ax.set_yticklabels(
        tick_labels,
        fontsize=TICK_FONTSIZE,
        fontweight=TICK_FONTWEIGHT
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(SPINE_LINEWIDTH)
    ax.spines["bottom"].set_linewidth(SPINE_LINEWIDTH)

    ax.tick_params(
        axis="both",
        which="major",
        width=SPINE_LINEWIDTH,
        length=4
    )

    ax.text(
        CORR_LABEL_X,
        CORR_LABEL_Y,
        "Correlation",
        fontsize=CORR_LABEL_FONTSIZE,
        fontweight=CORR_LABEL_FONTWEIGHT,
        rotation=CORR_LABEL_ROTATION,
        ha="center",
        va="center",
        clip_on=False,
        zorder=10
    )

def add_taylor_background_inset(ax, ref_std=1.0, max_std=1.75):
    """
    inset 小图 Taylor 背景线
    只画线，不画文字标签
    """
    theta = np.linspace(0, np.pi / 2, 500)

    # 标准差圆弧
    std_levels = np.arange(0.25, max_std + 0.001, 0.25)
    for s in std_levels:
        x = s * np.cos(theta)
        y = s * np.sin(theta)
        ax.plot(x, y, color="#c9d6dc", lw=0.55, zorder=0)

    # 相关系数射线
    corr_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6,
                   0.7, 0.8, 0.9, 0.95, 0.99]

    for c in corr_levels:
        th = np.arccos(c)
        x_ray = [0, max_std * np.cos(th)]
        y_ray = [0, max_std * np.sin(th)]
        ax.plot(x_ray, y_ray, color="#c9d6dc", lw=0.55, zorder=0)

    # 外边界圆弧
    x_outer = max_std * np.cos(theta)
    y_outer = max_std * np.sin(theta)
    ax.plot(x_outer, y_outer, color="black", lw=0.8, zorder=0)

    # RMSE 等值线
    rmse_levels = [0.3, 0.6, 0.9, 1.2, 1.5, 1.8]
    phi = np.linspace(0, 2 * np.pi, 1000)

    for e in rmse_levels:
        x_rmse = ref_std + e * np.cos(phi)
        y_rmse = e * np.sin(phi)

        mask = (
            (x_rmse >= 0) &
            (y_rmse >= 0) &
            (x_rmse ** 2 + y_rmse ** 2 <= max_std ** 2 + 1e-8)
        )

        ax.plot(
            x_rmse[mask],
            y_rmse[mask],
            ls="--",
            color="#9bb2bd",
            lw=0.6,
            zorder=0
        )

    # 参考标准差红色虚线
    x_ref = ref_std * np.cos(theta)
    y_ref = ref_std * np.sin(theta)
    ax.plot(x_ref, y_ref, color="red", lw=0.8, ls="--", zorder=0)

    ax.set_xticks([])
    ax.set_yticks([])
    
def auto_zoom_region(
    xs,
    ys,
    max_std=1.75,
    keep_ratio=0.8,
    padding_ratio=0.18,
    min_width=0.10,
    min_height=0.08
):
    """
    根据模型点聚集范围自动生成放大框
    """
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    n = len(xs)

    if n == 0:
        raise ValueError("xs 和 ys 不能为空")

    if n <= 3:
        x_sel = xs
        y_sel = ys
    else:
        x_med = np.median(xs)
        y_med = np.median(ys)

        d = np.sqrt((xs - x_med) ** 2 + (ys - y_med) ** 2)
        order = np.argsort(d)

        n_keep = max(3, int(np.ceil(n * keep_ratio)))
        idx = order[:n_keep]

        x_sel = xs[idx]
        y_sel = ys[idx]

    x_min = np.min(x_sel)
    x_max = np.max(x_sel)
    y_min = np.min(y_sel)
    y_max = np.max(y_sel)

    width = max(x_max - x_min, min_width)
    height = max(y_max - y_min, min_height)

    pad_x = width * padding_ratio
    pad_y = height * padding_ratio

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2

    zoom_xmin = x_center - width / 2 - pad_x
    zoom_xmax = x_center + width / 2 + pad_x
    zoom_ymin = y_center - height / 2 - pad_y
    zoom_ymax = y_center + height / 2 + pad_y

    zoom_xmin = max(0, zoom_xmin)
    zoom_ymin = max(0, zoom_ymin)
    zoom_xmax = min(max_std, zoom_xmax)
    zoom_ymax = min(max_std, zoom_ymax)

    return zoom_xmin, zoom_xmax, zoom_ymin, zoom_ymax

def choose_best_inset_position(
    xs,
    ys,
    zoom_xmin,
    zoom_xmax,
    zoom_ymin,
    zoom_ymax,
    max_std=1.75,
    inset_w_frac=0.24,
    inset_h_frac=0.20,
    margin_frac=0.06
):
    """
    自动选择 inset 在主图内部的位置，尽量避免贴边和重叠
    返回: best_x0, best_y0, best_w, best_h, best_name
    """

    xs = np.asarray(xs)
    ys = np.asarray(ys)

    candidates = [
        ("middle_left", margin_frac, 0.58),
        ("upper_mid", 0.34, 1 - margin_frac - inset_h_frac),
        ("middle_right", 1 - margin_frac - inset_w_frac, 0.58),
        ("lower_left", margin_frac + 0.02, 0.32),
        ("lower_mid", 0.36, 0.28),
    ]

    best_score = None
    best_pos = None
    rect_area = max(
        (zoom_xmax - zoom_xmin) * (zoom_ymax - zoom_ymin),
        1e-12
    )

    for name, x0a, y0a in candidates:
        x0 = x0a * max_std
        x1 = (x0a + inset_w_frac) * max_std
        y0 = y0a * max_std
        y1 = (y0a + inset_h_frac) * max_std

        # 与放大框重叠面积
        overlap_x = max(0, min(x1, zoom_xmax) - max(x0, zoom_xmin))
        overlap_y = max(0, min(y1, zoom_ymax) - max(y0, zoom_ymin))
        overlap_area = overlap_x * overlap_y

        # 覆盖模型点数量
        points_inside = np.sum(
            (xs >= x0) & (xs <= x1) &
            (ys >= y0) & (ys <= y1)
        )

        # 距离边缘太近惩罚
        edge_penalty = 0

        if (
            x0a < 0.04 or
            y0a < 0.04 or
            (x0a + inset_w_frac) > 0.96 or
            (y0a + inset_h_frac) > 0.96
        ):
            edge_penalty += 100

        # 靠近左上圆弧区域额外惩罚
        upper_left_penalty = 0

        if x0a < 0.18 and y0a > 0.70:
            upper_left_penalty += 40

        score = (
            overlap_area / rect_area * 100
            + points_inside * 8
            + edge_penalty
            + upper_left_penalty
        )

        if best_score is None or score < best_score:
            best_score = score
            best_pos = (x0a, y0a, inset_w_frac, inset_h_frac, name)

    return best_pos

def get_taylor_params_from_models(
    model_dict,
    X,
    y_true,
    color_dict=None,
    normalize=True
):
    """
    从训练好的模型字典中自动生成 Taylor 图参数
    """
    models = []
    stds = []
    corrs = []
    colors = []

    if color_dict is None:
        default_colors = [
            "#7ec9f5", "#ffa06a", "#78e66f", "#ef7baa", "#b6a0ff",
            "#c9b8bb", "#e996ff", "#b8c7c9", "#dce55c", "#76f5d4"
        ]

        color_dict = {}

        for i, name in enumerate(model_dict.keys()):
            color_dict[name] = default_colors[i % len(default_colors)]

    for name, model in model_dict.items():
        y_pred = model.predict(X)
        std, corr, _ = calc_taylor_stats(
            y_true,
            y_pred,
            normalize=normalize
        )

        models.append(name)
        stds.append(std)
        corrs.append(corr)
        colors.append(color_dict.get(name, "#333333"))

    return models, np.array(stds), np.array(corrs), colors

def plot_taylor_diagram(
    models,
    stds,
    corrs,
    colors,
    save_pdf_path="taylor_diagram.pdf",
    show=True,
    ref_std=REF_STD,
    max_std=MAX_STD,
    fig_width=FIG_WIDTH,
    fig_height=FIG_HEIGHT,
    dpi=DPI,
    add_inset=True,
    auto_zoom=True,
    keep_ratio=0.8,
    padding_ratio=0.18,
    zoom_xmin=None,
    zoom_xmax=None,
    zoom_ymin=None,
    zoom_ymax=None,

    # inset 尺寸
    inset_w_frac=0.24,
    inset_h_frac=0.20,
    inset_margin_frac=0.06,

    # 新增：inset 位置控制
    # "auto"：自动位置
    # "manual"：手动位置，使用 inset_bbox=(x0, y0, w, h)
    inset_position="auto",
    inset_bbox=None
):
    """
    绘制 Taylor 图，并可自动或手动控制 inset 子图位置。

    inset_position:
        "auto"   : 自动选择 inset 位置
        "manual" : 手动指定 inset 位置

    inset_bbox:
        当 inset_position="manual" 时生效。
        格式为 (x0, y0, w, h)，均为 axes fraction 坐标。
        例如：
        inset_bbox=(0.50, 0.44, 0.25, 0.20)
    """
    fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi)

    add_taylor_background(ax, ref_std=ref_std, max_std=max_std)

    # 观测点
    ax.scatter(
        ref_std,
        0,
        s=OBS_POINT_SIZE,
        c="black",
        edgecolors="black",
        label="Observed",
        zorder=5
    )

    # 模型点
    xy_points = []
    xs = []
    ys = []

    for model, std, corr, color in zip(models, stds, corrs, colors):
        x, y = corr_to_xy(std, corr)
        xy_points.append((model, x, y, color))
        xs.append(x)
        ys.append(y)

        ax.scatter(
            x,
            y,
            s=MODEL_POINT_SIZE,
            facecolors="none",
            edgecolors=color,
            linewidths=MODEL_POINT_LINEWIDTH,
            label=model,
            zorder=6
        )

    xs = np.array(xs)
    ys = np.array(ys)

    # 图例
    legend = ax.legend(
        loc="upper left",
        bbox_to_anchor=(LEGEND_BBOX_X, LEGEND_BBOX_Y),
        frameon=True,
        fancybox=False,
        edgecolor="#dddddd",
        fontsize=LEGEND_FONTSIZE,
        markerscale=LEGEND_MARKER_SCALE,
        handlelength=LEGEND_HANDLELENGTH,
        handletextpad=LEGEND_HANDLE_TEXT_PAD,
        borderpad=LEGEND_BORDER_PAD
    )

    for text in legend.get_texts():
        text.set_fontweight(LEGEND_FONTWEIGHT)

    if add_inset:
        if auto_zoom:
            zoom_xmin, zoom_xmax, zoom_ymin, zoom_ymax = auto_zoom_region(
                xs=xs,
                ys=ys,
                max_std=max_std,
                keep_ratio=keep_ratio,
                padding_ratio=padding_ratio
            )
        else:
            if None in [zoom_xmin, zoom_xmax, zoom_ymin, zoom_ymax]:
                raise ValueError(
                    "当 auto_zoom=False 时，必须手动提供 "
                    "zoom_xmin/zoom_xmax/zoom_ymin/zoom_ymax"
                )

        # 主图上的灰色放大框
        rect = Rectangle(
            (zoom_xmin, zoom_ymin),
            zoom_xmax - zoom_xmin,
            zoom_ymax - zoom_ymin,
            fill=False,
            edgecolor="gray",
            linewidth=RECT_LINEWIDTH,
            zorder=4
        )
        ax.add_patch(rect)

        if inset_position == "auto":
            best_x0, best_y0, best_w, best_h, best_name = (
                choose_best_inset_position(
                    xs=xs,
                    ys=ys,
                    zoom_xmin=zoom_xmin,
                    zoom_xmax=zoom_xmax,
                    zoom_ymin=zoom_ymin,
                    zoom_ymax=zoom_ymax,
                    max_std=max_std,
                    inset_w_frac=inset_w_frac,
                    inset_h_frac=inset_h_frac,
                    margin_frac=inset_margin_frac
                )
            )

        elif inset_position == "manual":
            if inset_bbox is None:
                raise ValueError(
                    "当 inset_position='manual' 时，必须提供 "
                    "inset_bbox=(x0, y0, w, h)"
                )

            if len(inset_bbox) != 4:
                raise ValueError(
                    "inset_bbox 必须是长度为 4 的元组，"
                    "例如 inset_bbox=(0.50, 0.44, 0.25, 0.20)"
                )

            best_x0, best_y0, best_w, best_h = inset_bbox
            best_name = "manual"

        else:
            raise ValueError(
                "inset_position 只能是 'auto' 或 'manual'"
            )

        axins = inset_axes(
            ax,
            width="100%",
            height="100%",
            bbox_to_anchor=(best_x0, best_y0, best_w, best_h),
            bbox_transform=ax.transAxes,
            loc="lower left",
            borderpad=0
        )

        axins.set_xlim(zoom_xmin, zoom_xmax)
        axins.set_ylim(zoom_ymin, zoom_ymax)
        axins.set_aspect("equal")

        # inset 背景线
        add_taylor_background_inset(
            axins,
            ref_std=ref_std,
            max_std=max_std
        )

        # 重画模型点
        for model, x, y, color in xy_points:
            axins.scatter(
                x,
                y,
                s=INSET_POINT_SIZE,
                facecolors="none",
                edgecolors=color,
                linewidths=INSET_POINT_LINEWIDTH,
                zorder=6
            )

        axins.set_xticks([])
        axins.set_yticks([])

        for spine in axins.spines.values():
            spine.set_edgecolor("gray")
            spine.set_linewidth(INSET_SPINE_LINEWIDTH)

        # 连接线
        con1 = ConnectionPatch(
            xyA=(zoom_xmin, zoom_ymax),
            coordsA="data",
            axesA=ax,
            xyB=(0, 0),
            coordsB="axes fraction",
            axesB=axins,
            color=CONNECT_LINE_COLOR,
            lw=CONNECT_LINEWIDTH,
            linestyle=CONNECT_LINE_STYLE,
            alpha=CONNECT_LINE_ALPHA,
            zorder=CONNECT_LINE_ZORDER,
            clip_on=False
        )

        con2 = ConnectionPatch(
            xyA=(zoom_xmax, zoom_ymax),
            coordsA="data",
            axesA=ax,
            xyB=(1, 0),
            coordsB="axes fraction",
            axesB=axins,
            color=CONNECT_LINE_COLOR,
            lw=CONNECT_LINEWIDTH,
            linestyle=CONNECT_LINE_STYLE,
            alpha=CONNECT_LINE_ALPHA,
            zorder=CONNECT_LINE_ZORDER,
            clip_on=False
        )

        fig.add_artist(con1)
        fig.add_artist(con2)

        print("自动放大框范围：")
        print(f"zoom_xmin = {zoom_xmin:.4f}")
        print(f"zoom_xmax = {zoom_xmax:.4f}")
        print(f"zoom_ymin = {zoom_ymin:.4f}")
        print(f"zoom_ymax = {zoom_ymax:.4f}")
        print(f"inset position = {best_name}")
        print(
            f"inset bbox = "
            f"({best_x0:.3f}, {best_y0:.3f}, "
            f"{best_w:.3f}, {best_h:.3f})"
        )

    plt.subplots_adjust(
        left=SUBPLOT_LEFT,
        right=SUBPLOT_RIGHT,
        bottom=SUBPLOT_BOTTOM,
        top=SUBPLOT_TOP
    )

    fig.savefig(save_pdf_path, format="pdf", bbox_inches="tight",dpi=1200)
    print(f"PDF 已保存到: {save_pdf_path}")

    if show:
        plt.show()
    else:
        plt.close(fig)
'''


### 模板 taylor 单元 11
来源 模型之间的泰勒图比较：从模型指标计算到局部放大 inset 优化.html

In [ ]:
ORIGINAL_PLOT_SOURCES['taylor']['blocks'][11] = r'''plot_taylor_diagram(
    # 模型名称列表
    # ["LGB", "XGB", "RF", "KNN", "DT", "ET"]
    models=models,
    # 模型标准差列表
    # std 越接近 1，说明模型波动幅度越接近观测值
    stds=stds,
    # 模型相关系数列表
    # corr 越接近 1，说明模型与观测值趋势越一致
    corrs=corrs,
    # 模型颜色列表
    colors=colors,
    # PDF 保存路径
    save_pdf_path="taylor_from_models_1.pdf",
    # 是否显示图像
    show=True,
    # 是否添加 inset 放大子图
    # add_inset=True  ：添加局部放大图
    # add_inset=False ：不添加局部放大图
    add_inset=True,
    # 是否自动计算放大区域
    auto_zoom=True,
    # 自动放大时保留的模型点比例
    keep_ratio=0.8,
    # 自动放大区域的边距比例
    padding_ratio=0.18,
    # inset 放大图位置控制方式
    #     "auto"：
    #         程序自动选择 inset 位置
    #     "manual"：
    #         手动指定 inset 位置
    #         此时必须设置 inset_bbox=(x0, y0, w, h)
    inset_position="manual",
    # inset 手动位置参数
    # inset_bbox 只在 inset_position="manual" 时生效
    #
    # 格式：
    #     inset_bbox=(x0, y0, w, h)
    #
    # 坐标系统：
    #     使用的是主图 ax 的 axes fraction 坐标
    #     也就是相对于主图绘图区的比例坐标
    #
    # 参数含义：
    #     x0 ：inset 左下角的横向位置
    #          0 表示主图最左侧
    #          1 表示主图最右侧
    #
    #     y0 ：inset 左下角的纵向位置
    #          0 表示主图最下方
    #          1 表示主图最上方
    #
    #     w  ：inset 宽度，占主图宽度的比例
    #
    #     h  ：inset 高度，占主图高度的比例
    #
    # 当前设置：
    #     (0.50, 0.44, 0.25, 0.20)
    #
    # 表示：
    #     inset 左下角位于主图宽度 50%、高度 44% 处；
    #     inset 宽度为主图宽度的 25%；
    #     inset 高度为主图高度的 20%。
    #
    # 调整方法：
    #     x0 增大：整体向右移动
    #     x0 减小：整体向左移动
    #     y0 增大：整体向上移动
    #     y0 减小：整体向下移动
    #     w  增大：inset 变宽
    #     w  减小：inset 变窄
    #     h  增大：inset 变高
    #     h  减小：inset 变矮
    #
    # 常用微调示例：
    #     inset_bbox=(0.54, 0.44, 0.25, 0.20)  # 往右
    #     inset_bbox=(0.46, 0.44, 0.25, 0.20)  # 往左
    #     inset_bbox=(0.50, 0.48, 0.25, 0.20)  # 往上
    #     inset_bbox=(0.50, 0.40, 0.25, 0.20)  # 往下
    #     inset_bbox=(0.50, 0.44, 0.30, 0.24)  # 放大
    #     inset_bbox=(0.50, 0.44, 0.22, 0.18)  # 缩小
    inset_bbox=(0.50, 0.44, 0.25, 0.20)
)
'''


### 模板 taylor 单元 13
来源 模型之间的泰勒图比较：从模型指标计算到局部放大 inset 优化.html

In [ ]:
ORIGINAL_PLOT_SOURCES['taylor']['blocks'][13] = r'''plot_taylor_diagram(
    models=models,
    stds=stds,
    corrs=corrs,
    colors=colors,
    save_pdf_path="taylor_from_models_2.pdf",
    show=True,
    add_inset=True,
    auto_zoom=True,
    keep_ratio=1,
    padding_ratio=0.18,
    inset_position="auto"
)
'''


### 模板 shap 单元 10
来源 期刊复现：多分类XGBoost模型解释SHAP柱状图与蜂群图组合绘制.html

In [ ]:
ORIGINAL_PLOT_SOURCES['shap']['blocks'][10] = r'''from matplotlib.patches import Rectangle, Patch
from matplotlib.colors import LinearSegmentedColormap


def _format_shap_tick(value):
    """格式化底部 SHAP value 小刻度标签。"""
    if abs(value) < 1e-12:
        return "0"

    text = f"{value:.2g}"

    if text.startswith("-0."):
        text = "-." + text.split(".")[1]
    elif text.startswith("0."):
        text = "." + text.split(".")[1]

    return text


def _nice_symmetric_tick_value(value):
    """根据 SHAP 绝对值范围生成较美观的正刻度值。"""
    if value <= 0 or not np.isfinite(value):
        return 0.1

    exponent = np.floor(np.log10(value))
    base = value / (10 ** exponent)

    if base <= 1:
        nice_base = 1
    elif base <= 2:
        nice_base = 2
    elif base <= 2.5:
        nice_base = 2.5
    elif base <= 5:
        nice_base = 5
    else:
        nice_base = 10

    return nice_base * (10 ** exponent)


def _prepare_shap_values(shap_values):
    """
    将不同形式的 SHAP 输出统一转换为：
    (样本数, 特征数, 类别数)
    """
    if hasattr(shap_values, "values"):
        shap_values = shap_values.values

    if isinstance(shap_values, list):
        shap_values = np.stack(shap_values, axis=-1)

    shap_values = np.asarray(shap_values)

    if shap_values.ndim == 2:
        shap_values = shap_values[:, :, np.newaxis]

    if shap_values.ndim != 3:
        raise ValueError(
            "shap_values 必须是三维数组，形状为 "
            "(样本数, 特征数, 类别数)。"
        )

    return shap_values


def plot_multiclass_shap_importance(
    # 多分类 SHAP 值，推荐形状为 (样本数, 特征数, 类别数)
    shap_values,

    # 测试集特征数据；如果是 pandas DataFrame，会自动使用列名作为 y 轴特征名
    X_test,

    # 类别名称列表，顺序必须和 shap_values 第三维类别顺序一致
    class_names,

    # 每个类别对应的柱状背景颜色，顺序必须和 class_names 一致
    class_colors,

    # 散点颜色映射，用于表示特征值从低到高；None 时使用默认蓝-黄-红配色
    scatter_cmap=None,

    # 是否保存为 PDF 文件
    save_pdf=True,

    # PDF 文件保存路径
    pdf_path="multiclass_shap_importance.pdf",

    # 是否额外保存为 PNG 文件
    save_png=False,

    # PNG 文件保存路径
    png_path="multiclass_shap_importance.png",

    # 图像尺寸，单位为英寸；适合双栏论文图
    figsize=(7.2, 4.8),

    # 顶部 Importance value 坐标轴最大值
    importance_max=0.5,

    # 每个类别散点中心在 x 轴上的位置；None 时自动均匀分布
    class_centers=None,

    # 每个特征背景堆叠柱状条的高度
    bar_height=0.56,

    # 背景柱状条透明度；值越小越淡
    bar_alpha=0.24,

    # 散点大小
    scatter_size=3.5,

    # 散点透明度
    scatter_alpha=0.85,

    # 随机种子，用于控制散点纵向 jitter，保证图形可复现
    random_state=42,

    # 底部 “SHAP value” 标签相对最后一个特征的下移距离
    bottom_label_offset=0.62,

    # 图例在 figure 坐标中的纵向位置
    legend_y=0.025,

    # 用于确定每个类别局部 SHAP 轴范围的分位数，避免极端值影响刻度
    shap_tick_percentile=99,

    # 局部 SHAP 轴正负刻度的最小绝对值
    min_symmetric_tick=0.01,

    # 每个类别局部 SHAP 小坐标轴占类别间距的比例；
    # 值越小，类别之间截断空白越明显
    local_axis_fraction=0.30,

    # 底部分段 x 轴线相对主图底部的位置；
    # 值越小，分段轴线越靠下
    segment_line_y=-0.055
):
    """
    绘制多类别 SHAP 值综合可视化图
    """
    #  绘图风格
    plt.rcParams.update({
        "font.family": "Arial",
        "font.size": 7,
        "font.weight": "normal",
        "axes.labelsize": 8,
        "axes.labelweight": "normal",
        "axes.titlesize": 9,
        "axes.titleweight": "normal",
        "axes.linewidth": 0.6,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "legend.fontsize": 7,
        "legend.frameon": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "figure.dpi": 300,
        "savefig.dpi": 600,
        "savefig.bbox": "tight"
    })

    rng = np.random.default_rng(random_state)
    shap_values = _prepare_shap_values(shap_values)
    n_samples, n_features, n_classes = shap_values.shape

    if len(class_names) != n_classes:
        raise ValueError("class_names 的长度必须等于 shap_values 的类别数。")

    if len(class_colors) != n_classes:
        raise ValueError("class_colors 的长度必须等于 shap_values 的类别数。")
    if isinstance(X_test, pd.DataFrame):
        feature_names = list(X_test.columns)
        X_values = X_test.values
    else:
        X_values = np.asarray(X_test)
        feature_names = [f"Feature {i + 1}" for i in range(X_values.shape[1])]

    if X_values.shape[0] != n_samples:
        raise ValueError("X_test 的样本数必须和 shap_values 的样本数一致。")

    if X_values.shape[1] != n_features:
        raise ValueError("X_test 的特征数必须和 shap_values 的特征数一致。")

    X_min = X_values.min(axis=0)
    X_max = X_values.max(axis=0)
    X_range = np.where(X_max - X_min == 0, 1, X_max - X_min)
    X_norm = (X_values - X_min) / X_range

    if scatter_cmap is None:
        scatter_cmap = LinearSegmentedColormap.from_list(
            "feature_value_cmap",
            ["#2166AC", "#67A9CF", "#F7F7F7", "#F4A582", "#B2182B"]
        )

    importance_raw = np.mean(np.abs(shap_values), axis=0)
    total_importance = importance_raw.sum(axis=1)

    order = np.argsort(total_importance)[::-1]

    feature_names_sorted = [feature_names[i] for i in order]
    X_norm_sorted = X_norm[:, order]
    shap_sorted = shap_values[:, order, :]
    importance_sorted = importance_raw[order, :]

    max_total_importance = total_importance.max()

    if max_total_importance == 0:
        importance_scaled = importance_sorted
    else:
        importance_scaled = importance_sorted / max_total_importance * importance_max

    if class_centers is None:
        class_centers = np.linspace(0.03, importance_max - 0.07, n_classes)
    else:
        class_centers = np.asarray(class_centers)

    if len(class_centers) != n_classes:
        raise ValueError("class_centers 的长度必须等于类别数。")

    if n_classes > 1:
        class_gap = np.min(np.diff(np.sort(class_centers)))
    else:
        class_gap = importance_max * 0.8

    local_half_width = class_gap * local_axis_fraction

    class_positive_ticks = []
    class_scales = []

    for c in range(n_classes):
        class_shap_vals = shap_sorted[:, :, c].ravel()
        class_shap_vals = class_shap_vals[np.isfinite(class_shap_vals)]

        if class_shap_vals.size == 0:
            positive_tick = min_symmetric_tick
        else:
            raw_tick = np.percentile(np.abs(class_shap_vals), shap_tick_percentile)
            positive_tick = _nice_symmetric_tick_value(raw_tick)
            positive_tick = max(positive_tick, min_symmetric_tick)

        class_positive_ticks.append(positive_tick)
        class_scales.append(local_half_width / positive_tick)

    class_positive_ticks = np.asarray(class_positive_ticks)
    class_scales = np.asarray(class_scales)
    fig, ax = plt.subplots(figsize=figsize)

    ax.set_xlim(-0.04, importance_max + 0.04)
    ax.set_ylim(-0.8, n_features - 0.2)
    ax.invert_yaxis()

    for j, feature in enumerate(feature_names_sorted):
        y = j

        left = 0
        for c in range(n_classes):
            width = importance_scaled[j, c]

            ax.add_patch(
                Rectangle(
                    (left, y - bar_height / 2),
                    width,
                    bar_height,
                    facecolor=class_colors[c],
                    edgecolor=class_colors[c],
                    alpha=bar_alpha,
                    linewidth=0.45,
                    zorder=1
                )
            )

            left += width

        for c in range(n_classes):
            shap_vals = shap_sorted[:, j, c]
            feature_vals = X_norm_sorted[:, j]
            jitter = rng.normal(0, 0.045, size=n_samples)

            positive_tick = class_positive_ticks[c]
            scale = class_scales[c]

            shap_vals_clipped = np.clip(shap_vals, -positive_tick, positive_tick)
            x_plot = class_centers[c] + shap_vals_clipped * scale

            ax.scatter(
                x_plot,
                y + jitter,
                c=feature_vals,
                cmap=scatter_cmap,
                s=scatter_size,
                alpha=scatter_alpha,
                linewidths=0,
                zorder=3
            )

    for center in class_centers:
        ax.axvline(center, color="#777777", linewidth=0.45, zorder=2)
    ax.set_yticks(range(n_features))
    ax.set_yticklabels(feature_names_sorted, fontsize=7)
    ax.tick_params(axis="y", length=0, pad=2)

    ax.set_xticks(np.arange(0, importance_max + 0.001, 0.1))
    ax.xaxis.set_ticks_position("top")
    ax.xaxis.set_label_position("top")
    ax.set_xlabel("Importance value", fontsize=9, labelpad=7)
    ax.tick_params(axis="x", labelsize=7, width=0.6, length=3, pad=2)

    minor_tick_positions = []
    minor_tick_labels = []

    for c, center in enumerate(class_centers):
        positive_tick = class_positive_ticks[c]
        tick_values = np.array([-positive_tick, 0.0, positive_tick])
        tick_positions = center + tick_values * class_scales[c]

        minor_tick_positions.extend(tick_positions)
        minor_tick_labels.extend([
            _format_shap_tick(-positive_tick),
            "0",
            _format_shap_tick(positive_tick)
        ])

    bottom_ax = ax.secondary_xaxis("bottom")
    bottom_ax.set_xticks(minor_tick_positions)
    bottom_ax.set_xticklabels(minor_tick_labels, fontsize=5.5)
    bottom_ax.tick_params(axis="x", length=2.2, width=0.5, pad=2)
    bottom_ax.spines["bottom"].set_visible(False)

    for label_i, label in enumerate(bottom_ax.get_xticklabels()):
        pos_in_group = label_i % 3
        if pos_in_group == 0:
            label.set_ha("right")
        elif pos_in_group == 1:
            label.set_ha("center")
        else:
            label.set_ha("left")
        label.set_clip_on(False)
    xaxis_transform = ax.get_xaxis_transform()

    for c, center in enumerate(class_centers):
        segment_left = center - local_half_width
        segment_right = center + local_half_width

        ax.plot(
            [segment_left, segment_right],
            [segment_line_y, segment_line_y],
            transform=xaxis_transform,
            color="black",
            linewidth=0.6,
            clip_on=False,
            zorder=5
        )

        for tick_x in [
            center - local_half_width,
            center,
            center + local_half_width
        ]:
            ax.plot(
                [tick_x, tick_x],
                [segment_line_y, segment_line_y + 0.018],
                transform=xaxis_transform,
                color="black",
                linewidth=0.5,
                clip_on=False,
                zorder=5
            )

    bottom_y = n_features + bottom_label_offset

    for center in class_centers:
        ax.text(
            center,
            bottom_y,
            "SHAP value",
            ha="center",
            va="top",
            fontsize=6,
            fontweight="normal"
        )

    for spine in ["right", "left", "bottom"]:
        ax.spines[spine].set_visible(False)

    ax.spines["top"].set_linewidth(0.6)
    cbar_ax = fig.add_axes([0.885, 0.19, 0.010, 0.64])

    sm = plt.cm.ScalarMappable(cmap=scatter_cmap)
    sm.set_array([])

    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_label("Feature value", fontsize=6, labelpad=4)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Low", "High"])
    cbar.ax.tick_params(labelsize=5.5, width=0.5, length=2)
    cbar.outline.set_linewidth(0.5)
    legend_handles = [
        Patch(
            facecolor=class_colors[i],
            edgecolor=class_colors[i],
            alpha=0.45,
            label=class_names[i]
        )
        for i in range(n_classes)
    ]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=n_classes,
        frameon=False,
        fontsize=7,
        handlelength=1.8,
        handleheight=0.8,
        columnspacing=1.4,
        bbox_to_anchor=(0.5, legend_y)
    )

    plt.subplots_adjust(left=0.14, right=0.86, top=0.86, bottom=0.15)

    if save_pdf:
        fig.savefig(pdf_path)

    if save_png:
        fig.savefig(png_path, dpi=600)

    return fig, ax
'''


### 模板 shap 单元 14
来源 期刊复现：多分类XGBoost模型解释SHAP柱状图与蜂群图组合绘制.html

In [ ]:
ORIGINAL_PLOT_SOURCES['shap']['blocks'][14] = r'''# 冷色系 Nature-like 配色：保持统一风格，同时提高类别区分度
class_colors = [
    "#4C78A8",  # MVT - muted blue
    "#72B7B2",  # SEDEX - teal
    "#54A24B",  # VMS - muted green
    "#B279A2",  # epithermal - muted purple
    "#9D755D"   # skarn - muted taupe
]

# 散点仍使用统一蓝-青-浅黄-橙红渐变，Feature value 区分更清楚
scatter_cmap = LinearSegmentedColormap.from_list(
    "scatter_cmap",
    [
        "#2166AC",  # Low
        "#67A9CF",
        "#F7F7F7",
        "#F4A582",
        "#B2182B"   # High
    ]
)

fig, ax = plot_multiclass_shap_importance(
    shap_values=shap_values,
    X_test=X_test,
    class_names=class_names,
    class_colors=class_colors,
    scatter_cmap=scatter_cmap,
    save_pdf=True,
    pdf_path="xgboost_multiclass_shap_importance-1.pdf",
    save_png=True,
    png_path="xgboost_multiclass_shap_importance-1.png",
    local_axis_fraction=0.34
)

plt.show()
'''


### 模板 network 单元 12
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][12] = r'''def get_class_interaction_matrices(
    shap_interaction_values,
    X,
    class_names=None
):
    """
    根据 SHAP interaction values 计算各个类别的平均绝对交互作用矩阵。

    Parameters
    ----------
    shap_interaction_values : np.ndarray
        SHAP 交互值，形状应为：
        (n_samples, n_features, n_features, n_classes)

    X : pd.DataFrame
        特征数据，用于获取特征名称。

    class_names : list, optional
        类别名称列表，例如：
        ["MVT", "SEDEX", "VMS", "epithermal", "skarn"]

    Returns
    -------
    interaction_matrices : dict
        key 为类别名称，value 为对应类别的交互作用矩阵 DataFrame。
    """

    feature_names = X.columns

    n_samples, n_features_1, n_features_2, n_classes = shap_interaction_values.shape

    if n_features_1 != n_features_2:
        raise ValueError("shap_interaction_values 的第 2 和第 3 个维度应相等。")

    if n_features_1 != len(feature_names):
        raise ValueError("SHAP 特征数量与 X 的列数不一致。")

    if class_names is None:
        class_names = [f"class_{i}" for i in range(n_classes)]

    if len(class_names) != n_classes:
        raise ValueError("class_names 的长度必须等于类别数量。")

    interaction_matrices = {}

    for class_idx, class_name in enumerate(class_names):
        # 提取当前类别的交互值
        # shape: (n_samples, n_features, n_features)
        class_interaction_values = shap_interaction_values[:, :, :, class_idx]

        # 对样本维度求平均绝对值
        # shape: (n_features, n_features)
        avg_interaction_matrix = np.mean(
            np.abs(class_interaction_values),
            axis=0
        )

        # 转为 DataFrame
        interaction_matrix_df = pd.DataFrame(
            avg_interaction_matrix,
            index=feature_names,
            columns=feature_names
        )

        interaction_matrices[class_name] = interaction_matrix_df

    return interaction_matrices
'''


### 模板 network 单元 16
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][16] = r'''def interaction_matrices_to_df(
    interaction_matrices,
    multiply_interaction_by_2=True,
    include_main_effect=False
):
    """
    将各类别的交互作用矩阵整理成长表 DataFrame。

    Parameters
    ----------
    interaction_matrices : dict
        key 为类别名称，value 为交互作用矩阵 DataFrame。
        例如：
        {
            "MVT": interaction_matrix_MVT,
            "SEDEX": interaction_matrix_SEDEX,
            ...
        }

    multiply_interaction_by_2 : bool, default=True
        是否将非对角线交互作用值乘以 2。
        SHAP interaction values 中，两个特征的交互作用通常分布在矩阵的两个对称位置，
        因此常见做法是对非对角线交互值乘以 2。

    include_main_effect : bool, default=False
        是否包含对角线上的主效应值。
        False 时只保留 feature1-feature2 的交互项。

    Returns
    -------
    interaction_df_all : pd.DataFrame
        整理后的长表，包含：
        class_name, feature1, feature2, interaction_value
    """

    interaction_data = []

    for class_name, interaction_matrix in interaction_matrices.items():

        feature_names = interaction_matrix.index

        for i, feature1 in enumerate(feature_names):
            for j, feature2 in enumerate(feature_names):

                if include_main_effect:
                    # 包含主效应时，保留 i <= j
                    condition = i <= j
                else:
                    # 不包含主效应时，只保留 i < j，避免重复
                    condition = i < j

                if condition:
                    interaction_value = interaction_matrix.loc[feature1, feature2]

                    # 非对角线交互作用值乘以 2
                    if i != j and multiply_interaction_by_2:
                        interaction_value = interaction_value * 2

                    interaction_data.append([
                        class_name,
                        feature1,
                        feature2,
                        interaction_value
                    ])

    interaction_df_all = pd.DataFrame(
        interaction_data,
        columns=[
            "class_name",
            "feature1",
            "feature2",
            "interaction_value"
        ]
    )

    return interaction_df_all
'''


### 模板 network 单元 21
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][21] = r'''def get_class_shap_importance(
    shap_values,
    X,
    class_names=None,
    return_format="dict"
):
    """
    计算各个类别下每个特征的 SHAP 绝对值均值。

    Parameters
    ----------
    shap_values : np.ndarray
        SHAP values，形状应为：
        (n_samples, n_features, n_classes)

    X : pd.DataFrame
        特征数据，用于获取特征名称。

    class_names : list, optional
        类别名称列表，例如：
        ["MVT", "SEDEX", "VMS", "epithermal", "skarn"]

    return_format : str, default="dict"
        返回格式：
        - "dict"：返回字典，每个类别一个 DataFrame
        - "long"：返回一个长表 DataFrame

    Returns
    -------
    importance_result : dict or pd.DataFrame
        各类别的特征重要性结果。
    """

    feature_names = X.columns

    # 如果 shap_values 是 list，兼容旧版本 SHAP 输出
    if isinstance(shap_values, list):
        n_classes = len(shap_values)
        shap_values_array = np.stack(shap_values, axis=-1)
    else:
        shap_values_array = shap_values
        n_classes = shap_values_array.shape[-1]

    if shap_values_array.ndim != 3:
        raise ValueError("shap_values 的形状应为 (n_samples, n_features, n_classes)")

    n_samples, n_features, n_classes = shap_values_array.shape

    if n_features != len(feature_names):
        raise ValueError("SHAP 特征数量与 X 的列数不一致。")

    if class_names is None:
        class_names = [f"class_{i}" for i in range(n_classes)]

    if len(class_names) != n_classes:
        raise ValueError("class_names 的长度必须等于类别数量。")

    importance_dict = {}

    for class_idx, class_name in enumerate(class_names):

        # 提取当前类别的 SHAP values
        # shape: (n_samples, n_features)
        shap_values_class = shap_values_array[:, :, class_idx]

        # 计算绝对值
        shap_values_abs = np.abs(shap_values_class)

        # 对样本维度求均值
        # shape: (n_features,)
        importance_values = np.mean(shap_values_abs, axis=0)

        # 整理成 DataFrame
        importance_df = pd.DataFrame({
            "feature": feature_names,
            "importance_value": importance_values
        })

        # 按重要性从大到小排序
        importance_df = importance_df.sort_values(
            by="importance_value",
            ascending=False
        ).reset_index(drop=True)

        importance_dict[class_name] = importance_df

    if return_format == "dict":
        return importance_dict

    elif return_format == "long":
        importance_data = []

        for class_name, df in importance_dict.items():
            temp_df = df.copy()
            temp_df.insert(0, "class_name", class_name)
            importance_data.append(temp_df)

        importance_df_all = pd.concat(
            importance_data,
            axis=0,
            ignore_index=True
        )

        return importance_df_all

    else:
        raise ValueError("return_format 只能是 'dict' 或 'long'")
'''


### 模板 network 单元 25
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][25] = r'''import networkx as nx
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.path import Path
from matplotlib.patches import PathPatch
import itertools

def plot_feature_interaction_and_importance_optimized_cb(
    interaction_df,
    importance_df,
    interaction_cmap=cm.coolwarm,
    importance_cmap=cm.Greens,
    title="Feature Interaction and Importance Network",
    save_path=None,

    # 背景/阴影
    background_circle_color="lightgray",
    background_style="curve_shadow",
    background_alpha=0.22,
    background_curve_strength=0.18,
    background_edge_color="lightgray",
    background_linewidth=0.8,

    # 曲线阴影参数
    shadow_color="lightgray",
    shadow_alpha=0.16,
    shadow_width_scale=2.8,
    shadow_base_width=1.2,
    shadow_all_pairs=True,
    shadow_zorder=0,

    # 画布和布局
    figsize=(8, 8),
    layout_scale=1.0,

    # 节点
    node_size_range=(80, 1300),

    # 边
    edge_width_range=(0.3, 5.5),
    edge_curve_strength=0.42,
    edge_curve_jitter=0.035,
    edge_alpha=0.50,
    edge_zorder=1,

    # 标签
    label_offset=0.18,
    label_fontsize=10,
    label_fontweight="bold",

    # 标题
    title_fontsize=15,

    # 图例
    cbar_title_fontsize=13,
    cbar_tick_fontsize=11,

    # 关键修改：默认不反转颜色条
    invert_cbar=False,

    cbar_width="62%",
    cbar_height="12%",
    cbar_title_y=0.45,
    cbar_borderpad=0.45,

    dpi=1200,
    random_seed=123
):
    """
    绘制特征交互强度和特征重要性网络图。

    interaction_df 需要包含：
        feature1, feature2, interaction_value

    importance_df 需要包含：
        feature, importance_value

    background_style:
        - "curve_shadow": 使用和边一致的弯曲线条生成灰色阴影
        - "curved_area": 使用节点外轮廓生成弯曲闭合背景
        - "circle": 完整圆背景
        - None: 不画背景
    """

    def add_curved_edge(
        ax,
        start,
        end,
        center=np.array([0.0, 0.0]),
        curve_strength=0.42,
        jitter=0.0,
        jitter_sign=1,
        color="gray",
        linewidth=1.0,
        alpha=0.5,
        zorder=1,
        rng=None
    ):
        start = np.array(start, dtype=float)
        end = np.array(end, dtype=float)
        center = np.array(center, dtype=float)

        mid = (start + end) / 2.0

        # 控制点向圆心收缩，使线条向内部弯曲
        control = mid * (1 - curve_strength) + center * curve_strength

        direction = end - start
        length = np.linalg.norm(direction)

        if length > 1e-8 and jitter > 0:
            normal = np.array([-direction[1], direction[0]]) / length

            if rng is not None:
                jitter_value = jitter_sign * jitter * rng.uniform(0.3, 1.0)
            else:
                jitter_value = jitter_sign * jitter

            control = control + normal * jitter_value

        path = Path(
            [start, control, end],
            [Path.MOVETO, Path.CURVE3, Path.CURVE3]
        )

        patch = PathPatch(
            path,
            facecolor="none",
            edgecolor=color,
            linewidth=linewidth,
            alpha=alpha,
            zorder=zorder,
            capstyle="round",
            joinstyle="round"
        )

        ax.add_patch(patch)
        return patch

    def add_curved_background_area(
        ax,
        pos,
        center=(0.0, 0.0),
        facecolor="lightgray",
        edgecolor="lightgray",
        alpha=0.25,
        curve_strength=0.18,
        linewidth=0.8,
        zorder=0
    ):
        center = np.array(center, dtype=float)

        ordered_nodes = sorted(
            pos.keys(),
            key=lambda n: np.arctan2(
                pos[n][1] - center[1],
                pos[n][0] - center[0]
            )
        )

        if len(ordered_nodes) < 3:
            return None

        points = [
            np.array(pos[n], dtype=float)
            for n in ordered_nodes
        ]

        vertices = []
        codes = []

        vertices.append(points[0])
        codes.append(Path.MOVETO)

        n = len(points)

        for i in range(n):
            p1 = points[i]
            p2 = points[(i + 1) % n]

            mid = (p1 + p2) / 2.0
            control = mid * (1 - curve_strength) + center * curve_strength

            vertices.extend([control, p2])
            codes.extend([Path.CURVE3, Path.CURVE3])

        vertices.append(points[0])
        codes.append(Path.CLOSEPOLY)

        path = Path(vertices, codes)

        patch = PathPatch(
            path,
            facecolor=facecolor,
            edgecolor=edgecolor,
            alpha=alpha,
            linewidth=linewidth,
            zorder=zorder,
            joinstyle="round",
            capstyle="round"
        )

        ax.add_patch(patch)
        return patch

    G = nx.Graph()

    for _, row in importance_df.iterrows():
        G.add_node(
            row["feature"],
            importance=float(row["importance_value"])
        )

    for _, row in interaction_df.iterrows():
        f1 = row["feature1"]
        f2 = row["feature2"]
        inter = float(row["interaction_value"])

        if f1 not in G:
            G.add_node(f1, importance=0.0)

        if f2 not in G:
            G.add_node(f2, importance=0.0)

        G.add_edge(f1, f2, interaction=inter)

    if len(G.nodes) == 0:
        raise ValueError("图中没有节点，请检查 importance_df 和 interaction_df。")

    pos = nx.circular_layout(G, scale=layout_scale)

    center_x, center_y = 0.0, 0.0
    center = np.array([center_x, center_y], dtype=float)
    background_circle_radius = layout_scale

    node_importance = [
        G.nodes[node].get("importance", 0.0)
        for node in G.nodes()
    ]

    min_importance = min(node_importance)
    max_importance = max(node_importance)

    if max_importance == min_importance:
        node_norm_values = [0.5 for _ in node_importance]
        importance_norm = mcolors.Normalize(
            vmin=0,
            vmax=max_importance if max_importance > 0 else 1
        )
    else:
        importance_norm = mcolors.Normalize(
            vmin=min_importance,
            vmax=max_importance
        )
        node_norm_values = [
            importance_norm(v)
            for v in node_importance
        ]

    node_size_min, node_size_max = node_size_range

    node_sizes = [
        node_size_min + v * (node_size_max - node_size_min)
        for v in node_norm_values
    ]
    node_colors = [
        importance_cmap(importance_norm(v))
        for v in node_importance
    ]

    edges_data = []

    for u, v, data in G.edges(data=True):
        edges_data.append(
            (
                data.get("interaction", 0.0),
                u,
                v
            )
        )

    edges_data.sort(key=lambda x: x[0])

    edge_values = [x[0] for x in edges_data]
    edge_list = [(x[1], x[2]) for x in edges_data]

    if len(edge_values) > 0:
        min_interaction = min(edge_values)
        max_interaction = max(edge_values)
    else:
        min_interaction = 0.0
        max_interaction = 1.0

    if max_interaction == min_interaction:
        edge_norm_values = [0.5 for _ in edge_values]
        interaction_norm = mcolors.Normalize(
            vmin=0,
            vmax=max_interaction if max_interaction > 0 else 1
        )
    else:
        interaction_norm = mcolors.Normalize(
            vmin=min_interaction,
            vmax=max_interaction
        )
        edge_norm_values = [
            interaction_norm(v)
            for v in edge_values
        ]

    edge_width_min, edge_width_max = edge_width_range

    edge_widths = [
        edge_width_min + v * (edge_width_max - edge_width_min)
        for v in edge_norm_values
    ]

    edge_colors = [
        interaction_cmap(interaction_norm(v))
        for v in edge_values
    ]

    fig = plt.figure(figsize=figsize)

    gs = fig.add_gridspec(
        nrows=2,
        ncols=1,
        height_ratios=[0.82, 0.18],
        hspace=0.02
    )

    ax_main = fig.add_subplot(gs[0, 0])

    gs_cbar = gs[1, 0].subgridspec(
        nrows=1,
        ncols=2,
        wspace=0.35
    )

    ax_cbar_interaction_box = fig.add_subplot(gs_cbar[0, 0])
    ax_cbar_importance_box = fig.add_subplot(gs_cbar[0, 1])

    ax_cbar_interaction_box.axis("off")
    ax_cbar_importance_box.axis("off")

    rng = np.random.default_rng(random_seed)

    if background_style == "curve_shadow":
        if shadow_all_pairs:
            shadow_pairs = list(itertools.combinations(list(G.nodes()), 2))
        else:
            shadow_pairs = edge_list

        for i, (u, v) in enumerate(shadow_pairs):
            x1, y1 = pos[u]
            x2, y2 = pos[v]

            if shadow_all_pairs:
                shadow_width = shadow_base_width
            else:
                if (u, v) in edge_list:
                    idx = edge_list.index((u, v))
                elif (v, u) in edge_list:
                    idx = edge_list.index((v, u))
                else:
                    idx = 0

                shadow_width = edge_widths[idx] * shadow_width_scale

            add_curved_edge(
                ax=ax_main,
                start=(x1, y1),
                end=(x2, y2),
                center=center,
                curve_strength=edge_curve_strength,
                jitter=0.0,
                jitter_sign=1,
                color=shadow_color,
                linewidth=shadow_width,
                alpha=shadow_alpha,
                zorder=shadow_zorder,
                rng=rng
            )

    elif background_style == "curved_area":
        add_curved_background_area(
            ax=ax_main,
            pos=pos,
            center=(center_x, center_y),
            facecolor=background_circle_color,
            edgecolor=background_edge_color,
            alpha=background_alpha,
            curve_strength=background_curve_strength,
            linewidth=background_linewidth,
            zorder=0
        )

    elif background_style == "circle":
        circle = plt.Circle(
            (center_x, center_y),
            background_circle_radius,
            facecolor=background_circle_color,
            edgecolor=background_edge_color,
            alpha=background_alpha,
            linewidth=background_linewidth,
            zorder=0
        )
        ax_main.add_patch(circle)

    elif background_style is None:
        pass

    else:
        raise ValueError(
            "background_style 只能是 'curve_shadow', 'curved_area', 'circle' 或 None。"
        )

    if len(edge_list) > 0:
        for i, ((u, v), width, color) in enumerate(
            zip(edge_list, edge_widths, edge_colors)
        ):
            x1, y1 = pos[u]
            x2, y2 = pos[v]

            sign = -1 if i % 2 == 0 else 1

            add_curved_edge(
                ax=ax_main,
                start=(x1, y1),
                end=(x2, y2),
                center=center,
                curve_strength=edge_curve_strength,
                jitter=edge_curve_jitter,
                jitter_sign=sign,
                color=color,
                linewidth=width,
                alpha=edge_alpha,
                zorder=edge_zorder,
                rng=rng
            )

    node_collection = nx.draw_networkx_nodes(
        G,
        pos,
        ax=ax_main,
        node_color=node_colors,
        node_size=node_sizes,
        linewidths=0.8,
        edgecolors="white",
        alpha=1.0
    )

    if node_collection is not None:
        node_collection.set_zorder(3)

    for node, coords in pos.items():
        x, y = coords

        lx = x * (1 + label_offset)
        ly = y * (1 + label_offset)

        if x > 0.15:
            ha = "left"
        elif x < -0.15:
            ha = "right"
        else:
            ha = "center"

        if y > 0.15:
            va = "bottom"
        elif y < -0.15:
            va = "top"
        else:
            va = "center"

        ax_main.text(
            lx,
            ly,
            str(node),
            fontsize=label_fontsize,
            fontweight=label_fontweight,
            ha=ha,
            va=va,
            zorder=4
        )

    if title:
        ax_main.set_title(
            title,
            fontsize=title_fontsize,
            fontweight="bold",
            pad=8
        )

    ax_main.axis("off")
    ax_main.set_aspect("equal", adjustable="box")

    lim = background_circle_radius * 1.38
    ax_main.set_xlim(-lim, lim)
    ax_main.set_ylim(-lim, lim)

    sm_interaction = cm.ScalarMappable(
        cmap=interaction_cmap,
        norm=interaction_norm
    )
    sm_interaction.set_array([])

    sm_importance = cm.ScalarMappable(
        cmap=importance_cmap,
        norm=importance_norm
    )
    sm_importance.set_array([])

    cax_interaction = inset_axes(
        ax_cbar_interaction_box,
        width=cbar_width,
        height=cbar_height,
        loc="lower center",
        borderpad=cbar_borderpad
    )

    cax_importance = inset_axes(
        ax_cbar_importance_box,
        width=cbar_width,
        height=cbar_height,
        loc="lower center",
        borderpad=cbar_borderpad
    )

    cbar_interaction = fig.colorbar(
        sm_interaction,
        cax=cax_interaction,
        orientation="horizontal"
    )

    cbar_importance = fig.colorbar(
        sm_importance,
        cax=cax_importance,
        orientation="horizontal"
    )

    ax_cbar_interaction_box.text(
        0.5,
        cbar_title_y,
        "Interaction Strength",
        ha="center",
        va="center",
        fontsize=cbar_title_fontsize,
        fontweight="normal",
        transform=ax_cbar_interaction_box.transAxes
    )

    ax_cbar_importance_box.text(
        0.5,
        cbar_title_y,
        "Feature Importance",
        ha="center",
        va="center",
        fontsize=cbar_title_fontsize,
        fontweight="normal",
        transform=ax_cbar_importance_box.transAxes
    )

    for cbar in [cbar_interaction, cbar_importance]:
        cbar.ax.xaxis.set_ticks_position("top")
        cbar.ax.xaxis.set_label_position("top")
        cbar.ax.tick_params(
            labelsize=cbar_tick_fontsize,
            length=0,
            pad=2
        )
        cbar.outline.set_visible(False)

    cbar_interaction.set_ticks([min_interaction, max_interaction])
    cbar_importance.set_ticks([min_importance, max_importance])

    cbar_interaction.ax.set_xticklabels([
        f"{min_interaction:.3f}",
        f"{max_interaction:.3f}"
    ])

    cbar_importance.ax.set_xticklabels([
        f"{min_importance:.3f}",
        f"{max_importance:.3f}"
    ])

    if invert_cbar:
        cbar_interaction.ax.invert_xaxis()
        cbar_importance.ax.invert_xaxis()

    if save_path:
        file_format = save_path.split(".")[-1] if "." in save_path else "pdf"

        plt.savefig(
            save_path,
            format=file_format,
            bbox_inches="tight",
            dpi=dpi
        )

        print(f"Plot saved to {save_path}")
    else:
        plt.show()

    return fig, ax_main
'''


### 模板 network 单元 26
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][26] = r'''from matplotlib.colors import LinearSegmentedColormap

interaction_cmap_literature = LinearSegmentedColormap.from_list(
    "interaction_literature",
    [
        "#3b3b8f",   # 弱交互：蓝紫色
        "#f2f2f2",   # 中等交互：浅灰
        "#e41a1c",   # 强交互：红色
    ]
)

importance_cmap_literature = LinearSegmentedColormap.from_list(
    "importance_literature",
    [
        "#e6f2e1",   # 低重要性：浅绿色
        "#9dcc9a",   # 中等重要性：中绿色
        "#5a9f62",   # 高重要性：深绿色
    ]
)
'''


### 模板 network 单元 27
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][27] = r'''fig, ax = plot_feature_interaction_and_importance_optimized_cb(
    # 当前类别的特征交互作用数据
    # 只筛选 class_name 为 "MVT" 的交互作用结果
    interaction_df_all[interaction_df_all["class_name"] == "MVT"],

    # 当前类别的特征重要性数据
    importance_dict["MVT"],

    # 交互作用边的颜色映射
    # 文献风格：红色-浅色-蓝紫色
    interaction_cmap=interaction_cmap_literature,

    # 节点特征重要性的颜色映射
    # 文献风格：浅绿色-深绿色
    importance_cmap=importance_cmap_literature,

    # 图标题
    title="MVT",

    # 背景样式
    # "curve_shadow" 表示绘制所有特征之间的浅灰色背景曲线
    background_style="curve_shadow",

    # 是否为所有特征对绘制背景阴影曲线
    # True 表示即使交互作用较弱，也绘制浅灰背景连线
    shadow_all_pairs=True,

    # 背景阴影曲线颜色
    shadow_color="lightgray",

    # 背景阴影曲线透明度
    # 数值越小越透明
    shadow_alpha=0.16,

    # 背景阴影曲线基础线宽
    # 数值越大，背景灰色连线越粗
    shadow_base_width=1.4,

    # 特征交互作用边的弯曲程度
    # 数值越大，边越弯曲
    edge_curve_strength=0.42,

    # 边弯曲时的随机扰动程度
    # 用于避免多条边完全重合
    edge_curve_jitter=0.035,

    # 交互作用边的透明度
    # 数值越大，边越明显
    edge_alpha=0.55,

    # 颜色条图例与主图之间的距离
    # 数值越大，颜色条图例整体越往上靠近主图
    cbar_borderpad=4,

    # 颜色条标题的垂直位置
    # 数值越大，标题相对于颜色条越往上
    cbar_title_y=1,

    # 交互作用边线宽范围
    # 最小交互作用对应 0.25，最大交互作用对应 6.0
    edge_width_range=(0.25, 6.0),

    # 节点大小范围
    # 特征重要性越大，节点越大
    node_size_range=(70, 1400),

    # 特征标签相对于节点的偏移距离
    # 数值越大，标签离节点越远
    label_offset=0.16,

    # 图片尺寸
    figsize=(8, 8),

    # 图片保存路径
    save_path="feature_network_MVT.png",
)
'''


### 模板 network 单元 29
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][29] = r'''fig, ax = plot_feature_interaction_and_importance_optimized_cb(
    interaction_df_all[interaction_df_all["class_name"] == "SEDEX"],
    importance_dict["SEDEX"],
    interaction_cmap=interaction_cmap_literature,
    importance_cmap=importance_cmap_literature,
    title="SEDEX",
    background_style="curve_shadow",
    shadow_all_pairs=True,
    shadow_color="lightgray",
    shadow_alpha=0.16,
    shadow_base_width=1.4,
    edge_curve_strength=0.42,
    edge_curve_jitter=0.035,
    edge_alpha=0.55,
    cbar_borderpad=4,
    cbar_title_y=1,
    edge_width_range=(0.25, 6.0),
    node_size_range=(70, 1400),
    label_offset=0.16,
    figsize=(8, 8),
    save_path="feature_network_SEDEX.png",
)
'''


### 模板 network 单元 31
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][31] = r'''fig, ax = plot_feature_interaction_and_importance_optimized_cb(
    interaction_df_all[interaction_df_all["class_name"] == "VMS"],
    importance_dict["VMS"],
    interaction_cmap=interaction_cmap_literature,
    importance_cmap=importance_cmap_literature,
    title="VMS",
    background_style="curve_shadow",
    shadow_all_pairs=True,
    shadow_color="lightgray",
    shadow_alpha=0.16,
    shadow_base_width=1.4,
    edge_curve_strength=0.42,
    edge_curve_jitter=0.035,
    edge_alpha=0.55,
    cbar_borderpad=4,
    cbar_title_y=1,
    edge_width_range=(0.25, 6.0),
    node_size_range=(70, 1400),
    label_offset=0.16,
    figsize=(8, 8),
    save_path="feature_network_VMS.png",
)
'''


### 模板 network 单元 33
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][33] = r'''fig, ax = plot_feature_interaction_and_importance_optimized_cb(
    interaction_df_all[interaction_df_all["class_name"] == "epithermal"],
    importance_dict["epithermal"],
    interaction_cmap=interaction_cmap_literature,
    importance_cmap=importance_cmap_literature,
    title="epithermal",
    background_style="curve_shadow",
    shadow_all_pairs=True,
    shadow_color="lightgray",
    shadow_alpha=0.16,
    shadow_base_width=1.4,
    edge_curve_strength=0.42,
    edge_curve_jitter=0.035,
    edge_alpha=0.55,
    cbar_borderpad=4,
    cbar_title_y=1,
    edge_width_range=(0.25, 6.0),
    node_size_range=(70, 1400),
    label_offset=0.16,
    figsize=(8, 8),
    save_path="feature_network_epithermal.png",
)
'''


### 模板 network 单元 35
来源 期刊复现：告别枯燥！多分类SHAP网络图读懂变量重要性与交互作用如何共同影响模型预测.html

In [ ]:
ORIGINAL_PLOT_SOURCES['network']['blocks'][35] = r'''fig, ax = plot_feature_interaction_and_importance_optimized_cb(
    interaction_df_all[interaction_df_all["class_name"] == "skarn"],
    importance_dict["skarn"],
    interaction_cmap=interaction_cmap_literature,
    importance_cmap=importance_cmap_literature,
    title="skarn",
    background_style="curve_shadow",
    shadow_all_pairs=True,
    shadow_color="lightgray",
    shadow_alpha=0.16,
    shadow_base_width=1.4,
    edge_curve_strength=0.42,
    edge_curve_jitter=0.035,
    edge_alpha=0.55,
    cbar_borderpad=4,
    cbar_title_y=1,
    edge_width_range=(0.25, 6.0),
    node_size_range=(70, 1400),
    label_offset=0.16,
    figsize=(8, 8),
    save_path="feature_network_skarn.png",
)
'''


### 4.1 数据适配、原模板执行与矢量导出
本单元中的函数直接调用上面内嵌模板。每次代码替换均写入`execution_provenance.json`。

In [ ]:
import gc
"""Data adapters only: execute the user's original plotting functions/cells.

No independently implemented plot layouts. Sources are the six supplied HTML /
notebook packages. Training, splitting and predictions are not changed here.
Every executed block, source hash and permitted substitution is recorded.
"""
from pathlib import Path
import argparse
import contextlib
import hashlib
import io
import json
import re
import time
import warnings
import xml.etree.ElementTree as ET

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.text import Text
from matplotlib import font_manager
from matplotlib.ticker import PercentFormatter
from sklearn import metrics

PROJECT = ROOT
WORKSPACE = PROJECT.parent
ANGLES = [30, 45, 60, 90, 120]
_ORIGINAL_PLOT_FEATURES = [f'raw_{i:02d}' for i in range(13)] + [
    'window_peak_drr', 'window_mean_drr', 'window_rms_drr']
DISPLAY = {'window_peak_drr':'Window peak', 'window_mean_drr':'Window mean',
           'window_rms_drr':'Window RMS'}
SPECS = {
    'shap': ('期刊复现：多分类XGBoost*', '*.html'),
    'network': ('期刊复现：告别枯燥*', '*.html'),
    'taylor': ('机器学习模型之间的泰勒*', '*.html'),
    'metrics': ('期刊配图：如何有效*', '分类.ipynb'),
    'optuna': ('基于Optuna*', '基于Optuna的机器学习超参数调优与自定义可视化分析.ipynb'),
    'roc': ('多分类如何*', '多分类如何绘制ROC曲线--宏平均ROC曲线.ipynb'),
    'cvroc': ('期刊配图：交叉*', '期刊配图：交叉验证与ROC曲线相结合的带置信区间可视化.ipynb'),
}


def sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _plot_write_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf8')


def read_originals():
    # All plotting blocks live in earlier cells of this notebook.
    return json.loads(json.dumps(ORIGINAL_PLOT_SOURCES))


def style():
    plt.rcdefaults()
    for path in (PROJECT/'fonts').glob('*.ttf'):
        font_manager.fontManager.addfont(str(path))
    plt.rcParams.update({'font.family':'Arial', 'font.sans-serif':['Arial'],
        'svg.fonttype':'none', 'pdf.fonttype':42, 'ps.fonttype':42,
        'axes.unicode_minus':False, 'mathtext.fontset':'custom',
        'mathtext.rm':'Arial', 'mathtext.it':'Arial:italic', 'mathtext.bf':'Arial:bold'})


class OriginalExecutor:
    def __init__(self, sources, dest):
        self.sources, self.dest = sources, Path(dest)
        self.figures, self.executions, self.status = [], [], []
        self.original_save = Figure.savefig
        self.original_show = plt.show
        self.target = None
        self.saved = set()

    def env(self):
        return {'np':np, 'pd':pd, 'plt':plt, '__name__':'original_plot_template'}

    def execute(self, family, block, env, changes=None):
        original = self.sources[family]['blocks'][block]
        code = original
        for old, new in (changes or {}).items():
            if old not in code:
                raise ValueError(f'Missing explicit substitution: {family}[{block}]: {old}')
            code = code.replace(old, new)
        self.executions.append(dict(family=family, block=block,
            source=self.sources[family]['source'], source_sha256=self.sources[family]['sha256'],
            original_code_sha256=hashlib.sha256(original.encode()).hexdigest(),
            substitutions=changes or {}, unchanged=code==original))
        # Extracted source is data-authorized plotting code. Do not execute its
        # example Excel reads, model fits, downloads, or stored output blocks.
        exec(compile(code, f"{self.sources[family]['source']}#block-{block}", 'exec'), env)

    def _export(self, fig):
        if id(fig) in self.saved:
            return
        if self.target is None:
            raise RuntimeError('Original plot called export without an assigned output')
        path = self.dest/self.target
        path.parent.mkdir(parents=True, exist_ok=True)
        # Font/export requirements only; never change plot geometry or colours.
        for artist in fig.findobj():
            if isinstance(artist, Text):
                artist.set_fontfamily('Arial')
            if hasattr(artist, 'set_rasterized'):
                artist.set_rasterized(False)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', UserWarning)
            for suffix in ['.pdf', '.svg']:
                self.original_save(fig, path.with_suffix(suffix), format=suffix[1:], bbox_inches='tight')
        svg = ET.parse(path.with_suffix('.svg')).getroot()
        assert not any(e.tag.endswith('}image') for e in svg.iter()), path
        assert any(e.tag.endswith('}text') for e in svg.iter()), path
        self.figures.append(dict(svg=str(path.with_suffix('.svg')), pdf=str(path.with_suffix('.pdf')),
            family=self.family, context=self.context, source=self.sources[self.family]['source'],
            source_sha256=self.sources[self.family]['sha256']))
        self.saved.add(id(fig))

    @contextlib.contextmanager
    def capture(self, target, family, context):
        self.target, self.family, self.context = target, family, context
        self.saved = set()
        Figure.savefig = lambda fig,*args,**kwargs:self._export(fig)
        plt.show = lambda *args,**kwargs:self._export(plt.gcf())
        style()
        try:
            yield
        finally:
            Figure.savefig = self.original_save
            plt.show = self.original_show
            plt.close('all')
            gc.collect()


def roc_arrays(part):
    y = part.y_true.to_numpy(int)
    p = part[[f'p{i}' for i in range(5)]].to_numpy(float)
    fprs, tprs, aucs = {}, {}, {}
    for c in range(5):
        fprs[c], tprs[c], _ = metrics.roc_curve((y==c).astype(int), p[:,c])
        aucs[c] = metrics.auc(fprs[c],tprs[c])
    all_fpr = np.unique(np.concatenate(list(fprs.values())))
    mean_tpr = np.mean([np.interp(all_fpr,fprs[c],tprs[c]) for c in range(5)],axis=0)
    return fprs,tprs,aucs,all_fpr,mean_tpr,float(metrics.auc(all_fpr,mean_tpr))


def fold_roc(pred):
    rows=[]
    for fold,part in pred[pred.split=='nested_oof'].groupby('fold'):
        _,_,_,fpr,tpr,area=roc_arrays(part)
        rows.extend(dict(fold=int(fold),fpr=float(x),tpr=float(y),roc_auc=area) for x,y in zip(fpr,tpr))
    return pd.DataFrame(rows)


_SOURCE_CACHE = None


def original_history_figure(study):
    """Training progress also uses the supplied history cell, without file I/O."""
    global _SOURCE_CACHE
    if _SOURCE_CACHE is None:
        _SOURCE_CACHE = read_originals()
    trials=[t for t in study.trials if t.state.name=='COMPLETE' and t.value is not None]
    if not trials:
        return None
    env={'np':np,'pd':pd,'plt':plt,'optimization_history':pd.DataFrame({
        'trial_number':[t.number for t in trials],'value':[t.value for t in trials]})}
    code=_SOURCE_CACHE['optuna']['blocks'][3].replace('.idxmin()','.idxmax()').replace('"MSE"','"Macro F1"')
    original_save,original_show=Figure.savefig,plt.show
    try:
        Figure.savefig=lambda *args,**kwargs:None
        plt.show=lambda *args,**kwargs:None
        style()
        exec(compile(code,'user-optuna-history-cell','exec'),env)
        return plt.gcf()
    finally:
        Figure.savefig,plt.show=original_save,original_show


def run_original_template_plots(root, out, *, detail_models=None):
    """Re-use saved results. Default: detail plots for all 42 trained pipelines."""
    import optuna
    import xgboost as xgb
    root,out=Path(root),Path(out)
    dest=out/'figures/original_templates'
    audit=out/'original_template_audit';audit.mkdir(parents=True,exist_ok=True)
    sources=read_originals();runner=OriginalExecutor(sources,dest)
    comparison=_read_plot_csv(out/'model_comparison.csv')
    metrics_table=_read_plot_csv(out/'all_classification_metrics.csv')
    before={str(p):sha(p) for p in out.glob('*/*/model.joblib')}
    immutable=[out/'selected_by_nested_cv.json',out/'model_comparison.csv',out/'all_classification_metrics.csv',out/'studies.sqlite3']
    before.update({str(p):sha(p) for p in immutable})
    _plot_write_json(audit/'original_sources.json',sources)
    _plot_write_json(out/'selected_for_paper.json',dict(models=cv_selected,top3=top3,basis='Outer nested-CV macro F1; no test-driven ranking'))
    cache={}
    for scheme in ['row_random','cycle_grouped']:
        names=comparison[comparison.scheme==scheme].sort_values(
            ['nested_cv_f1_mean','nested_cv_f1_sd','model'],ascending=[False,True,True]).model.tolist()
        for name in names:
            pred=_read_plot_csv(out/scheme/name/'predictions.csv')
            cache[scheme,name]=pred
            if detail_models is not None and name not in detail_models:
                continue
            test=pred[pred.split=='test']
            env=runner.env();env.update(ytest=test.y_true.to_numpy(),y_pred=test.y_pred.to_numpy(),PercentFormatter=PercentFormatter)
            # DataFrame carries physical class labels through the unchanged heatmap cell.
            env['confusion_matrix']=lambda yt,yp:pd.DataFrame(metrics.confusion_matrix(yt,yp,labels=range(5)),index=ANGLES,columns=ANGLES)
            cell=sources['roc']['blocks'][4]
            # The cell imports confusion_matrix; replace only that data-producing statement.
            with runner.capture(f'{scheme}/{name}/confusion_matrix','roc',dict(scheme=scheme,model=name)):
                runner.execute('roc',4,env,{'conf_matrix = confusion_matrix(ytest, y_pred)': 'conf_counts = confusion_matrix(ytest, y_pred, labels=range(5))\nrow_totals = conf_counts.sum(axis=1, keepdims=True)\nconf_matrix = pd.DataFrame(np.divide(conf_counts, row_totals, out=np.zeros_like(conf_counts, dtype=float), where=row_totals != 0), index=[30,45,60,90,120], columns=[30,45,60,90,120])', "fmt='d', cmap='YlGnBu', cbar_kws={'shrink': 0.75})": "fmt='.1%', cmap='YlGnBu', vmin=0, vmax=1, cbar_kws={'shrink': 0.75, 'format': PercentFormatter(xmax=1, decimals=0)})"})
            fpr,tpr,areas,mfpr,mtpr,marea=roc_arrays(test)
            env=runner.env();env.update(rf_FPR=fpr,rf_TPR=tpr,rf_AUC=areas,rf_FPR_final=mfpr,rf_TPR_final=mtpr,rf_AUC_final=marea)
            changes={'Random Forest':name}
            changes.update({f'Class {i+1} ROC':f'{angle} deg ROC' for i,angle in enumerate(ANGLES)})
            with runner.capture(f'{scheme}/{name}/test_multiclass_roc','roc',dict(scheme=scheme,model=name)):
                runner.execute('roc',6,env,changes)
            env=runner.env();env.update(roc_data_df=fold_roc(pred),mean_fpr=np.linspace(0,1,100))
            for block,label in [(2,'cv_fold_roc'),(3,'cv_mean_roc')]:
                # Original shading is +/- one fold SD, never a confidence interval.
                changes={'with Confidence Interval':'with Fold Standard Deviation'} if block==3 else {}
                with runner.capture(f'{scheme}/{name}/{label}','cvroc',dict(scheme=scheme,model=name)):
                    runner.execute('cvroc',block,env,changes)
            study=optuna.load_study(study_name=f'{scheme}__{name}__final',storage=f'sqlite:///{(out/"studies.sqlite3").as_posix()}')
            complete=[t for t in study.trials if t.state==optuna.trial.TrialState.COMPLETE and t.value is not None]
            env=runner.env();env['optimization_history']=pd.DataFrame({'trial_number':[t.number for t in complete],'value':[t.value for t in complete]})
            with runner.capture(f'{scheme}/{name}/optuna_history','optuna',dict(scheme=scheme,model=name,complete=len(complete))):
                runner.execute('optuna',3,env,{'.idxmin()':'.idxmax()', '"MSE"':'"Macro F1"'})
            try:
                if len(set(t.value for t in complete))<2:
                    raise ValueError('Constant completed objective; fANOVA importance is undefined')
                importance=optuna.importance.get_param_importances(study,evaluator=optuna.importance.FanovaImportanceEvaluator(seed=42))
                env['Hyperparameter_Importances']=pd.DataFrame(list(importance.items()),columns=['Parameter','Importance'])
                with runner.capture(f'{scheme}/{name}/optuna_importance','optuna',dict(scheme=scheme,model=name)):
                    runner.execute('optuna',5,env)
            except (ValueError,RuntimeError) as exc:
                runner.status.append(dict(scheme=scheme,model=name,figure='optuna_importance',status='unavailable',reason=str(exc)))
            trials=pd.DataFrame([dict(value=t.value,**t.params) for t in complete])
            numeric=[c for c in trials if c!='value' and pd.api.types.is_numeric_dtype(trials[c]) and trials[c].nunique()>1]
            # Prefer the exact parameters from the original XGBoost templates.
            two=['gamma','learning_rate'] if {'gamma','learning_rate'}<=set(numeric) else numeric[:2]
            three=['colsample_bytree','gamma','learning_rate'] if {'colsample_bytree','gamma','learning_rate'}<=set(numeric) else numeric[:3]
            env['Parameter_Combinations']=trials
            if len(two)==2:
                substitutions={a:b for a,b in zip(['gamma','learning_rate'],two) if a!=b}
                with runner.capture(f'{scheme}/{name}/optuna_2d','optuna',dict(scheme=scheme,model=name,parameters=two)):
                    runner.execute('optuna',7,env,substitutions)
            else:runner.status.append(dict(scheme=scheme,model=name,figure='optuna_2d',status='unavailable',reason='Fewer than two varied numeric parameters'))
            if len(three)==3:
                substitutions={a:b for a,b in zip(['colsample_bytree','gamma','learning_rate'],three) if a!=b}
                substitutions['Objective MSE']='Objective Macro F1'
                with runner.capture(f'{scheme}/{name}/optuna_3d','optuna',dict(scheme=scheme,model=name,parameters=three)):
                    runner.execute('optuna',8,env,substitutions)
            else:runner.status.append(dict(scheme=scheme,model=name,figure='optuna_3d',status='unavailable',reason='Fewer than three varied numeric parameters'))
            print(f'Original templates: {scheme}/{name}; {len(runner.figures)} figures',flush=True)

        # The supplied comparison template shows six models. Feed six at a time
        # so every one of the 21 models is covered without changing its geometry.
        table=metrics_table[(metrics_table.scheme==scheme)&(metrics_table.split=='test')].set_index('model').loc[names]
        fields=['accuracy','precision_macro','recall_macro','f1_macro','auc_macro']
        renamed=['Accuracy','Precision','Recall','F1-score','ROC-AUC']
        for start in range(0,len(names),6):
            frame=table.iloc[start:start+6][fields].copy();frame.columns=renamed;frame['Model']=frame.index
            env=runner.env();env['results_df']=frame.reset_index(drop=True)
            page=start//6+1
            for block,label in [(2,'metric_radar'),(3,'metric_polar_bars')]:
                changes={'plot_polar_bars(results_df, 5, 6)':'plot_polar_bars(results_df, 5, len(results_df))'} if block==3 else {}
                with runner.capture(f'{scheme}/comparison/{label}_page_{page}','metrics',dict(scheme=scheme,models=frame.Model.tolist())):
                    runner.execute('metrics',block,env,changes)
        env=runner.env()
        env.update(roc_data_df=fold_roc(cache[scheme,'XGB']),roc_data_rf=fold_roc(cache[scheme,'RF']),
            roc_data_lgb=fold_roc(cache[scheme,'LGBM']),roc_data_lr=fold_roc(cache[scheme,'LR']))
        with runner.capture(f'{scheme}/comparison/cv_four_models','cvroc',dict(scheme=scheme,models=['XGB','RF','LGBM','LR'])):
            runner.execute('cvroc',8,env)

        if ENABLE_TAYLOR:
            env=runner.env();runner.execute('taylor',10,env)
            original_colors=['#7ec9f5','#78e66f','#ef7baa','#c9b8bb','#ffa06a','#76f5d4']
            for split in ['nested_oof','test']:
                for start in range(0,len(names),6):
                    subset=names[start:start+6];stds=[];corrs=[]
                    for name in subset:
                        part=cache[scheme,name].query('split==@split')
                        std,corr,_=env['calc_taylor_stats'](part.angle_true,part.angle_pred)
                        if not np.isfinite(corr) or corr<0:
                            raise ValueError(f'Original positive-quadrant Taylor template cannot plot {scheme}/{name}/{split}: {corr}')
                        stds.append(std);corrs.append(corr)
                    env.update(models=subset,stds=np.array(stds),corrs=np.array(corrs),colors=original_colors[:len(subset)])
                    page=start//6+1
                    for variant,block in [('plain',8),('inset_manual',11),('inset_auto',13)]:
                        runner.execute('taylor',7 if variant=='plain' else 10,env)
                        with runner.capture(f'{scheme}/taylor/{split}_page_{page}_{variant}','taylor',dict(scheme=scheme,split=split,models=subset)):
                            # Use the original function and its original supplied call.
                            with contextlib.redirect_stdout(io.StringIO()):runner.execute('taylor',block,env)

        data=_read_plot_csv(root/'data/peak_windows_13points.csv')
        assignment=_read_plot_csv(out/scheme/'split_assignment.csv')
        test=data.merge(assignment[['window_id','split']],on='window_id',validate='one_to_one').query("split=='test'").reset_index(drop=True)
        pipeline=joblib.load(out/scheme/'XGB/model.joblib')
        features=_ORIGINAL_PLOT_FEATURES[:pipeline.n_features_in_];x=test[features].to_numpy(float)
        transformed=pipeline[:-1].transform(x);booster=pipeline[-1].get_booster();dm=xgb.DMatrix(transformed)
        contrib=booster.predict(dm,pred_contribs=True,strict_shape=True)
        tensor=booster.predict(dm,pred_interactions=True,strict_shape=True)
        raw_margin=booster.predict(dm,output_margin=True,strict_shape=True)
        values=contrib[:,:,:-1].transpose(0,2,1);base=contrib[:,:,-1]
        interactions=tensor[:,:,:-1,:-1].transpose(0,2,3,1)
        errors=dict(additivity=float(np.abs(values.sum(axis=1)+base-raw_margin).max()),
            symmetry=float(np.abs(interactions-interactions.transpose(0,2,1,3)).max()),
            interaction_row_sum=float(np.abs(interactions.sum(axis=2)-values).max()))
        assert max(errors.values())<2e-5,errors
        pred=cache[scheme,'XGB'].query("split=='test'").set_index('window_id').loc[test.window_id]
        assert np.array_equal(pipeline.predict(x),pred.y_pred.to_numpy())
        assert np.allclose(pipeline.predict_proba(x),pred[[f'p{i}' for i in range(5)]].to_numpy(),atol=1e-7)
        folder=audit/scheme;folder.mkdir(exist_ok=True)
        np.savez_compressed(folder/'xgb_tree_shap.npz',values=values,base_values=base,
            interactions=interactions,raw_margin=raw_margin,data=x,window_ids=test.window_id.to_numpy(dtype=str))
        test[['window_id','cycle_id','target_angle_deg']+features].to_csv(folder/'test_inputs.csv',index=False,encoding='utf-8-sig')
        importance=np.mean(np.abs(values),axis=0)
        pd.DataFrame(importance,index=features,columns=ANGLES).to_csv(folder/'mean_absolute_shap_raw_margin.csv',encoding='utf-8-sig')
        X_test=test[features].rename(columns=DISPLAY)
        env=runner.env();env.update(shap_values=values,X_test=X_test,class_names=[f'{a} deg' for a in ANGLES])
        runner.execute('shap',10,env)
        with runner.capture(f'{scheme}/XGB/xgboost_multiclass_shap_importance-1','shap',dict(scheme=scheme,model='XGB',scale='raw class margin',n=len(test),features=features)):
            # The source example has 12 rows; our 16 rows otherwise place the
            # segmented-axis line through its label. Use its existing argument.
            runner.execute('shap',14,env,{'local_axis_fraction=0.34':
                'local_axis_fraction=0.34, bottom_label_offset=1.0'})
        for block in [12,16,21,25,26]:runner.execute('network',block,env)
        matrices=env['get_class_interaction_matrices'](interactions,X_test,env['class_names'])
        pairs=env['interaction_matrices_to_df'](matrices,multiply_interaction_by_2=True,include_main_effect=False)
        importance_dict=env['get_class_shap_importance'](values,X_test,env['class_names'])
        pairs.to_csv(folder/'interaction_pairs_original_definition.csv',index=False,encoding='utf-8-sig')
        # Zero interactions are absent data edges. The original code's light-gray
        # all-pairs shadow remains a decorative background, exactly as supplied.
        env.update(interaction_df_all=pairs[pairs.interaction_value>1e-8],importance_dict=importance_dict)
        edge_counts={}
        for c,(angle,original_class,block) in enumerate(zip(ANGLES,['MVT','SEDEX','VMS','epithermal','skarn'],[27,29,31,33,35])):
            with runner.capture(f'{scheme}/XGB/feature_network_{angle}','network',dict(scheme=scheme,model='XGB',class_angle=angle)):
                with contextlib.redirect_stdout(io.StringIO()):runner.execute('network',block,env,{original_class:f'{angle} deg'})
            edge_counts[angle]=int(((pairs.class_name==f'{angle} deg')&(pairs.interaction_value>1e-8)).sum())
        _plot_write_json(folder/'shap_validation.json',dict(errors=errors,test_windows=len(test),
            test_cycles=int(test.cycle_id.nunique()),scale='raw class margin',nonzero_interaction_pairs=edge_counts,
            importance_axis='Original template rescales total mean absolute SHAP so the maximum bar is 0.5',
            point_axes='Original template uses a separate local SHAP axis per class; 99th percentile clipping retained'))

    assert all(sha(p)==value for p,value in before.items()),'Training artifact changed during plot-only run'
    assert sources==ORIGINAL_PLOT_SOURCES,'Embedded original code was changed'
    _plot_write_json(audit/'execution_provenance.json',runner.executions)
    _plot_write_json(audit/'figure_manifest.json',runner.figures)
    _plot_write_json(audit/'unavailable_templates.json',runner.status)
    # Audit every former figure. Non-template output is retired, not silently
    # restyled or presented as one of the user's reference figures.
    old=[p for p in (out/'figures').rglob('*.svg') if dest not in p.parents]
    pd.DataFrame([dict(old_svg=str(p),old_sha256=sha(p),status='retired_from_current_figure_index',
        reason='Previous independently implemented plot; superseded by direct original-code execution or no supplied template') for p in old]).to_csv(audit/'previous_figure_audit.csv',index=False,encoding='utf-8-sig')
    pd.DataFrame([dict(svg=str(Path(f['svg']).relative_to(out)),pdf=str(Path(f['pdf']).relative_to(out)),family=f['family']) for f in runner.figures]).to_csv(out/'figure_index.csv',index=False,encoding='utf-8-sig')
    _plot_write_json(audit/'validation.json',dict(figure_count=len(runner.figures),
        previous_figures_audited=len(old),original_sources_unchanged=True,original_code_embedded=True,training_artifacts_unchanged=True,
        self_authored_plot_functions=0,all_figures_editable_svg=True,
        explanation_models=cv_selected,benchmark_winners_unchanged=True,unavailable_count=len(runner.status)))
    print('ORIGINAL TEMPLATE EXPORT COMPLETE',len(runner.figures),flush=True)
    return runner.figures




def _read_plot_csv(*args,**kwargs):
    # Preserve near-equal probabilities exactly for ROC/AUC ordering.
    kwargs.setdefault('float_precision','round_trip')
    return pd.read_csv(*args,**kwargs)


## 5. 全模型SHAP计算与原模板适配
解释只在拟合后进行，不用于反向选择测试集表现。GNB、LDA等采用PermutationExplainer；支持的树模型保留TreeSHAP交互，其余模型不构造伪TreeSHAP网络。不同输出尺度的SHAP绝对值不能直接比较。

In [ ]:
"""Compute validated SHAP tensors; execute only the user's original plot cells.

The plotting executor is supplied by the embedded notebook (or the existing
original-code adapter when this file is run directly). No plot is designed here.
"""
from pathlib import Path
import contextlib
import gc
import hashlib
import io
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

SHAP_FEATURES = [f'raw_{i:02d}' for i in range(13)] + [
    'window_peak_drr', 'window_mean_drr', 'window_rms_drr']
SHAP_ANGLES = [30, 45, 60, 90, 120]
SHAP_TREE_MODELS = {'XGB', 'LGBM', 'CAT', 'DT', 'RF', 'ET', 'HGB', 'GBDT', 'BAG'}
SHAP_DISPLAY = {'window_peak_drr': 'Window peak', 'window_mean_drr': 'Window mean',
                'window_rms_drr': 'Window RMS'}
SHAP_MODEL_NAMES = {
    'LR': 'Logistic Regression', 'RIDGE': 'Ridge Classifier with Probability Calibration',
    'LINEAR_SVC': 'Linear Support Vector Classifier with Probability Calibration',
    'SVM': 'Support Vector Machine', 'SGD': 'Stochastic Gradient Descent Classifier',
    'KNN': 'K-Nearest Neighbors', 'GNB': 'Gaussian Naive Bayes',
    'LDA': 'Linear Discriminant Analysis', 'QDA': 'Quadratic Discriminant Analysis',
    'GPC': 'Gaussian Process Classifier', 'DT': 'Decision Tree', 'RF': 'Random Forest',
    'ET': 'Extremely Randomized Trees', 'GBDT': 'Gradient Boosting Decision Trees',
    'HGB': 'Histogram-based Gradient Boosting', 'ADA': 'Adaptive Boosting',
    'BAG': 'Bootstrap Aggregating', 'XGB': 'Extreme Gradient Boosting',
    'LGBM': 'Light Gradient Boosting Machine', 'CAT': 'Categorical Boosting',
    'ANN': 'Artificial Neural Network'}


def _all_shap_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def _all_shap_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf8')


def _all_shap_csv(*args, **kwargs):
    kwargs.setdefault('float_precision', 'round_trip')
    return pd.read_csv(*args, **kwargs)


def _all_shap_input(root, out, scheme):
    data = _all_shap_csv(Path(root)/'data/peak_windows_13points.csv')
    assignment = _all_shap_csv(Path(out)/scheme/'split_assignment.csv')
    work = data.merge(assignment[['window_id', 'split']], on='window_id', validate='one_to_one')
    train = work.query("split=='train'")
    test = work.query("split=='test'").reset_index(drop=True)
    # Same empirical background rule as the existing GNB/ANN analysis.
    background = pd.DataFrame([
        part.sort_values('window_id').iloc[len(part)//2]
        for _, part in train.groupby('cycle_id')]).reset_index(drop=True)
    assert set(background.window_id).issubset(train.window_id)
    assert set(background.window_id).isdisjoint(test.window_id)
    if scheme == 'cycle_grouped':
        assert set(background.cycle_id).isdisjoint(test.cycle_id)
    assert len(test) > 0 and len(background) > 0
    return train, test, background


def _all_shap_multiclass_array(value, n, f, classes=5, interaction=False):
    """Normalize public SHAP APIs, including CatBoost's list of class tensors."""
    shape = (n, f, f, classes) if interaction else (n, f, classes)
    if isinstance(value, list):
        array = np.stack(value, axis=-1)
    else:
        array = np.asarray(value)
    if array.shape != shape:
        raise ValueError(f'Unexpected SHAP shape {array.shape}; expected {shape}')
    return np.asarray(array, dtype=float)


def _all_shap_tree_tensors(pipeline, name, x):
    """TreeSHAP on fitted trees, including exact additive ensemble decomposition."""
    import shap
    estimator = pipeline[-1]
    z = np.asarray(pipeline[:-1].transform(x), dtype=float)
    n, f = x.shape
    assert z.shape == x.shape, 'A feature-changing transform needs an explicit mapping.'
    if name == 'XGB':
        # XGBoost 3.2 vector base scores exceed SHAP 0.49's Python parser.
        # Its native exact TreeSHAP has the same values/interaction definition.
        import xgboost as xgb
        dm = xgb.DMatrix(z)
        booster = estimator.get_booster()
        contribution = booster.predict(dm, pred_contribs=True, strict_shape=True)
        tensor = booster.predict(dm, pred_interactions=True, strict_shape=True)
        values = contribution[:, :, :-1].transpose(0, 2, 1).astype(float)
        base = contribution[:, :, -1].astype(float)
        interactions = tensor[:, :, :-1, :-1].transpose(0, 2, 3, 1).astype(float)
        target = booster.predict(dm, output_margin=True, strict_shape=True).astype(float)
        method, scale = 'Native XGBoost TreeSHAP', 'raw class margin'
    elif name == 'GBDT':
        # Multiclass GBDT is an additive sum of public fitted regression trees.
        # Shapley values and Shapley interaction values are linear in that sum.
        values = np.zeros((n, f, 5), dtype=float)
        interactions = np.zeros((n, f, f, 5), dtype=float)
        base = np.zeros((n, 5), dtype=float)
        tree_predictions = np.zeros((n, 5), dtype=float)
        for stage in estimator.estimators_:
            assert len(stage) == 5
            for c, tree in enumerate(stage):
                ex = shap.TreeExplainer(tree, feature_perturbation='tree_path_dependent')
                values[:, :, c] += estimator.learning_rate*np.asarray(ex.shap_values(z))
                interactions[:, :, :, c] += estimator.learning_rate*np.asarray(ex.shap_interaction_values(z))
                base[:, c] += estimator.learning_rate*float(np.asarray(ex.expected_value).item())
                tree_predictions[:, c] += estimator.learning_rate*tree.predict(z)
        target = np.asarray(estimator.decision_function(z), dtype=float)
        intercept = target-tree_predictions
        if not np.allclose(intercept, intercept[0], rtol=0, atol=1e-10):
            raise ValueError('GBDT initialization varies with features; constant-base decomposition is invalid.')
        base += intercept[0]
        method, scale = 'Additive per-class regression-tree TreeSHAP', 'raw class margin'
    elif name == 'BAG':
        from sklearn.tree import DecisionTreeClassifier
        # Bagging probabilities are the arithmetic mean of fitted tree probabilities.
        # Map feature subsampling and class columns before averaging exact tensors.
        values = np.zeros((n, f, 5), dtype=float)
        interactions = np.zeros((n, f, f, 5), dtype=float)
        base = np.zeros((n, 5), dtype=float)
        trees = estimator.estimators_
        for tree, feature_ids in zip(trees, estimator.estimators_features_):
            if not isinstance(tree, DecisionTreeClassifier):
                raise TypeError('Exact BAG adapter requires fitted DecisionTreeClassifier bases.')
            feature_ids = np.asarray(feature_ids, dtype=int)
            if len(set(feature_ids)) != len(feature_ids):
                raise ValueError('Duplicate bootstrap feature indices require a different coalition mapping.')
            ex = shap.TreeExplainer(tree, feature_perturbation='tree_path_dependent')
            v = np.asarray(ex.shap_values(z[:, feature_ids]))
            inter = np.asarray(ex.shap_interaction_values(z[:, feature_ids]))
            expected = np.asarray(ex.expected_value)
            assert v.shape == (n, len(feature_ids), len(tree.classes_))
            assert inter.shape == (n, len(feature_ids), len(feature_ids), len(tree.classes_))
            for j, label in enumerate(tree.classes_):
                c = int(label)
                assert 0 <= c < 5 and c == label
                values[:, feature_ids, c] += v[:, :, j]/len(trees)
                for local_i, global_i in enumerate(feature_ids):
                    interactions[:, global_i, feature_ids, c] += inter[:, local_i, :, j]/len(trees)
                base[:, c] += expected[j]/len(trees)
        target = np.asarray(pipeline.predict_proba(x), dtype=float)
        method, scale = 'Averaged bagged-tree probability TreeSHAP', 'class probability'
    else:
        ex = shap.TreeExplainer(estimator, feature_perturbation='tree_path_dependent')
        values = _all_shap_multiclass_array(ex.shap_values(z), n, f)
        interactions = _all_shap_multiclass_array(ex.shap_interaction_values(z), n, f, interaction=True)
        base = np.broadcast_to(np.asarray(ex.expected_value, dtype=float), (n, 5)).copy()
        method = 'TreeExplainer (tree_path_dependent)'
        if name in {'DT', 'RF', 'ET'}:
            target = np.asarray(pipeline.predict_proba(x), dtype=float)
            scale = 'class probability'
        elif name == 'LGBM':
            target = np.asarray(estimator.predict(z, raw_score=True), dtype=float)
            scale = 'raw class margin'
        elif name == 'CAT':
            target = np.asarray(estimator.predict(z, prediction_type='RawFormulaVal', thread_count=1), dtype=float)
            scale = 'raw class margin'
        elif name == 'HGB':
            target = np.asarray(estimator.decision_function(z), dtype=float)
            scale = 'raw class margin'
        else:
            raise ValueError(name)
    return values, base, interactions, target, method, scale


def _all_shap_permutation(pipeline, x, background, features, permutations):
    import shap
    masker = shap.maskers.Independent(background, max_samples=len(background))
    explainer = shap.PermutationExplainer(pipeline.predict_proba, masker,
        feature_names=features, seed=42)
    explanation = explainer(x, max_evals=(2*len(features)+1)*permutations,
        batch_size=128, silent=True)
    return np.asarray(explanation.values, dtype=float), np.asarray(explanation.base_values, dtype=float)


def _all_shap_formal_hashes(out):
    out = Path(out)
    names = {'model.joblib', 'best_parameters.json', 'predictions.csv', 'metrics.csv',
        'per_class_metrics.csv', 'outer_fold_metrics.csv', 'split_assignment.csv',
        'model_comparison.csv', 'all_classification_metrics.csv', 'all_per_class_metrics.csv'}
    return {str(p.relative_to(out)): _all_shap_hash(p) for p in out.rglob('*')
        if p.is_file() and (p.name in names or p.suffix in {'.db', '.sqlite', '.sqlite3'})}


def calculate_all_model_shap(root, out, *, permutations=128, schemes=None, models=None):
    """Explain all held-out windows. Resume only when every numerical input matches."""
    import shap
    root, out = Path(root), Path(out)
    schemes = schemes or ['row_random', 'cycle_grouped']
    comparison = _all_shap_csv(out/'model_comparison.csv')
    settings = json.loads((out/'settings.json').read_text(encoding='utf8'))
    feature_set = settings['feature_set']
    if feature_set not in {'raw_only', 'raw_plus_stats'}:
        raise ValueError(f'Unsupported feature set: {feature_set}')
    features = SHAP_FEATURES if feature_set == 'raw_plus_stats' else SHAP_FEATURES[:13]
    dest = out/'all_models_shap'
    dest.mkdir(parents=True, exist_ok=True)
    before = _all_shap_formal_hashes(out)
    _all_shap_json(dest/'formal_artifact_hashes_before.json', before)
    summaries = []
    with threadpool_limits(limits=1), warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='X does not have valid feature names.*')
        for scheme in schemes:
            train, test, bg = _all_shap_input(root, out, scheme)
            x = test[features].to_numpy(float)
            background = bg[features].to_numpy(float)
            names = comparison.loc[comparison.scheme == scheme, 'model'].tolist()
            if models is not None:
                names = [name for name in names if name in models]
            for name in names:
                start = time.perf_counter()
                folder = dest/scheme/name
                folder.mkdir(parents=True, exist_ok=True)
                model_path = out/scheme/name/'model.joblib'
                pipeline = joblib.load(model_path)
                probability = np.asarray(pipeline.predict_proba(x), dtype=float)
                prediction = _all_shap_csv(out/scheme/name/'predictions.csv')
                heldout = prediction.query("split=='test'").set_index('window_id').loc[test.window_id]
                assert np.array_equal(np.asarray(pipeline.predict(x)).reshape(-1), heldout.y_pred.to_numpy())
                pred_error = float(np.max(np.abs(probability-heldout[[f'p{i}' for i in range(5)]].to_numpy())))
                assert pred_error < 1e-7, (scheme, name, pred_error)
                assert np.array_equal(pipeline.classes_, np.arange(5))
                tree = name in SHAP_TREE_MODELS
                key = dict(adapter_version=1, method_family='TreeSHAP' if tree else 'PermutationExplainer',
                    permutations=None if tree else permutations, seed=42, features=features,
                    model_sha256=_all_shap_hash(model_path), data_sha256=_all_shap_hash(root/'data/peak_windows_13points.csv'),
                    train_windows=train.window_id.tolist(), background_windows=bg.window_id.tolist(),
                    test_windows=test.window_id.tolist(), shap_version=shap.__version__)
                cache, key_path = folder/'values.npz', folder/'calculation_key.json'
                hit = cache.exists() and key_path.exists() and json.loads(key_path.read_text(encoding='utf8')) == key
                print(f'SHAP {scheme}/{name}: '+('verified cache' if hit else 'calculating '+key['method_family']), flush=True)
                if hit:
                    with np.load(cache, allow_pickle=False) as saved:
                        values, base, target = saved['values'], saved['base_values'], saved['explained_output']
                        interactions = saved['interactions'] if 'interactions' in saved.files else None
                        assert np.array_equal(saved['data'], x)
                    metadata = json.loads((folder/'summary.json').read_text(encoding='utf8'))
                    method, scale = metadata['method'], metadata['output_scale']
                elif tree:
                    values, base, interactions, target, method, scale = _all_shap_tree_tensors(pipeline, name, x)
                else:
                    values, base = _all_shap_permutation(pipeline, x, background, features, permutations)
                    interactions, target = None, probability
                    method, scale = 'PermutationExplainer (128 forward/reverse permutations)' if permutations == 128 else f'PermutationExplainer ({permutations} forward/reverse permutations)', 'class probability'
                assert values.shape == (len(test), len(features), 5)
                assert base.shape == target.shape == (len(test), 5)
                assert np.isfinite(values).all() and np.isfinite(base).all() and np.isfinite(target).all()
                errors = dict(additivity=float(np.max(np.abs(values.sum(axis=1)+base-target))),
                              saved_prediction_max_abs_error=pred_error)
                if interactions is not None:
                    assert interactions.shape == (len(test), len(features), len(features), 5)
                    assert np.isfinite(interactions).all()
                    errors.update(interaction_symmetry=float(np.max(np.abs(interactions-interactions.transpose(0, 2, 1, 3)))),
                        interaction_row_sum=float(np.max(np.abs(interactions.sum(axis=2)-values))))
                tolerance = 2e-5 if name == 'XGB' else 1e-7
                assert max(errors.values()) < tolerance, (scheme, name, errors)
                if scale == 'raw class margin':
                    shifted = target-target.max(axis=1, keepdims=True)
                    softmax = np.exp(shifted)/np.exp(shifted).sum(axis=1, keepdims=True)
                    errors['softmax_probability'] = float(np.max(np.abs(softmax-probability)))
                    assert errors['softmax_probability'] < tolerance
                else:
                    assert np.allclose(target, probability, rtol=0, atol=tolerance)
                importance = np.mean(np.abs(values), axis=0)
                summary = dict(scheme=scheme, model=name, full_name=SHAP_MODEL_NAMES[name], method=method,
                    output_scale=scale, test_windows=len(test), test_cycles=int(test.cycle_id.nunique()),
                    features=len(features), feature_names=features, background_windows=len(bg) if not tree else None,
                    background_definition='One real training window per cycle' if not tree else 'Fitted tree path statistics; no external background',
                    permutations=None if tree else permutations, seed=42,
                    interaction_available=interactions is not None, interaction_method='Exact TreeSHAP interactions' if tree else None,
                    interaction_unavailable_reason=None if tree else 'The supplied network requires exact TreeSHAP interactions; probability Permutation SHAP does not return that tensor.',
                    errors=errors, tolerance=tolerance, model_sha256=key['model_sha256'],
                    data_sha256=key['data_sha256'], cache_reused=hit,
                    elapsed_seconds=round(time.perf_counter()-start, 3),
                    top_features=[features[i] for i in np.argsort(importance.sum(axis=1))[::-1][:5]],
                    limitations='Model attribution only. Marginal masking or tree-path assumptions do not resolve correlated features. Windows from one cycle are dependent. Raw margins and probabilities have different units. No causal interpretation.',
                    original_plot_scale='Original stacked bars normalize the maximum total importance to 0.5. Per-class local SHAP axes and original 99th-percentile clipping are retained.')
                if not hit:
                    payload = dict(values=values, base_values=base, explained_output=target,
                        data=x, predicted_probabilities=probability, window_ids=test.window_id.to_numpy(dtype=str))
                    if interactions is not None:
                        payload['interactions'] = interactions
                    np.savez_compressed(cache, **payload)
                    _all_shap_json(key_path, key)
                _all_shap_json(folder/'summary.json', summary)
                test[['window_id', 'cycle_id', 'target_angle_deg']+features].to_csv(folder/'explained_test_windows.csv', index=False, encoding='utf-8-sig')
                if not tree:
                    bg[['window_id', 'cycle_id', 'target_angle_deg']+features].to_csv(folder/'training_background.csv', index=False, encoding='utf-8-sig')
                pd.DataFrame(importance, index=pd.Index(features, name='feature'), columns=SHAP_ANGLES).to_csv(folder/'mean_absolute_shap.csv', encoding='utf-8-sig')
                summaries.append(summary)
                _all_shap_json(dest/'calculation_progress.json', summaries)
                print(f'  validated; scale={scale}; interactions={interactions is not None}; seconds={summary["elapsed_seconds"]}', flush=True)
    assert before == _all_shap_formal_hashes(out), 'Formal training artifacts changed during SHAP calculation.'
    _all_shap_json(dest/'calculation_summary.json', summaries)
    table = pd.DataFrame([{k: v for k, v in s.items() if k not in {'errors', 'top_features', 'feature_names'}} | {
        'max_additivity_error': s['errors']['additivity'], 'top_features': '; '.join(s['top_features'])} for s in summaries])
    table.to_csv(dest/'model_method_summary.csv', index=False, encoding='utf-8-sig')
    return summaries


def render_all_model_shap(root, out, summaries, *, executor_class, source_reader):
    """Run the supplied original SHAP and network cells, without new plot functions."""
    root, out = Path(root), Path(out)
    audit = out/'all_models_shap'
    sources = source_reader()
    runner = executor_class(sources, out/'figures/original_templates')
    before = _all_shap_formal_hashes(out)
    for summary in summaries:
        scheme, name = summary['scheme'], summary['model']
        folder = audit/scheme/name
        with np.load(folder/'values.npz', allow_pickle=False) as saved:
            values = saved['values']
            x = saved['data']
            interactions = saved['interactions'] if 'interactions' in saved.files else None
        features = summary.get('feature_names', SHAP_FEATURES)
        frame = pd.DataFrame(x, columns=features).rename(columns=SHAP_DISPLAY)
        env = runner.env()
        env.update(shap_values=values, X_test=frame, class_names=[f'{a} deg' for a in SHAP_ANGLES])
        runner.execute('shap', 10, env)
        filename = 'xgboost_multiclass_shap_importance-1' if name == 'XGB' else 'multiclass_shap_importance'
        context = dict(scheme=scheme, model=name, method=summary['method'], scale=summary['output_scale'],
            n=len(x), features=features, numeric_file=str(folder/'values.npz'))
        with runner.capture(f'{scheme}/{name}/{filename}', 'shap', context):
            # Use the original exposed spacing argument for 16 rather than 12 rows.
            runner.execute('shap', 14, env, {'local_axis_fraction=0.34':
                'local_axis_fraction=0.34, bottom_label_offset=1.0'})
        if interactions is not None:
            for block in [12, 16, 21, 25, 26]:
                runner.execute('network', block, env)
            matrices = env['get_class_interaction_matrices'](interactions, frame, env['class_names'])
            pairs = env['interaction_matrices_to_df'](matrices, multiply_interaction_by_2=True, include_main_effect=False)
            importance = env['get_class_shap_importance'](values, frame, env['class_names'])
            pairs.to_csv(folder/'interaction_pairs_original_definition.csv', index=False, encoding='utf-8-sig')
            env.update(interaction_df_all=pairs[pairs.interaction_value > 1e-8], importance_dict=importance)
            counts = {}
            for angle, original_class, block in zip(SHAP_ANGLES, ['MVT', 'SEDEX', 'VMS', 'epithermal', 'skarn'], [27, 29, 31, 33, 35]):
                with runner.capture(f'{scheme}/{name}/feature_network_{angle}', 'network', context | dict(class_angle=angle)):
                    with contextlib.redirect_stdout(io.StringIO()):
                        runner.execute('network', block, env, {original_class: f'{angle} deg'})
                counts[str(angle)] = int(((pairs.class_name == f'{angle} deg') & (pairs.interaction_value > 1e-8)).sum())
            summary['nonzero_interaction_pairs'] = counts
            _all_shap_json(folder/'summary.json', summary)
        print(f'ORIGINAL SHAP EXPORT {scheme}/{name}: {6 if interactions is not None else 1} figures', flush=True)
        del env
        gc.collect()
    assert before == _all_shap_formal_hashes(out)
    _all_shap_json(audit/'calculation_summary.json', summaries)
    _all_shap_json(audit/'figure_manifest.json', runner.figures)
    _all_shap_json(audit/'execution_provenance.json', runner.executions)
    # Merge by actual output path: the two existing XGB sets are replaced in place.
    main_audit = out/'original_template_audit'
    old_manifest = json.loads((main_audit/'figure_manifest.json').read_text(encoding='utf8')) if (main_audit/'figure_manifest.json').exists() else []
    merged = {str(Path(record['svg']).resolve()): record for record in old_manifest}
    merged.update({str(Path(record['svg']).resolve()): record for record in runner.figures})
    all_figures = list(merged.values())
    _all_shap_json(main_audit/'figure_manifest.json', all_figures)
    pd.DataFrame([dict(svg=str(Path(f['svg']).relative_to(out)), pdf=str(Path(f['pdf']).relative_to(out)),
        family=f['family']) for f in all_figures]).to_csv(out/'figure_index.csv', index=False, encoding='utf-8-sig')
    _all_shap_json(audit/'validation.json', dict(models_explained=len(summaries),
        model_names=sorted({s['model'] for s in summaries}), schemes=sorted({s['scheme'] for s in summaries}),
        all_models_have_original_shap_combo=True, tree_interaction_models=sorted(SHAP_TREE_MODELS),
        shap_combo_figures=len(summaries), network_figures=sum(5 for s in summaries if s['interaction_available']),
        figure_count=len(runner.figures), total_original_template_figures=len(all_figures),
        formal_training_artifacts_unchanged=True, self_authored_plot_functions=0,
        output_formats=['SVG', 'PDF'], font='Arial', output_text_editable=True))
    if (main_audit/'validation.json').exists():
        main_validation = json.loads((main_audit/'validation.json').read_text(encoding='utf8'))
        main_validation.update(figure_count=len(all_figures), all_model_shap=True,
            all_model_shap_validation=str(audit/'validation.json'))
        _all_shap_json(main_audit/'validation.json', main_validation)
    print(f'ALL MODEL SHAP COMPLETE: {len(summaries)} model/split combinations; {len(runner.figures)} SHAP figures; {len(all_figures)} total original figures', flush=True)
    return runner.figures


def run_all_model_shap(root, out, *, executor_class, source_reader, permutations=128):
    summaries = calculate_all_model_shap(root, out, permutations=permutations)
    figures = render_all_model_shap(root, out, summaries,
        executor_class=executor_class, source_reader=source_reader)
    return summaries, figures


## 6. 执行训练与评估
### 6.1 Optuna及训练日志

In [ ]:
STORAGE='sqlite:///'+(OUT/'studies.sqlite3').resolve().as_posix()
STUDY_INDEX=[]
FIT_WARNINGS=[]
def fit_model(model,x_values,y_values,context):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        model.fit(x_values,y_values)
    if caught:
        FIT_WARNINGS.append(dict(context=context,n=len(caught),messages=' | '.join(sorted(set(str(w.message)[:250] for w in caught)))))
    return model
def history_figure(study,title):
    return original_history_figure(study)

def live_callback(handle,title):
    def cb(study,trial):
        if trial.number%LIVE_EVERY==0:
            fig=history_figure(study,title)
            if fig is None:return
            vectorize(fig);buf=StringIO();fig.savefig(buf,format='svg',bbox_inches='tight')
            if handle[0] is None:handle[0]=display(SVG(buf.getvalue()),display_id=True)
            else:handle[0].update(SVG(buf.getvalue()))
            plt.close(fig)
    return cb
def tune(name,scheme,global_indices,stage,n_trials,seed):
    xx=X[global_indices];yy=y[global_indices];gg=groups[global_indices]
    folds=make_folds(yy,gg,scheme,CFG['inner_folds'],seed)
    study_name=f'{scheme}__{name}__{stage}'
    sampler=optuna.samplers.TPESampler(seed=seed,n_startup_trials=5)
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5,n_warmup_steps=1)
    study=optuna.create_study(study_name=study_name,storage=STORAGE,load_if_exists=RESUME,
        direction='maximize',sampler=sampler,pruner=pruner)
    study.set_user_attr('scope','current training subset only');study.set_user_attr('actual_inner_folds',len(folds))
    def objective(trial):
        params=suggest_params(trial,name);scores=[]
        for fold,(a,b) in enumerate(folds):
            model=build_model(name,params,seed+fold,scheme,yy[a],gg[a])
            fit_model(model,xx[a],yy[a],f'{study_name}/trial{trial.number}/fold{fold}')
            p=aligned_probabilities(model,xx[b])
            scores.append(skm.f1_score(yy[b],p.argmax(1),average='macro',labels=np.arange(5),zero_division=0))
            trial.report(float(np.mean(scores)),step=fold)
            if trial.should_prune():raise optuna.TrialPruned()
        trial.set_user_attr('fold_macro_f1',scores)
        return float(np.mean(scores))
    callbacks=[live_callback([None],f'{SCHEME_LABELS[scheme]} / {name}')] if LIVE_OPTUNA and stage=='final' else []
    remaining=max(0,n_trials-len(study.trials))
    if remaining:study.optimize(objective,n_trials=remaining,callbacks=callbacks,catch=(ValueError,RuntimeError,FloatingPointError),show_progress_bar=False)
    completed=[t for t in study.trials if t.state==TrialState.COMPLETE]
    if not completed:raise RuntimeError(f'{study_name}: no successful trials')
    dest=OUT/scheme/name;dest.mkdir(parents=True,exist_ok=True)
    write_csv(study.trials_dataframe(),dest/f'{stage}_all_trials.csv')
    STUDY_INDEX.append(dict(scheme=scheme,model=name,stage=stage,study_name=study_name,
        completed=len(completed),actual_inner_folds=len(folds),best_value=study.best_value))
    return study


### 6.2 嵌套交叉验证与最终拟合

In [ ]:
RUNS=[];RUN_ERRORS=[]
def train_one(name,scheme):
    start=time.perf_counter();train_idx,test_idx=SPLITS[scheme]
    # 分组训练每类只有3～4周期时，外层至少3折以保证每个外层fit子集每类>=2周期，才能再做内层CV。
    outer_requested=max(3,CFG['outer_folds']) if scheme=='cycle_grouped' else CFG['outer_folds']
    outer=make_folds(y[train_idx],groups[train_idx],scheme,outer_requested,SEED+100)
    oof_p=np.full((len(train_idx),5),np.nan);oof_fold=np.full(len(train_idx),-1);fold_rows=[];fold_params=[]
    for k,(a,b) in enumerate(outer):
        fit_indices=train_idx[a];val_indices=train_idx[b]
        study=tune(name,scheme,fit_indices,f'outer{k+1}',CFG['nested_trials'],SEED+200+k)
        model=build_model(name,study.best_params,SEED+k,scheme,y[fit_indices],groups[fit_indices])
        fit_model(model,X[fit_indices],y[fit_indices],f'{scheme}/{name}/outer{k+1}/refit')
        p=aligned_probabilities(model,X[val_indices]);oof_p[b]=p;oof_fold[b]=k+1
        mm,_=classification_metrics(y[val_indices],p);mm['fold']=k+1;fold_rows.append(mm)
        fold_params.append(dict(fold=k+1,params=study.best_params,inner_cv_macro_f1=study.best_value,
            fit_windows=len(a),validation_windows=len(b)))
    assert np.isfinite(oof_p).all() and (oof_fold>0).all()
    final_study=tune(name,scheme,train_idx,'final',CFG['final_trials'],SEED+500)
    model=build_model(name,final_study.best_params,SEED,scheme,y[train_idx],groups[train_idx])
    fit_model(model,X[train_idx],y[train_idx],f'{scheme}/{name}/final')
    ptr=aligned_probabilities(model,X[train_idx]);pte=aligned_probabilities(model,X[test_idx])
    all_metrics=[];all_pc=[]
    for split,yy,pp in [('train_in_sample',y[train_idx],ptr),('nested_oof',y[train_idx],oof_p),('test',y[test_idx],pte)]:
        mm,pc=classification_metrics(yy,pp);all_metrics.append(dict(split=split,**mm));pc.insert(0,'split',split);all_pc.append(pc)
    frames=[prediction_frame(train_idx,ptr,'train_in_sample'),prediction_frame(test_idx,pte,'test')]
    poof=prediction_frame(train_idx,oof_p,'nested_oof');poof['fold']=oof_fold;frames.append(poof)
    dest=OUT/scheme/name
    write_csv(pd.DataFrame(all_metrics),dest/'metrics.csv');write_csv(pd.concat(all_pc),dest/'per_class_metrics.csv')
    write_csv(pd.concat(frames),dest/'predictions.csv');write_csv(pd.DataFrame(fold_rows),dest/'outer_fold_metrics.csv')
    report=skm.classification_report(y[test_idx],pte.argmax(1),labels=np.arange(5),target_names=CLASS_LABELS,zero_division=0,output_dict=True)
    report_frame=pd.DataFrame({k:v for k,v in report.items() if isinstance(v,dict)}).T
    report_frame.loc['accuracy']=[np.nan,np.nan,float(report['accuracy']),len(test_idx)]
    report_frame=report_frame.reindex(CLASS_LABELS+['accuracy','macro avg','weighted avg'])
    write_csv(report_frame.rename_axis('class').reset_index(),dest/'classification_report_test.csv')
    write_json(fold_params,dest/'outer_fold_parameters.json')
    write_json(dict(params=final_study.best_params,inner_cv_macro_f1=final_study.best_value,
        selected_by='nested outer-fold macro F1',scope=SCHEME_LABELS[scheme],scaling=SCALE_MAP[name]),dest/'best_parameters.json')
    joblib.dump(model,dest/'model.joblib')
    folds=pd.DataFrame(fold_rows)
    result=dict(scheme=scheme,model=name,nested_cv_f1_mean=folds.f1_macro.mean(),
        nested_cv_f1_sd=folds.f1_macro.std(ddof=1),final_inner_cv_f1=final_study.best_value,
        actual_outer_folds=len(outer),elapsed_s=time.perf_counter()-start,folder=str(dest))
    result.update({f'test_{k}':v for k,v in all_metrics[-1].items() if k!='split'})
    return result


### 6.3 训练21模型×2种划分
排名只依据训练侧的外层交叉验证宏F1，保留逐折波动。测试集不用于调参或挑选模型。

In [ ]:
if RUN_MODE=='train_and_plot':
    for scheme in ['row_random','cycle_grouped']:
        print('\nSCHEME:',SCHEME_LABELS[scheme],flush=True)
        for number,name in enumerate(MODELS,1):
            print(f'[{number:02d}/21] {name}',end=' ',flush=True)
            try:
                result=train_one(name,scheme);RUNS.append(result)
                print(f"done; nested F1={result['nested_cv_f1_mean']:.3f}; {result['elapsed_s']:.1f}s",flush=True)
            except Exception as exc:
                RUN_ERRORS.append(dict(scheme=scheme,model=name,error=repr(exc)))
                print('FAILED:',repr(exc),flush=True)
            write_csv(pd.DataFrame(RUNS),OUT/'model_comparison.csv')
            write_csv(pd.DataFrame(RUN_ERRORS,columns=['scheme','model','error']),OUT/'run_errors.csv')
    write_csv(pd.DataFrame(STUDY_INDEX),OUT/'study_index.csv')
    write_csv(pd.DataFrame(FIT_WARNINGS),OUT/'fit_warnings.csv')
    write_csv(pd.DataFrame(CALIBRATION_LOG),OUT/'calibration_internal_fallbacks.csv')
    if RUN_ERRORS:raise RuntimeError(f'{len(RUN_ERRORS)}条模型流程失败，见run_errors.csv；必须修复后再作为完整运行。')
else:
    _saved_comparison=OUT/'model_comparison.csv'
    if not _saved_comparison.exists():raise FileNotFoundError('没有已有训练结果；请设置RUN_MODE=train_and_plot。')
    RUNS=pd.read_csv(_saved_comparison,float_precision='round_trip').to_dict('records')
    for _row in RUNS:_row['folder']=str(OUT/_row['scheme']/_row['model'])
    RUN_ERRORS=[]
    print('复用42条已保存训练结果；本次不训练。')
comparison=pd.DataFrame(RUNS)
assert len(comparison)==42 and comparison.groupby('scheme').model.nunique().eq(21).all()
cv_selected={scheme:part.sort_values(['nested_cv_f1_mean','nested_cv_f1_sd','model'],ascending=[False,True,True]).iloc[0].model
          for scheme,part in comparison.groupby('scheme')}
write_json(cv_selected,OUT/'selected_by_nested_cv.json')
selected = dict(cv_selected)
top3 = {scheme: part.sort_values(["nested_cv_f1_mean", "nested_cv_f1_sd", "model"],
             ascending=[False, True, True]).head(3).model.tolist()
        for scheme, part in comparison.groupby("scheme")}
write_json(dict(models=selected, top3=top3, basis="Outer nested-CV macro F1"), OUT/'selected_for_paper.json')
display(comparison[['scheme','model','nested_cv_f1_mean','nested_cv_f1_sd','test_accuracy','test_f1_macro','test_auc_macro']].round(4))
print('Outer-CV highest-ranked models:', cv_selected)
print('Top three per split:', top3)


### 6.4 汇总全部指标与逐类别报告

In [ ]:
metric_tables=[];perclass_tables=[]
for run in RUNS:
    for name,store in [('metrics.csv',metric_tables),('per_class_metrics.csv',perclass_tables)]:
        frame=pd.read_csv(Path(run['folder'])/name,float_precision='round_trip')
        frame.insert(0,'model',run['model']);frame.insert(0,'scheme',run['scheme']);store.append(frame)
all_metrics=pd.concat(metric_tables,ignore_index=True);all_per_class=pd.concat(perclass_tables,ignore_index=True)
if RUN_MODE=='train_and_plot':
    write_csv(all_metrics,OUT/'all_classification_metrics.csv');write_csv(all_per_class,OUT/'all_per_class_metrics.csv')
for scheme,name in selected.items():
    print(SCHEME_LABELS[scheme],name)
    display(all_metrics[(all_metrics.scheme==scheme)&(all_metrics.model==name)].round(4))
    display(pd.read_csv(OUT/scheme/name/'classification_report_test.csv').round(4))


## 7. 绘图
### 7.1 百分比混淆矩阵、ROC、Optuna和指标图
同一原模板遍历全部模型。百分比混淆矩阵的对角线是该真实类别的正确分类率（召回率）。

In [ ]:
ORIGINAL_FIGURES=run_original_template_plots(ROOT,OUT)
for scheme in ['row_random','cycle_grouped']:
    display(SVG(filename=str(OUT/'figures/original_templates'/scheme/'XGB/xgboost_multiclass_shap_importance-1.svg')))


### 7.2 全模型SHAP与类别交互图
正式配置默认128次置换；smoke默认2次，仅用于程序验证。所有结果保存在当前运行输出目录。

In [ ]:
SHAP_SUMMARIES, ALL_MODEL_SHAP_FIGURES = run_all_model_shap(
    ROOT, OUT, executor_class=OriginalExecutor, source_reader=read_originals, permutations=SHAP_PERMUTATIONS)
_all_figures={str(Path(f['svg']).resolve()):f for f in ORIGINAL_FIGURES}
_all_figures.update({str(Path(f['svg']).resolve()):f for f in ALL_MODEL_SHAP_FIGURES})
ORIGINAL_FIGURES=list(_all_figures.values())
display(pd.DataFrame([{k:s[k] for k in ['scheme','model','method','output_scale','interaction_available','test_windows']}
    for s in SHAP_SUMMARIES]))
print('全模型SHAP图:',len(ALL_MODEL_SHAP_FIGURES),'全部原模板图:',len(ORIGINAL_FIGURES))


### 7.3 查看模型解释样例

In [ ]:
SHAP_PREVIEW_MODEL='GNB'
for _scheme in ['row_random','cycle_grouped']:
    _base='xgboost_multiclass_shap_importance-1' if SHAP_PREVIEW_MODEL=='XGB' else 'multiclass_shap_importance'
    print(_scheme,SHAP_PREVIEW_MODEL)
    display(SVG(filename=str(OUT/'figures/original_templates'/_scheme/SHAP_PREVIEW_MODEL/(_base+'.svg'))))


### 7.4 复用现有组图排版函数
只拼接生成的SVG/PDF面板，保留曲线、色带、数值与原面板文字。

In [ ]:
import math
import pymupdf as fitz
import xml.etree.ElementTree as ET
svg_ns = "http://www.w3.org/2000/svg"
ET.register_namespace("", svg_ns)
ET.register_namespace("xlink", "http://www.w3.org/1999/xlink")
arial = Path(ARIAL_PATH)
PAPER_FIGURE_MANIFEST = []
def panel_group(key, paths, labels, columns=2, caption='', cell_width=None, output_dir=None):
    """Place untouched source SVG/PDF pages; only panel headings are added."""
    paths=[Path(p) for p in paths]
    dimensions=[]
    for p in paths:
        with fitz.open(p.with_suffix('.pdf')) as d: dimensions.append((d[0].rect.width,d[0].rect.height))
    cw=cell_width or max(w for w,h in dimensions)
    # Uniform scale preserves each source panel's aspect ratio and axes.
    ch=max(h*cw/w for w,h in dimensions)+28
    rows=math.ceil(len(paths)/columns); gap=16; pad=12
    width=2*pad+columns*cw+(columns-1)*gap
    height=2*pad+rows*ch+(rows-1)*gap
    svg=ET.Element(f'{{{svg_ns}}}svg',{'width':f'{width}pt','height':f'{height}pt','viewBox':f'0 0 {width} {height}'})
    ET.SubElement(svg,f'{{{svg_ns}}}rect',{'width':str(width),'height':str(height),'fill':'white'})
    pdf=fitz.open(); page=pdf.new_page(width=width,height=height)
    page.insert_font(fontname='ArialTask',fontfile=str(arial))
    source_records=[]
    for i,(p,label,(sw,sh)) in enumerate(zip(paths,labels,dimensions)):
        x=pad+(i%columns)*(cw+gap); y=pad+(i//columns)*(ch+gap)
        head=ET.SubElement(svg,f'{{{svg_ns}}}text',{'x':str(x+4),'y':str(y+17),'font-family':'Arial','font-size':'15','font-weight':'bold','fill':'black'})
        head.text=f'({chr(97+i)}) {label}'
        page.insert_text((x+4,y+17),head.text,fontname='ArialTask',fontsize=15)
        node=ET.parse(p).getroot(); vb=node.attrib.get('viewBox',f'0 0 {sw} {sh}').split(); vw,vh=float(vb[2]),float(vb[3])
        prefix=f'{key}_{i}_'; ids={e.attrib['id']:prefix+e.attrib['id'] for e in node.iter() if 'id' in e.attrib}
        for e in node.iter():
            if 'id' in e.attrib: e.attrib['id']=ids[e.attrib['id']]
            for a,v in list(e.attrib.items()):
                if a=='id': continue
                for old,new in ids.items():
                    v=v.replace(f'url(#{old})',f'url(#{new})')
                    if v==f'#{old}':v=f'#{new}'
                e.attrib[a]=v
        group=ET.SubElement(svg,f'{{{svg_ns}}}g',{'transform':f'translate({x},{y+26}) scale({cw/vw})'})
        for child in list(node): group.append(child)
        with fitz.open(p.with_suffix('.pdf')) as original:
            page.show_pdf_page(fitz.Rect(x,y+26,x+cw,y+26+sh*cw/sw),original,0)
        source_records.append({'svg':str(p.resolve()),'svg_sha256':sha(p),'pdf_sha256':sha(p.with_suffix('.pdf')),'label':label})
    target=Path(output_dir or OUT)/key
    ET.ElementTree(svg).write(target.with_suffix('.svg'),encoding='utf8',xml_declaration=True)
    pdf.save(target.with_suffix('.pdf'),garbage=4,deflate=True)
    scale=min(3,2600/width)
    page.get_pixmap(matrix=fitz.Matrix(scale,scale),alpha=False).save(target.with_suffix('.png'))
    pdf.close()
    record={'key':key,'caption':caption,'svg':str(target.with_suffix('.svg')),'pdf':str(target.with_suffix('.pdf')),'png':str(target.with_suffix('.png')),'source_panels':source_records,'operation':'Vector placement only; original axes, colors, curves, data and labels retained'}
    PAPER_FIGURE_MANIFEST.append(record)
    return record


### 7.5 九模型与分类组图、前三模型SHAP组图

In [ ]:
PAPER_GROUPS = {
    "discriminant": ["GNB", "LDA", "QDA"],
    "boundary": ["LINEAR_SVC", "SVM", "KNN"],
    "ensemble": ["RF", "ET", "XGB"],
}
NINE_MODELS = ["GNB", "LDA", "QDA", "LINEAR_SVC", "SVM", "KNN", "RF", "ET", "XGB"]
for _scheme in ["cycle_grouped", "row_random"]:
    _folder = OUT / "figures" / "paper_groups" / _scheme
    _folder.mkdir(parents=True, exist_ok=True)
    _panels = OUT / "figures/original_templates" / _scheme
    for _family, _names in PAPER_GROUPS.items():
        panel_group("confusion_matrix_"+_family, [_panels/n/"confusion_matrix.svg" for n in _names],
                    _names, columns=2, output_dir=_folder)
    panel_group("nine_confusion_matrix", [_panels/n/"confusion_matrix.svg" for n in NINE_MODELS],
                NINE_MODELS, columns=3, output_dir=_folder)
    panel_group("nine_multiclass_roc", [_panels/n/"test_multiclass_roc.svg" for n in NINE_MODELS],
                NINE_MODELS, columns=3, output_dir=_folder)
    _shap_paths = [_panels/n/("xgboost_multiclass_shap_importance-1.svg" if n=="XGB" else "multiclass_shap_importance.svg")
                   for n in top3[_scheme]]
    panel_group("top3_multiclass_shap", _shap_paths, top3[_scheme], columns=1, output_dir=_folder)
write_json(PAPER_FIGURE_MANIFEST, OUT / "paper_figure_manifest.json")
print("Paper composites:", len(PAPER_FIGURE_MANIFEST))
display(SVG(filename=str(OUT/"figures/paper_groups/cycle_grouped/nine_confusion_matrix.svg")))


### 7.6 完整文字标签与按类别编辑
为SVG保留完整文字对象并增加类别标记，同时写出可供用户在Illustrator中手动运行的字体选择脚本。本Notebook不调用桌面应用。

In [ ]:
# 只增加文字类别标记，保留原SVG中的每个文字、坐标与绘制顺序。
import re
from collections import Counter
def tag_complete_svg_labels(path):
    tree = ET.parse(path)
    root = tree.getroot()
    parents = {child: parent for parent in root.iter() for child in parent}
    texts = [t for t in root.iter() if t.tag.endswith("}text")]
    before = [(ET.tostring(t), dict(t.attrib)) for t in texts]
    roles = Counter()
    for number, t in enumerate(texts, 1):
        value = "".join(t.itertext())
        ancestors = []
        p = parents.get(t)
        while p is not None:
            ancestors.append(p.get("id", "")); p = parents.get(p)
        context = "/".join(ancestors)
        if re.fullmatch(r"raw_\d\d|Window (peak|RMS|mean)", value): role = "05_FEATURES"
        elif "legend" in context: role = "04_LEGENDS"
        elif re.search(r"[xy]tick", context): role = "03_TICKS"
        elif re.fullmatch(r"[+−-]?\d*\.?\d+%?", value): role = "06_VALUES"
        elif "matplotlib.axis" in context: role = "02_AXIS_LABELS"
        else: role = "01_TITLES"
        parent = parents[t]
        wrapper = ET.Element("{"+svg_ns+"}g", id=f"TXT_{role}_LABEL_{number:05d}")
        place = list(parent).index(t)
        parent.remove(t); wrapper.append(t); parent.insert(place, wrapper)
        roles[role] += 1
    assert before == [(ET.tostring(t), dict(t.attrib)) for t in texts]
    tree.write(path, encoding="utf-8", xml_declaration=True)
    return dict(roles)
text_category_audit = {str(p.relative_to(OUT)): tag_complete_svg_labels(p) for p in sorted((OUT/"figures").rglob("*.svg"))}
write_json(text_category_audit, OUT/"svg_text_category_audit.json")
_FONT_HELPER_JSX = "// Run manually in Illustrator: File > Scripts > Other Script.\n// Operates on complete text frames; does not break labels into characters.\n(function () {\n    if (!app.documents.length) { alert('请先打开本包中的SVG或AI文件。'); return; }\n    var names = ['全部文字', '01 图题与子图字母', '02 坐标轴与指标标题', '03 刻度字母与数字', '04 图例字母与数字', '05 特征名称', '06 图内数值', '07 框架说明', '08 其他完整标签'];\n    var dialog = new Window('dialog', '按类别整体调整字体与字号');\n    dialog.orientation = 'column'; dialog.alignChildren = 'fill';\n    dialog.add('statictext', undefined, '字母和数字作为完整标签一起设置，保留图形与数据。');\n    var category = dialog.add('dropdownlist', undefined, names); category.selection = 0;\n    var fontGroup = dialog.add('group'); fontGroup.add('statictext', undefined, '字体（留空保留）：');\n    var fontName = fontGroup.add('edittext', undefined, 'ArialMT'); fontName.characters = 25;\n    var sizeGroup = dialog.add('group'); sizeGroup.add('statictext', undefined, '字号pt（留空保留）：');\n    var sizeInput = sizeGroup.add('edittext', undefined, ''); sizeInput.characters = 10;\n    var action = dialog.add('group');\n    var onlySelect = action.add('button', undefined, '仅选中这一类');\n    var apply = action.add('button', undefined, '统一应用');\n    action.add('button', undefined, '取消', {name: 'cancel'});\n    var applyChanges = false;\n    onlySelect.onClick = function () { applyChanges = false; dialog.close(1); };\n    apply.onClick = function () { applyChanges = true; dialog.close(1); };\n    if (dialog.show() !== 1) return;\n    var index = category.selection.index, selected = [], d = app.activeDocument;\n    function findRole(frame) {\n        var p = frame, levels = 0;\n        while (p && p.typename !== 'Document' && levels++ < 50) {\n            var n = p.name || '', m = /TXT[ _]+(0[1-8])[ _]+/.exec(n);\n            if (m) return Number(m[1]);\n            if (/^(0[1-8])_/.test(n) && /图题|坐标|刻度|图例|特征|数值|框架|完整标签/.test(n)) return Number(n.substr(0, 2));\n            p = p.parent;\n        }\n        return -1;\n    }\n    for (var i = 0; i < d.textFrames.length; i++) {\n        var t = d.textFrames[i];\n        if (index === 0 || findRole(t) === index) selected.push(t);\n    }\n    if (!selected.length) { alert('未找到该类别。请打开本次提供的分类SVG或AI副本。'); return; }\n    var chosenFont = null, requestedSize = null;\n    if (applyChanges) {\n        var f = fontName.text.replace(/^\\s+|\\s+$/g, '');\n        if (f === 'Arial') f = 'ArialMT';\n        if (f) { try { chosenFont = app.textFonts.getByName(f); } catch (error) { alert('未找到字体：' + f + '\\n请输入已安装字体的PostScript名称，如ArialMT。'); return; } }\n        if (sizeInput.text.replace(/\\s/g, '')) {\n            requestedSize = Number(sizeInput.text);\n            if (!(requestedSize > 0 && requestedSize < 1000)) { alert('字号应为0至1000之间的正数。'); return; }\n        }\n    }\n    d.selection = null;\n    for (i = 0; i < selected.length; i++) {\n        selected[i].selected = true;\n        if (applyChanges) {\n            if (chosenFont) selected[i].textRange.characterAttributes.textFont = chosenFont;\n            if (requestedSize !== null) selected[i].textRange.characterAttributes.size = requestedSize;\n        }\n    }\n    app.redraw();\n    alert('已' + (applyChanges ? '统一调整' : '选中') + selected.length + '个完整文字标签。字母、下划线、数字保留在同一文字对象中。');\n})();\n"
(OUT/"按类别统一调整文字.jsx").write_text(_FONT_HELPER_JSX, encoding="utf-8-sig")
print("已为", len(text_category_audit), "张SVG加入完整标签类别标记。")


## 8. 最终核验
核对42套训练、42套SHAP、预测概率、矢量文字，以及全部百分比混淆矩阵中的25个数值。

In [ ]:
from xml.etree import ElementTree as ET
assert len(RUNS)==42 and not RUN_ERRORS
assert len(ORIGINAL_FIGURES)>0
assert len(SHAP_SUMMARIES)==42
assert len([f for f in ALL_MODEL_SHAP_FIGURES if f['family']=='shap'])==42
for figure in ORIGINAL_FIGURES:
    path=Path(figure['svg']);svg=ET.parse(path).getroot()
    assert not any(el.tag.endswith('}image') for el in svg.iter()),path
    assert any(el.tag.endswith('}text') for el in svg.iter()),path
for run in RUNS:
    pred=pd.read_csv(Path(run['folder'])/'predictions.csv')
    assert np.allclose(pred[[f'p{i}' for i in range(5)]].sum(axis=1),1,atol=1e-8)
write_json(dict(profile=PROFILE,successful_runs=len(RUNS),models_per_scheme=21,
    formal_run=PROFILE=='full',nested_validation=True,selected_by_nested_cv=cv_selected,
    selected_for_paper=selected,svg_count=len(ORIGINAL_FIGURES),svg_text_editable=True,
    svg_embedded_images=0,plotting='User original functions/cells with logged data substitutions',
    original_code_provenance=str(OUT/'original_template_audit/execution_provenance.json'),
    all_model_shap=True,shap_model_split_count=len(SHAP_SUMMARIES),
    all_model_shap_provenance=str(OUT/'all_models_shap/execution_provenance.json')),
    OUT/'validation_summary.json')
print('原模板图索引:',OUT/'figure_index.csv')
print('原代码执行记录:',OUT/'original_template_audit/execution_provenance.json')
print('数据和模型比较:',OUT/'model_comparison.csv')
# Independently reconcile percentage matrices with frozen predictions.
from collections import Counter
import re
percentage_audit = []
for run in RUNS:
    pred = pd.read_csv(Path(run['folder'])/'predictions.csv', float_precision='round_trip').query("split == 'test'")
    cm = skm.confusion_matrix(pred.y_true, pred.y_pred, labels=range(5))
    normalized = cm / cm.sum(axis=1, keepdims=True)
    assert np.allclose(normalized.sum(axis=1), 1)
    path = OUT/'figures/original_templates'/run['scheme']/run['model']/'confusion_matrix.svg'
    labels = [node.text for node in ET.parse(path).getroot().iter()
              if node.tag.endswith('}text') and re.fullmatch(r'\d+\.\d%', node.text or '')]
    assert Counter(labels) == Counter(f"{v:.1%}" for v in normalized.ravel()), path
    percentage_audit.append(dict(scheme=run['scheme'], model=run['model'], rows_sum_to_one=True, matched_25_cells=True))
write_json(percentage_audit, OUT/'percentage_confusion_validation.json')
summary = json.loads((OUT/'validation_summary.json').read_text(encoding='utf8'))
summary.update(profile=PROFILE, formal_run=(PROFILE=='full'), percentage_confusion_matrices=True,
               shap_permutations=SHAP_PERMUTATIONS, paper_composites=len(PAPER_FIGURE_MANIFEST),
               source_data_hashes=_input_hashes, font_files_bundled=False)
write_json(summary, OUT/'validation_summary.json')
print('Validation completed for profile:', PROFILE)
print('Outputs:', OUT)


## 运行产物与复现范围

- `artifacts/<配置>/model_comparison.csv`：21模型两种划分的性能比较。
- 每个模型目录保存最终参数、模型、冻结测试预测与外层逐折指标。
- `figures/original_templates/`：原模板输出的单模型SVG/PDF。
- `figures/paper_groups/`：分类组图、九模型组图与前三模型SHAP。
- `all_models_shap/`：解释张量、背景样本索引、加和一致性与执行记录。

内置数据是一个记录中的重复事件。当前分组评估不能替代跨器件、跨材料批次或跨日期验证。SHAP是模型归因，不能据此宣称材料机制因果成立。正式结果须使用`full`重新计算；实际结果以当前运行目录的输出为准。